# Know Your Data: Exploring and Preparing Image Datasets at the Wildwood Farm Park
## 0. Setup & the park

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Run the setup once, meet the park, and see what you will be able to do at the end.</p>
</div>""")

<callout: objective>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-banner" style="background:#0B6E79;color:#FFFFFF;border-left:6px solid #06464D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-size:22px;font-weight:bold;line-height:1.3;">Know Your Data: Exploring and Preparing Image Datasets at the Wildwood Farm Park</div>
<p style="margin:6px 0 0 0;"><strong>At the end of this notebook you will be able to:</strong></p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">Inspect an image dataset qualitatively and quantitatively, and say whether it fits a task.</li>
<li style="margin:2px 0;">Find and fix common dataset errors (duplicates, flipped/rotated copies, bad labels or masks) before
   they leak into training.</li>
<li style="margin:2px 0;">Choose a split and a metric that match the use case, and explain how imbalance, object size and leakage
   look different for classification, detection and segmentation.</li>
</ol>
<p style="margin:6px 0 0 0;">Questions are marked 🟢 beginner · 🟡 intermediate · 🔴 advanced (optional).</p>
</div>""")

<callout: banner>

**Run the next three cells now, in order, and read on while they work.** The last downloads the
Play Dataset (about 115 MB): **a minute or two the first time**, instant after that.

<details><summary>What the three cells do</summary>

The first installs the one package Colab lacks, fixes the random seed, sets the colours and defines `use()`,
the helper that lets you write small functions without ever blocking "Run all".

</details>

In [ ]:
# @title Setup (run me first) { display-mode: "form" }
import ast
import inspect
import os
import random
import sys
import textwrap

import matplotlib as mpl
import matplotlib.pyplot as plt  # noqa: F401  (every later section plots with plt)
import numpy as np

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("NB_SMOKE") == "1"  # automated tests: tiny data, CPU, seconds. Never set this by hand.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB:  # Jupytext turns the commented magic below into a real `%pip install -q imagehash` (N-021)
    pass
    %pip install -q imagehash

# --- Theme (SPEC L-003): white background, dark teal text, light grid, one colour per class everywhere ---
CLASSES = ["cat", "dog", "bird", "horse", "sheep", "zebra"]
CLASS_COLORS = {
    "cat": "#E07A5F",
    "dog": "#3D405B",
    "bird": "#F2CC8F",
    "horse": "#81B29A",
    "sheep": "#9C89B8",
    "zebra": "#2A9D8F",
}
INK = "#21434A"
mpl.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "text.color": INK,
        "axes.labelcolor": INK,
        "axes.edgecolor": INK,
        "axes.titlecolor": INK,
        "xtick.color": INK,
        "ytick.color": INK,
        "axes.grid": True,
        "grid.color": "#DDE5E7",
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.prop_cycle": mpl.cycler(color=[CLASS_COLORS[c] for c in CLASSES]),
        "figure.dpi": 100,
        "font.size": 11,
    }
)

# --- Results every section writes into (SPEC §9) ---
RESULTS = {}
RESULTS["class_colors"] = dict(CLASS_COLORS)


# --- use(): run a learner's function next to the reference solution (N-029) ---
def _is_stub(fn) -> bool:
    """True if fn's body is only a docstring plus `...` or `pass` (the learner has not written it yet)."""
    try:
        node = ast.parse(textwrap.dedent(inspect.getsource(fn))).body[0]
        body = node.body
    except (OSError, TypeError, SyntaxError, IndexError, AttributeError):
        return False
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, "value", None), str):
        body = body[1:]  # skip the docstring
    return all(
        isinstance(n, ast.Pass) or (isinstance(n, ast.Expr) and getattr(n.value, "value", None) is Ellipsis)
        for n in body
    )


def _same(a, b) -> bool:
    if hasattr(a, "equals"):  # pandas
        return bool(a.equals(b))
    try:
        if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
            return bool(np.array_equal(a, b))
        return bool(a == b)
    except Exception:
        return False


def _short(x, n=60) -> str:
    s = repr(x)
    return s if len(s) <= n else s[: n - 1] + "…"


def _diff(mine, ref) -> str:
    """One short line on how `mine` differs from `ref` (e.g. 'cat: 12 vs 14')."""
    if isinstance(mine, dict) and isinstance(ref, dict):
        keys = list(ref) + [k for k in mine if k not in ref]
        bad = [k for k in keys if not _same(mine.get(k, "—"), ref.get(k, "—"))]
        return ", ".join(f"{k}: {_short(mine.get(k, '—'))} vs {_short(ref.get(k, '—'))}" for k in bad[:4])
    if isinstance(mine, (list, tuple, set)) and isinstance(ref, (list, tuple, set)):
        try:
            extra, missing = set(mine) - set(ref), set(ref) - set(mine)
            parts = []
            if missing:
                parts.append(f"missing {_short(sorted(missing, key=str))}")
            if extra:
                parts.append(f"extra {_short(sorted(extra, key=str))}")
            return " · ".join(parts) or f"same items, different order or length ({len(mine)} vs {len(ref)})"
        except TypeError:
            pass
    return f"{_short(mine)} vs {_short(ref)}"


def use(fn, *args, reference, **kwargs):
    """Run `fn(*args, **kwargs)` next to `reference(*args, **kwargs)`; ALWAYS return the reference's result.

    - stub (body is `...`) or NotImplementedError → "using the reference solution";
    - otherwise ✅ when both agree, ❌ with a short diff when they do not.
    The notebook continues with the reference either way, so "Run all" never breaks on your code.
    """
    expected = reference(*args, **kwargs)
    name = getattr(fn, "__name__", "your function")
    if _is_stub(fn):
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    try:
        mine = fn(*args, **kwargs)
    except NotImplementedError:
        print(f"✍️  {name}: not written yet, using the reference solution.")
        return expected
    except Exception as err:  # a learner's bug must not stop the notebook
        print(f"❌ {name} raised {type(err).__name__}: {err} · using the reference solution.")
        return expected
    if _same(mine, expected):
        print(f"✅ {name} matches the reference solution.")
    else:
        print(f"❌ {name} differs from the reference: {_diff(mine, expected)} · using the reference.")
    return expected


# --- answer_box(): a worked answer computed at runtime, closed until opened (N-024, SPEC L-005) ---
ANSWER_BG, ANSWER_INK = "#F6EFE4", "#5C4027"  # the build's `answer` callout colours


def answer_box(title, lines=(), fig=None, table=None):
    """Draw a collapsed 💡 toggle holding printed lines, a DataFrame and/or matplotlib figure(s)."""
    import base64
    import html
    import io

    from IPython.display import HTML, display

    parts = []
    if lines:
        text = html.escape("\n".join(lines))
        parts.append(f'<pre style="white-space:pre-wrap;margin:6px 0 0 0;font-size:0.92em;">{text}</pre>')
    if table is not None:
        parts.append(table.to_html(border=0))
    for f in fig if isinstance(fig, (list, tuple)) else [fig]:
        if f is None:
            continue
        buf = io.BytesIO()
        f.savefig(buf, format="png", bbox_inches="tight")
        plt.close(f)
        png = base64.b64encode(buf.getvalue()).decode()
        parts.append(f'<img src="data:image/png;base64,{png}" style="max-width:100%;margin-top:6px;">')
    box = (f"background:{ANSWER_BG};color:{ANSWER_INK};border-left:6px solid {ANSWER_INK};border-radius:6px;"
           "padding:12px 14px;line-height:1.5;font-size:14px;width:100%;box-sizing:border-box;margin:2px 0;")
    display(HTML(f'<div style="{box}"><details style="margin:0;"><summary style="cursor:pointer;'
                 f'font-weight:bold;">💡 {html.escape(title)}</summary>{"".join(parts)}</details></div>'))


print(f"Colab: {IN_COLAB} · smoke mode: {SMOKE} · seed: {SEED} · imagehash: ", end="")
try:
    import imagehash  # noqa: F401

    print("ok")
except ImportError:
    print("missing (run this cell again)")

In [ ]:
# @title Data helpers (download with cache, build the tables) { display-mode: "form" }
import json
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

# Where the committed manifests live. In Colab they are fetched from the repository; locally they are read
# from ./data. ⬇️ DATA_BASE_URL: change this if the repository moves or you work on a fork.
DATA_BASE_URL = "https://raw.githubusercontent.com/FHNW-VISED/eda-teaching-notebook/main/data/"
DATA_DIR = Path("data")
IMAGE_CACHE = DATA_DIR / "cache" / "coco"
SMOKE_IMAGES_PER_CLASS = 6


def _github_headers() -> dict:
    """Authorization header from the Colab secret GITHUB_TOKEN, if the repository is private (SPEC §5)."""
    if not IN_COLAB:
        return {}
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return {"Authorization": f"token {token}"} if token else {}
    except Exception:
        return {}


def download_file(url: str, dest: Path, headers: dict | None = None, attempts: int = 3) -> Path | None:
    """Download `url` to `dest` once; a half-finished download is written to `.part` first (N-019)."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(attempts):
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=60) as resp, open(part, "wb") as fh:
                fh.write(resp.read())
            part.rename(dest)
            return dest
        except (urllib.error.URLError, TimeoutError, OSError):
            if attempt == attempts - 1:
                part.unlink(missing_ok=True)
                return None
    return None


def fetch_manifest(name: str) -> Path:
    """Return the local path of a committed manifest, downloading it from DATA_BASE_URL if needed."""
    path = DATA_DIR / name
    if not path.exists():
        if download_file(DATA_BASE_URL + name, path, headers=_github_headers()) is None:
            raise RuntimeError(
                f"Could not download {name} from {DATA_BASE_URL}. Check DATA_BASE_URL, or add a GITHUB_TOKEN "
                "secret in Colab if the repository is private."
            )
    return path


def download_images(rows: list[dict], workers: int = 16) -> dict[int, Path]:
    """Download every image in `rows` ('id', 'coco_url', 'file_name') into IMAGE_CACHE; one progress line."""
    done, paths = 0, {}

    def one(row):
        return row["id"], download_file(row["coco_url"], IMAGE_CACHE / row["file_name"])

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for image_id, path in pool.map(one, rows):
            done += 1
            if path is not None:
                paths[image_id] = path
            print(f"\r  images ready: {done}/{len(rows)}", end="", flush=True)
    print()
    return paths


def build_tables(coco: dict, paths: dict[int, Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Tidy tables from a COCO-format dict: `images` (one row per photo), `annotations` (one per animal)."""
    cat_name = {c["id"]: c["name"] for c in coco["categories"]}
    lic_name = {lic["id"]: lic["name"] for lic in coco.get("licenses", [])}
    anns = pd.DataFrame(
        {
            "id": a["id"],
            "image_id": a["image_id"],
            "class": cat_name[a["category_id"]],
            "bbox": a["bbox"],
            "area": a["area"],
            "segmentation": a["segmentation"],
            "iscrowd": a["iscrowd"],
        }
        for a in coco["annotations"]
        if a["image_id"] in paths
    )
    per_image_class = anns.groupby("image_id")["class"].first()
    n_animals = anns.groupby("image_id").size()
    imgs = pd.DataFrame(
        {
            "id": im["id"],
            "file": im["file_name"],
            "path": str(paths[im["id"]]),
            "width": im["width"],
            "height": im["height"],
            "class": per_image_class.get(im["id"]),
            "n_animals": int(n_animals.get(im["id"], 0)),
            "license": lic_name.get(im["license"], str(im["license"])),
        }
        for im in coco["images"]
        if im["id"] in paths
    )
    imgs = imgs.sort_values("id").reset_index(drop=True)
    anns = anns.sort_values("id").reset_index(drop=True)
    return imgs, anns

In [ ]:
# @title Load the manifest, download the images, build the tables { display-mode: "form" }
COCO_RAW = json.loads(fetch_manifest("coco_animals_subset.json").read_text())
_cat_name = {c["id"]: c["name"] for c in COCO_RAW["categories"]}
_image_class = {a["image_id"]: _cat_name[a["category_id"]] for a in COCO_RAW["annotations"]}

_rows = COCO_RAW["images"]
if SMOKE:  # same code path, fewer images (N-014)
    _rng = random.Random(SEED)
    _rows = []
    for c in CLASSES:
        _of_class = [im for im in COCO_RAW["images"] if _image_class[im["id"]] == c]
        _rows += sorted(_rng.sample(_of_class, SMOKE_IMAGES_PER_CLASS), key=lambda im: im["id"])

print(f"{len(_rows)} images to fetch from images.cocodataset.org (cached in {IMAGE_CACHE}/)")
_paths = download_images(_rows)
if len(_paths) < len(_rows):
    print(f"  ⚠️ {len(_rows) - len(_paths)} images could not be downloaded and are left out; run again.")

images, annotations = build_tables(COCO_RAW, _paths)
RESULTS["n_images"] = int(len(images))

_counts = Counter(images["class"])
print(f"images: {len(images)} · animals (annotations): {len(annotations)}")
print("per class: " + " · ".join(f"{c} {_counts[c]}" for c in CLASSES))
images.drop(columns="license").head()  # the licence column stays in the table, not on screen (N-045)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Before you start</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">A Google account and Python basics. No model is trained here; that is next week.</li>
<li style="margin:2px 0;">When a question says <em>predict</em> or <em>guess</em>, say or think your answer <strong>before</strong> you run the cell. You do
  not need to type it.</li>
</ul>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Your project dataset, and the Karpathy reference</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">The notebook runs in Colab, where a CPU runtime is enough, and assumes the lecture on image datasets.</li>
<li style="margin:2px 0;">Bring your project dataset as one folder per class (section 10): mount Google Drive or upload a zip
  there.</li>
<li style="margin:2px 0;">The notebook follows the first part of Karpathy's
  <a href="https://karpathy.github.io/2019/04/25/recipe/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;"><em>A Recipe for Training Neural Networks</em></a>: know your data
  before you touch a model.</li>
</ul>
</details>
</div>""")

<callout: theory>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 The Wildwood Farm Park and what it wants from its cameras</div>
<p style="margin:6px 0 0 0;">Cameras watch the 🐑🐴 farm paddock (sheep, horses), the 🐦 aviary pond (birds), the 🦓 savanna (zebras)
and the 🐱🐶 visitor yard (cats and dogs wander through). Three jobs:</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Recognise</strong> (classification): who is in front of this camera? <strong>A dog in the sheep paddock</strong> triggers
   an alert; missing that dog is worse than a false alarm.</li>
<li style="margin:2px 0;"><strong>Count</strong> (detection): a daily headcount of sheep, zebras and birds.</li>
<li style="margin:2px 0;"><strong>Outline</strong> (segmentation): how much of each enclosure each animal uses.</li>
</ol>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">How big the birds are</summary>
<p style="margin:6px 0 0 0;">Far-away birds are tiny. Birds cover few pixels.</p>
</details>
</div>""")

<callout: theory>

The park has no training data yet, so we practise on a **Play Dataset**: COCO 2017 photos of the same six
animals, each photo showing one animal class.

<details><summary>The two tables, and several animals per photo</summary>

The cell above gave you two tables: `images` (one row per photo: size, class, number of animals) and
`annotations` (one row per animal: its box and mask).

Several photos hold more than one animal (`n_animals`), which hints at a question in section 4. Everything
you learn here is a checklist you will reuse on your own dataset in section 10.

</details>

### How the ✍️ cells work

About four times you write a small function. **Run the reference cell first**, then the stub as is; then
fill in the body and run it again to see the message change. A wrong or unfinished function never stops
the notebook.

<details><summary>What each ✍️ cell comes with</summary>

Each comes with its signature and a `use(my_function, …, reference=…)` line that compares your result with
the reference solution and then continues with the reference.

</details>

In [ ]:
# @title Reference solution — run this cell; open it only if you are stuck { display-mode: "form" }
def _ref_double(x):
    return 2 * x

In [ ]:
# ✍️ YOUR CODE
def double(x):
    """Return twice x.

    e.g. double(21) → 42
    """
    ...


result = use(double, 21, reference=_ref_double)
print("the notebook continues with:", result)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a ✍️ cell → <strong>do</strong> run it as it is first; an unfinished function falls back to the
  reference.</li>
<li style="margin:2px 0;"><strong>When you see</strong> <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">images</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">annotations</code> → <strong>do</strong> think "one row per photo, one row per animal".</li>
</ul>
</div>""")

<callout: takeaway>

## 1. Image datasets

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Know where image datasets come from and what makes a good one, then hold the Play Dataset against that
checklist.</p>
</div>""")

<callout: objective>

The park has no training data yet. Before anyone mounts cameras and hires annotators, ask:
*does a usable dataset already exist?* First, see what a dataset looks like.

**What is a label here?** In this notebook a photo comes with a class, a box and a mask. The next cell draws
one photo from the `images` and `annotations` tables with all three.

<details><summary>How would you validate labels?</summary>

Draw a random sample of N photos, check each label by eye and count the mistakes; the share of mistakes in
the sample estimates the share in the whole set.

</details>

In [ ]:
# @title Show one photo with its class, its box and its mask { display-mode: "form" }
import matplotlib.patches as _patches
from PIL import Image as _Image
from pycocotools import mask as _mask_utils


def _label_mask(segmentation, height: int, width: int) -> np.ndarray:
    """Decode a COCO segmentation (polygons or RLE) into a 0/1 array of the photo's size."""
    if isinstance(segmentation, list):
        rle = _mask_utils.merge(_mask_utils.frPyObjects(segmentation, height, width))
    elif isinstance(segmentation["counts"], list):
        rle = _mask_utils.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation
    return _mask_utils.decode(rle)


_single = images[images["n_animals"] == 1]
_row = (_single if len(_single) else images).sample(1, random_state=SEED).iloc[0]
_ann = annotations[annotations["image_id"] == _row["id"]].iloc[0]
_color = CLASS_COLORS[_ann["class"]]
_photo = np.array(_Image.open(_row["path"]).convert("RGB"))
_m = _label_mask(_ann["segmentation"], int(_row["height"]), int(_row["width"])).astype(bool)

_shown = _photo.copy()
_rgb = np.array([int(_color[i : i + 2], 16) for i in (1, 3, 5)])
_shown[_m] = (0.45 * _shown[_m] + 0.55 * _rgb).astype(np.uint8)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(_shown)
_bx, _by, _bw, _bh = _ann["bbox"]
ax.add_patch(_patches.Rectangle((_bx, _by), _bw, _bh, fill=False, edgecolor=_color, linewidth=3))
ax.set_title(f"class: {_ann['class']} · box and mask drawn from the tables")
ax.axis("off")
plt.show()
print(f"class = {_ann['class']!r} · box (x, y, width, height) = {[round(v) for v in _ann['bbox']]} px")
print(f"mask covers {int(_m.sum()):,} of {_m.size:,} pixels ({_m.mean():.0%} of the photo)")

Before judging the dataset, look at it: six photos of each class.

In [ ]:
# @title Six example photos per class, framed in the class colour { display-mode: "form" }
_per_class = 6
fig, axes = plt.subplots(_per_class, len(CLASSES), figsize=(13, 2.2 * _per_class))
for j, c in enumerate(CLASSES):
    _of_class = images[images["class"] == c]
    _picked = _of_class.sample(min(_per_class, len(_of_class)), random_state=SEED)
    for i in range(_per_class):
        ax = axes[i, j]
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(CLASS_COLORS[c])
            spine.set_linewidth(4)
        if i < len(_picked):
            _thumb = _Image.open(_picked.iloc[i]["path"]).convert("RGB")
            _thumb.thumbnail((256, 256))
            ax.imshow(_thumb)
        if i == 0:
            ax.set_title(c, color=INK, fontweight="bold")
fig.tight_layout()
plt.show()

Having a dataset is not the same as having a *good* one. This is the checklist from the lecture:

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · What makes a good dataset</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Does it fit the question?</strong> The images, the classes and the setting must match the job you want the
  model to do.<br><em>Park example:</em> the alert needs a dog seen from a fixed camera in the sheep paddock, not a
  dog in a holiday snapshot taken from two metres away.</li>
<li style="margin:2px 0;"><strong>Enough samples</strong>, <strong>2–10 classes</strong> and <strong>about 500×500 px</strong>: the numbers to check, explained below.</li>
<li style="margin:2px 0;"><strong>Classes, distributions, imbalance, bias.</strong> Know them, and take them into account in training (for
  example with a weighted loss).<br>
  <em>Park example:</em> if 70 % of the photos are sheep and 3 % are dogs, the model sees few dogs to learn from,
  and the dog is the animal the alert exists for (section 4).</li>
<li style="margin:2px 0;"><strong>Variability matches the target population</strong>, and <strong>labels are present and validated</strong>: every image has
  a label, and someone checked a sample of them.<br>
  <em>Park example:</em> a photo of a zebra labelled "horse" teaches the model the wrong thing, and nobody notices
  until the zebra headcount is off.</li>
<li style="margin:2px 0;"><strong>Objects are identifiable and separable by eye.</strong></li>
<li style="margin:2px 0;"><strong>What do experts and the literature say</strong> about this data and this task?</li>
<li style="margin:2px 0;"><strong>Get a qualitative and a quantitative sense of the data</strong> (Karpathy, 2019): look at many images, then
  measure.</li>
</ul>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">The reasons and park examples behind each point</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Enough samples:</strong> a few hundred to a few thousand images, depending on how many classes you have.
  <em>Park example:</em> 100 photos per animal (600 in all) is a fair start; 12 zebra photos is not.</li>
<li style="margin:2px 0;"><strong>2–10 classes:</strong> more classes need more photos each, and similar classes get confused.
  <em>Park example:</em> six animals is fine; splitting sheep and horses into 40 breeds would not be.</li>
<li style="margin:2px 0;"><strong>About 500×500 px:</strong> check the input size of the models you plan to use.
  <em>Park example:</em> a bird across the pond may cover only 20×20 px of a 640×480 photo; shrink the photo to
  224×224 and it is a handful of pixels.</li>
<li style="margin:2px 0;"><strong>Variability:</strong> lighting, viewpoints, backgrounds and seasons should look like the ones
  the model will meet.
  <em>Park example:</em> take pictures of the park in winter and in summer, in sun and at dusk. A model that only
  saw sunny July afternoons will meet snow and low light in January.</li>
<li style="margin:2px 0;"><strong>Separable by eye:</strong> if you cannot tell them apart, the model will struggle too.
  <em>Park example:</em> a bird that is a dark speck over the pond, or sheep standing so close that their outlines
  merge into one woolly shape.</li>
<li style="margin:2px 0;"><strong>Experts and literature:</strong> before trusting COCO's masks, read the audit that found some of them poor; it
  tells you what to look for.</li>
<li style="margin:2px 0;"><strong>Qualitative and quantitative:</strong> look at many images and see how big the animals are and how much of the
  photo they cover. Is it the same for each class? Can you already see imbalances just by looking?</li>
</ul>
</details>
</div>""")

<callout: theory>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Predict before you run: the park wants to recognise, count and outline six animals from fixed cameras.</p>
</div>""")

<callout: questions>

**Q1.** 🟢 Which bullets do you expect the Play Dataset to pass (✅) as it is, and which one worries you
most for the dog-in-the-paddock alert? Write one line, then run the next cell. It goes through the
checklist: where the tables from section 0 can answer a bullet it computes the answer (✅ fine, ⚠️ needs
attention); where only your eyes can, it says so (👀).

In [ ]:
# @title Q1 Apply the checklist to the Play Dataset { display-mode: "form" }
# Every number comes from the `images` table of section 0.
n_images = len(images)
counts = images["class"].value_counts().reindex(CLASSES).fillna(0).astype(int)
n_classes = int((counts > 0).sum())
imbalance = counts.max() / max(counts.min(), 1)
median_w, median_h = int(images["width"].median()), int(images["height"].median())
share_labelled = float((images["n_animals"] > 0).mean())
share_several = float((images["n_animals"] > 1).mean())

checklist = [
    (
        "⚠️",
        "Fits the question?",
        "the park's six animals, but Flickr holiday photos, not fixed cameras → judge in section 2",
    ),
    (
        "✅" if n_images >= 300 else "⚠️",
        "A few hundred to a few thousand samples",
        f"{n_images} images" + ("" if n_images >= 300 else " (a tiny test sample; the full set has ~600)"),
    ),
    ("✅" if 2 <= n_classes <= 10 else "⚠️", "2–10 classes", f"{n_classes}: " + ", ".join(CLASSES)),
    (
        "✅" if min(median_w, median_h) >= 224 else "⚠️",
        "About 500×500 px",
        f"median {median_w}×{median_h} px; most classifiers take 224–512 px, so there is room to resize",
    ),
    (
        "⚠️" if imbalance > 2 else "✅",
        "Classes, distributions, imbalance, bias",
        " · ".join(f"{c} {counts[c]}" for c in CLASSES)
        + f" → largest / smallest = {imbalance:.1f}× (weighted loss? next week)",
    ),
    ("👀", "Variability matches the target population", "lighting, distance, backgrounds: look in section 2"),
    (
        "✅" if share_labelled == 1 else "⚠️",
        "Labels present",
        f"{share_labelled:.0%} of images have at least one box and mask",
    ),
    (
        "⚠️",
        "Labels validated",
        "COCO-ReM (Singh et al., 2024) found ~2.4 % of val masks poor, plus duplicate masks → sections 5, 8",
    ),
    (
        "👀",
        "Objects identifiable and separable by eye",
        f"{share_several:.0%} of images hold several animals; check the grids in section 2",
    ),
    (
        "📚",
        "What experts and the literature say",
        "Lin et al. 2014 (COCO) and Singh et al. 2024 (COCO-ReM) document the data and its mask errors",
    ),
    ("📓", "Qualitative and quantitative sense (Karpathy 2019)", "sections 2–4 of this notebook"),
]
for status, bullet, finding in checklist:
    print(f"{status} {bullet}: {finding}")

Every ⚠️ and 👀 names the later section that settles it; that is the order we follow.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q1)</summary>
<p style="margin:6px 0 0 0;">Sample size, class count and resolution pass without work. The two that matter for
the alert are <em>fit</em> (holiday photos of dogs are not a camera on a paddock) and <em>imbalance</em> (the biggest
class has over twice the images of the smallest, so a model can score well while ignoring a rare animal).
Section 4 makes that concrete for the dog and picks the alert's metric.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Find a dataset, or label your own</div>
<p style="margin:6px 0 0 0;">Where to look for a dataset, and how to label your own images:</p>
<p style="margin:6px 0 0 0;"><strong>Find a dataset</strong></p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><a href="https://www.kaggle.com/datasets" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Kaggle datasets</a>: a huge community collection, good for a first look.</li>
<li style="margin:2px 0;"><a href="https://datasetsearch.research.google.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Google Dataset Search</a>: one search box over many repositories.</li>
<li style="margin:2px 0;"><a href="https://www.v7labs.com/open-datasets" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">V7 open datasets</a>: a curated list of open computer-vision data
  (plus <a href="https://www.v7labs.com/blog" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">their blog</a> with overviews of the well-known ones).</li>
<li style="margin:2px 0;"><a href="https://universe.roboflow.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Roboflow Universe</a>: community datasets, many already labelled with boxes
  or masks.</li>
<li style="margin:2px 0;"><a href="https://lila.science" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">LILA</a>: wildlife and camera-trap images, the closest match to the park's cameras.</li>
<li style="margin:2px 0;"><a href="https://pytorch.org/vision/stable/datasets.html" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">torchvision datasets</a>: the classics, one import away.</li>
<li style="margin:2px 0;"><a href="https://cocodataset.org" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">COCO</a>: a general-purpose dataset; our Play Dataset is a slice of it.</li>
</ul>
<p style="margin:6px 0 0 0;"><strong>Label your own images</strong></p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><a href="https://supervisely.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Supervisely</a>: a labelling platform for boxes, polygons and masks.</li>
<li style="margin:2px 0;"><a href="https://www.v7labs.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">V7</a>: browser labelling, free for education.</li>
<li style="margin:2px 0;"><a href="https://roboflow.com/annotate" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Roboflow Annotate</a>: browser labelling that exports to common formats.</li>
<li style="margin:2px 0;"><a href="https://labelstud.io" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Label Studio</a>: open source, runs on your own machine.</li>
</ul>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a dataset you did not label yourself → <strong>do</strong> run the checklist before you trust it, and
  note which points only your eyes can answer.</li>
<li style="margin:2px 0;"><strong>When you see</strong> "labels are present" → <strong>do</strong> look at a few photos with their box and mask, then
  spot-check a random sample and count the mistakes.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a ⚠️ or 👀 on the checklist → <strong>do</strong> write down which later check will settle it.</li>
</ul>
</div>""")

<callout: takeaway>

## 2. Become one with the data

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Look at many images before computing anything, and write down what you see.</p>
</div>""")

<callout: objective>

Section 1 left three 👀 questions only your eyes can answer (most people skip this step). Karpathy puts
this step first in his recipe:

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-recipe" style="background:#E8F5E9;color:#1B5E20;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🧾 Recipe · Step 1: Become one with the data</div>
<p style="margin:6px 0 0 0;"><em>"The first step to training a neural net is to not touch any neural net code at all and instead begin by
thoroughly inspecting your data. This step is critical. I like to spend copious amount of time (measured in
units of hours) scanning through thousands of examples, understanding their distribution and looking for
patterns."</em></p>
<p style="margin:6px 0 0 0;">Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">A Recipe for Training Neural
Networks</a> (2019).</p>
</div>""")

<callout: recipe>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · The three 👀 questions from section 1</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">👀 Does the data <strong>fit</strong> the park's cameras?</li>
<li style="margin:2px 0;">👀 Does its <strong>variability</strong> match what the cameras will see?</li>
<li style="margin:2px 0;">👀 Are the animals <strong>identifiable by eye</strong>?</li>
</ul>
</div>""")

<callout: go-deeper>

We will not spend hours, but we will spend a few minutes. Scroll back to the class grid of section 1 (six
photos per class, framed in the class colour used in every plot of this notebook) and look at:

- the backgrounds and the lighting,
- the distance to the animal,
- anything else in the photo.

In [ ]:
# @title Grid helpers { display-mode: "form" }
from IPython.display import display
from PIL import Image


def load_thumb(path, size: int = 256) -> Image.Image:
    """Open an image as a small RGB thumbnail (fast: JPEG decodes at reduced size)."""
    im = Image.open(path)
    im.draft("RGB", (size, size))
    im = im.convert("RGB")
    im.thumbnail((size, size))
    return im


def sample_rows(class_name: str, n: int) -> pd.DataFrame:
    """A fixed random sample of `n` rows of `images` for one class (seeded: re-runs show the same photos)."""
    of_class = images[images["class"] == class_name]
    n = min(n, len(of_class))
    idx = random.Random(SEED).sample(sorted(of_class.index), n)
    return of_class.loc[sorted(idx)]


def frame(ax, color: str, width: float = 4) -> None:
    """Draw a coloured frame around an image axis and hide ticks and grid."""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(width)


def show_grid(class_name: str, n: int, cols: int = 6, label_animals: bool = True) -> None:
    """Show `n` photos of one class in a grid, framed in CLASS_COLORS[class_name]."""
    rows_df = sample_rows(class_name, n)
    n = len(rows_df)
    n_rows = max(1, -(-n // cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.2 * cols, 2.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[class_name])
        if label_animals:
            plural = "s" if row["n_animals"] != 1 else ""
            ax.set_title(f"{row['n_animals']} {class_name}{plural}", fontsize=8)
    fig.suptitle(f"{class_name} · {n} of {int((images['class'] == class_name).sum())} photos", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">The park's cameras are fixed, outdoors, and often far from the animal.</p>
</div>""")

<callout: questions>

**Q2.** 🟢 Look at the class grid above (section 1). Which class looks *least* like a fixed park camera's
view, and why? One line.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q2)</summary>
<p style="margin:6px 0 0 0;">Holiday snapshots, not camera-trap frames. Cats and dogs are the farthest from a
park camera: indoors, up close, with people and sofas. Sheep, horses and zebras are at least outdoors and
sometimes far away, closer to what the paddock and savanna cameras will see.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · Pick a class</div>
<p style="margin:6px 0 0 0;">Look at more photos of one class: twelve is enough to start noticing patterns. <strong>Look for:</strong> which class has
the most photos that a paddock camera would never take?</p>
<p style="margin:6px 0 0 0;">Set <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">CLASS_TO_SHOW</code> (<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">cat, dog, bird, horse, sheep, zebra</code>) and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">N_SHOW</code> (how many photos you want), then
run the cell. Try <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"dog"</code> (the alert class) and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"bird"</code> (the tiny-box class).</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
CLASS_TO_SHOW = "dog"
N_SHOW = 12

if CLASS_TO_SHOW not in CLASSES:
    raise SystemExit(f"CLASS_TO_SHOW must be one of {CLASSES}, not {CLASS_TO_SHOW!r}")
if not isinstance(N_SHOW, int) or N_SHOW < 1:
    raise SystemExit(f"N_SHOW must be a positive integer, not {N_SHOW!r}")
n_show = min(N_SHOW, 6) if SMOKE else N_SHOW  # smoke mode caps the grid, same code path (N-014)
show_grid(CLASS_TO_SHOW, n_show)

💡 Each title says how many animals of that class the photo holds. More than one is common (section 4).

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · One label per image is a simplification</div>
<p style="margin:6px 0 0 0;">A photo with several animals still gets one label, so "one label per image" already hides something
(section 4).</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Karpathy's step ends with writing things down. Edit the next markdown cell (double-click it).</p>
</div>""")

<callout: questions>

**Q3.** 🟢 Write three specific observations about the photos you just looked at, each one hinting at
something the park should check later.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · What a good observation looks like</div>
<p style="margin:6px 0 0 0;">Specific beats vague: "many dog photos are indoors with a person's legs in the frame", not "dogs look
different". Good topics: fit to the cameras, lighting, distance, several animals, people, odd photos.</p>
</div>""")

<callout: go-deeper>

**My three observations** (double-click to edit):

1. …
2. …
3. …

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q3)</summary>
<p style="margin:6px 0 0 0;">, for the dog grid on the full set. Yours will differ; that is the point.</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Setting:</strong> most dogs are indoors or in a street, a metre away; the paddock camera sees one far off.</li>
<li style="margin:2px 0;"><strong>People:</strong> a person, hand or leash is in many photos: "person + leash = dog" is a shortcut to learn.</li>
<li style="margin:2px 0;"><strong>Odd photos:</strong> two or three dogs, a costume, a screen: labelling decisions (sections 4 and 5).</li>
</ol>
<p style="margin:6px 0 0 0;">Keep your three lines: section 10 asks for the same three on your own dataset.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a pattern in the photos (indoors, people, several animals) → <strong>do</strong> write it down as a
  specific observation, and turn it into a check later.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a gap between the photos and the park's cameras (distance, lighting, setting) → <strong>do</strong>
  flag it now, before any model exists.</li>
<li style="margin:2px 0;"><strong>When you see</strong> several animals or an odd subject in one photo → <strong>do</strong> note it as a labelling decision
  for sections 4 and 5.</li>
</ul>
</div>""")

<callout: takeaway>

## 3. Image inspection

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Measure sizes, aspect ratios, colour modes and colour statistics, and read a cluster of odd values as a clue
about where the photos came from.</p>
</div>""")

<callout: objective>

Now the computer measures every photo once. No label is needed, so this is the first check you can run on
*any* folder of images, including your own in section 10.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-recipe" style="background:#E8F5E9;color:#1B5E20;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🧾 Recipe · Step 1, continued: search, filter, sort</div>
<p style="margin:6px 0 0 0;"><em>"Once you get a qualitative sense it is also a good idea to write some simple code to search/filter/sort
by whatever you can think of (e.g. type of label, size of annotations, number of annotations, etc.) and
visualize their distributions and the outliers along any axis. The outliers especially almost always uncover
some bugs in data quality or preprocessing."</em></p>
<p style="margin:6px 0 0 0;">Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">A Recipe for Training Neural
Networks</a> (2019).</p>
</div>""")

<callout: recipe>

The numbers collected per photo, each in one line:

- **width, height, aspect:** the pixels stored in the file, and width ÷ height (1 square, above 1
  landscape, below 1 portrait).
- **mode:** how the file stores colour: `RGB` three channels, `L` one grey channel.
- **brightness:** the mean grey level of the photo: 0 black, 255 white. **contrast:** its standard
  deviation: low is flat or foggy, high is deep shadow next to bright highlight.
- **colourfulness:** how far the colours are from grey. A grayscale picture scores exactly 0.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Every number, and how it is measured</div>
<p style="margin:6px 0 0 0;">Brightness, contrast and colour are measured on a small thumbnail of the photo.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>long_side:</strong> the larger of width and height. <strong>megapixels:</strong> width × height ÷ 1,000,000.</li>
<li style="margin:2px 0;"><strong>mode:</strong> also <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">RGBA</code> (RGB plus transparency) and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">CMYK</code> (print inks).</li>
<li style="margin:2px 0;"><strong>brightness:</strong> R, G, B averaged per pixel, then averaged over the photo.</li>
<li style="margin:2px 0;"><strong>colourfulness</strong> (Hasler &amp; Süsstrunk 2003): <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">sqrt(σ_rg² + σ_yb²) + 0.3 · sqrt(μ_rg² + μ_yb²)</code>, with
  <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">rg = R − G</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">yb = (R + G)/2 − B</code> per pixel, and σ, μ their standard deviation and mean.</li>
<li style="margin:2px 0;"><strong>mean_r, mean_g, mean_b:</strong> the average value of each colour channel.</li>
</ul>
</div>""")

<callout: go-deeper>

One collapsed cell defines the function that measures a single photo; the next one runs it on every photo
of the Play Dataset and keeps the numbers in a table called `image_stats`.

In [ ]:
# @title Measurement helpers { display-mode: "form" }
from matplotlib.patches import Rectangle
from PIL import ImageDraw

HIST_BINS = 32
CHANNEL_COLORS = {"R": "#C0392B", "G": "#27AE60", "B": "#2980B9"}
COLOUR_MODES = ["RGB", "L", "RGBA", "CMYK"]  # the modes we always report, even when the count is 0


def measure_image(path, thumb: int) -> tuple[dict, np.ndarray]:
    """Open one photo and return (stats, channel_hist).

    stats: width, height, mode (as stored in the file), aspect (width / height), long_side, megapixels,
    brightness (mean grey level 0–255 on a thumbnail), contrast (its standard deviation), colourfulness
    (Hasler & Süsstrunk 2003: 0 for a grayscale picture), mean_r / mean_g / mean_b.
    channel_hist: a (3, HIST_BINS) array, the density histogram of the R, G and B channels.
    """
    with Image.open(path) as im:
        mode, (w, h) = im.mode, im.size  # read these before decoding: they describe the file itself
        im.draft("RGB", (thumb, thumb))  # fast reduced-size JPEG decode (no effect on other formats)
        rgb = im.convert("RGB")
        rgb.thumbnail((thumb, thumb))
        arr = np.asarray(rgb, dtype=np.float32)
    grey = arr.mean(axis=2)
    channels = arr.reshape(-1, 3)
    rg = channels[:, 0] - channels[:, 1]  # red minus green
    yb = (channels[:, 0] + channels[:, 1]) / 2 - channels[:, 2]  # yellow (mean of R and G) minus blue
    hist = np.stack(
        [np.histogram(channels[:, k], bins=HIST_BINS, range=(0, 256), density=True)[0] for k in range(3)]
    )
    stats = {
        "width": w,
        "height": h,
        "mode": mode,
        "aspect": w / h,
        "long_side": max(w, h),
        "megapixels": w * h / 1e6,
        "brightness": float(grey.mean()),
        "contrast": float(grey.std()),
        "colourfulness": float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())),
        "mean_r": float(channels[:, 0].mean()),
        "mean_g": float(channels[:, 1].mean()),
        "mean_b": float(channels[:, 2].mean()),
    }
    return stats, hist


def show_rows(rows_df: pd.DataFrame, title: str, caption=None, cols: int = 6) -> None:
    """A small gallery of the photos in `rows_df` (rows of image_stats), each framed in its class colour.

    `caption(row) -> str` writes the title above each photo; by default the size and the colour mode.
    """
    if caption is None:

        def caption(row):
            return f"{row['class']} · {row['width']}×{row['height']} {row['mode']}"

    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.4 * cols, 2.7 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        ax.axis("on")
        ax.imshow(load_thumb(row["path"]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row), fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def _fmt(value) -> str:
    """A number for a caption: 640 stays 640, 1.33333 becomes 1.333."""
    return f"{value:.4g}"


def _closest_to_median(stat: str):
    """The row of `image_stats` whose value of `stat` is closest to the median (the 'typical' photo)."""
    col = image_stats[stat]
    return image_stats.iloc[int((col - col.median()).abs().to_numpy().argmin())]


def _photo(ax, row, caption: str) -> None:
    """Draw one photo with a caption, framed in its class colour."""
    ax.axis("on")
    ax.imshow(load_thumb(row["path"]))
    frame(ax, CLASS_COLORS[row["class"]])
    ax.set_title(caption, fontsize=8)


def show_extremes(stat: str, n: int = 4) -> None:
    """The `n` photos with the lowest value of `stat`, the median ('average') photo, and the `n` highest.

    `stat` is a numeric column of `image_stats` (e.g. "brightness", "aspect", "megapixels", "colourfulness").
    Three rows: lowest (the minimum first), the typical photo in the middle, highest (the maximum first).
    """
    numeric = [c for c in image_stats.columns if pd.api.types.is_numeric_dtype(image_stats[c]) and c != "id"]
    if stat not in numeric:
        raise SystemExit(f"stat must be one of {numeric}, not {stat!r}")
    ordered = image_stats.sort_values(stat)
    n = max(1, min(n, len(ordered) // 2))

    def caption(tag, row):
        size = f"{row['width']}×{row['height']} {row['mode']}"
        return f"{tag}: {stat} = {_fmt(row[stat])}\n{row['class']} · {size}"

    fig, axes = plt.subplots(3, n, figsize=(2.4 * n, 2.8 * 3), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes[0], ordered.head(n).iterrows(), strict=False):
        _photo(ax, row, caption("lowest", row))
    _photo(axes[1, n // 2], _closest_to_median(stat), caption("median", _closest_to_median(stat)))
    for ax, (_, row) in zip(axes[2], ordered.tail(n)[::-1].iterrows(), strict=False):
        _photo(ax, row, caption("highest", row))
    fig.suptitle(f"{stat}: the {n} lowest, the typical (median) photo, the {n} highest", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_min_median_max(stats: list[str]) -> None:
    """Per statistic: the photo with its minimum, a typical one (median) and the maximum; two per row.

    Two extra blocks show photos with an odd value: colourfulness close to 0, stored as `L` or as `RGB`.
    """
    spec = []
    for stat in stats:
        items = [
            ("min", image_stats.loc[image_stats[stat].idxmin()]),
            ("median", _closest_to_median(stat)),
            ("max", image_stats.loc[image_stats[stat].idxmax()]),
        ]
        spec.append((stat, [(f"{tag} = {_fmt(row[stat])}\n{row['class']}", row) for tag, row in items]))
    no_colour = image_stats["colourfulness"] < 1
    for mode, label in (("L", "colourfulness ≈ 0\nstored as L"), ("RGB", "colourfulness ≈ 0\nstored as RGB")):
        odd = image_stats[no_colour & (image_stats["mode"] == mode)].head(3)
        if len(odd):
            items = [(f"{_fmt(r['colourfulness'])}\n{r['class']}", r) for _, r in odd.iterrows()]
            spec.append((label, items))
    n_rows = -(-len(spec) // 2)  # two blocks of three photos side by side, a narrow gap between them
    widths = {"width_ratios": [1, 1, 1, 0.45, 1, 1, 1]}
    fig, axes = plt.subplots(n_rows, 7, figsize=(13, 2.0 * n_rows), squeeze=False, gridspec_kw=widths)
    for ax in axes.ravel():
        ax.axis("off")
    for k, (label, items) in enumerate(spec):
        r, side = divmod(k, 2)
        for j, (caption, row) in enumerate(items):
            _photo(axes[r, side * 4 + j], row, caption)
        axes[r, side * 4].set_ylabel(label, color=INK, fontsize=10, fontweight="bold")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
# @title Measure every photo once and keep the numbers in image_stats { display-mode: "form" }
# (a few seconds: each JPEG is decoded at thumbnail size)
THUMB = 64 if SMOKE else 128  # smaller thumbnails in smoke mode, same code (N-014)

_records, _hists = [], []
for _, _row in images.iterrows():
    _stats, _hist = measure_image(_row["path"], THUMB)
    _records.append({"id": _row["id"], "class": _row["class"], "path": _row["path"], **_stats})
    _hists.append(_hist)
image_stats = pd.DataFrame(_records)
CHANNEL_HISTS = np.stack(_hists)  # (n_images, 3, HIST_BINS)
RESULTS["image_stats_n"] = int(len(image_stats))

_same_size = image_stats[["width", "height"]].to_numpy() == images[["width", "height"]].to_numpy()
_agree = int(_same_size.all(axis=1).sum())
print(f"measured {len(image_stats)} photos · the manifest's width/height matches the file for {_agree}")
STAT_COLUMNS = ["width", "height", "aspect", "megapixels", "brightness", "contrast", "colourfulness"]
image_stats[STAT_COLUMNS].agg(["min", "median", "mean", "max"]).T.round(2)

💡 Read the `min` and `max` of every row first: a width of 0, a maximum of 6000 or a brightness of 254 each
tell a story. A
number is a summary; the photos behind it are the evidence. The next cell shows the photo with the minimum,
a typical one (the median) and the maximum of every statistic.

In [ ]:
# @title Show the min, median and max photo of every statistic { display-mode: "form" }
show_min_median_max(STAT_COLUMNS)

💡 Look at the median photo first: that is a typical photo of this dataset. The extremes show how far it
stretches. The last blocks hold photos with *no colour at all*, stored in two different ways.

### Sizes and shapes

A model sees every photo at one fixed size, so the first question is how far the photos are from that.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Predict before you run. These photos were uploaded to Flickr by hundreds of people and collected by the COCO
team; the park's photos will come from a few fixed cameras.</p>
</div>""")

<callout: questions>

**Q4.** 🟢 Do you expect all photos to have the same size? What shape would the dots of a width-vs-height
scatter make: one point, a line, or a cloud? Pick the closest one (the real plot may not match any exactly),
then run the cell.

In [ ]:
# @title Q4 hint: three possible scatter shapes { display-mode: "form" }
# Three tiny example scatters of width (across) against height (up), to pick from.
_rng = np.random.default_rng(SEED)
_line_w = _rng.uniform(300, 640, 40)
_examples = {
    "one point\nall photos the same size": (np.full(40, 640.0), np.full(40, 480.0)),
    "a line\nsizes differ, same shape": (_line_w, _line_w * 0.75),
    "a cloud\nany size, any shape": (_rng.uniform(200, 640, 40), _rng.uniform(200, 640, 40)),
}
fig, axes = plt.subplots(1, 3, figsize=(8.4, 2.6))
for ax, (title, (xs, ys)) in zip(axes, _examples.items(), strict=True):
    ax.scatter(xs, ys, s=14, color=INK, alpha=0.7, edgecolor="none")
    ax.set_xlim(0, 700)
    ax.set_ylim(0, 700)
    ax.set_xlabel("width (px)", fontsize=8)
    ax.set_ylabel("height (px)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_title(title, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title Q4 Plot image sizes and aspect ratios by class { display-mode: "form" }
# Left: every photo as a dot (width across, height up) in its class colour; the dashed line is a square
# photo. Right: the aspect ratio width/height, stacked by class.
fig, (ax_sc, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
for c in CLASSES:
    s = image_stats[image_stats["class"] == c]
    ax_sc.scatter(s["width"], s["height"], s=22, alpha=0.65, color=CLASS_COLORS[c], label=c, edgecolor="none")
_lim = float(max(image_stats["width"].max(), image_stats["height"].max())) * 1.08
ax_sc.plot([0, _lim], [0, _lim], "--", color=INK, lw=1)
ax_sc.set_xlim(0, _lim)
ax_sc.set_ylim(0, _lim)
ax_sc.set_xlabel("width (px)")
ax_sc.set_ylabel("height (px)")
ax_sc.set_title("Image sizes")
ax_sc.legend(frameon=False, fontsize=9, markerscale=1.6)

_bins = np.linspace(image_stats["aspect"].min() * 0.98, image_stats["aspect"].max() * 1.02, 28)
ax_hist.hist(
    [image_stats.loc[image_stats["class"] == c, "aspect"] for c in CLASSES],
    bins=_bins,
    stacked=True,
    color=[CLASS_COLORS[c] for c in CLASSES],
    label=CLASSES,
)
for ratio, name in [(3 / 4, "3:4"), (1.0, "1:1"), (4 / 3, "4:3")]:
    ax_hist.axvline(ratio, color=INK, lw=0.8, ls=":")
    ax_hist.text(ratio, ax_hist.get_ylim()[1] * 0.97, name, ha="center", va="top", fontsize=8, color=INK)
ax_hist.set_xlabel("aspect ratio = width / height  (left of 1: portrait, right: landscape)")
ax_hist.set_ylabel("photos")
ax_hist.set_title("Aspect ratios")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q4)</summary>
<p style="margin:6px 0 0 0;">Closest to a line: two lines meeting in an L. COCO fetched its Flickr photos at
one standard size, so the longest side is capped: width = cap for landscape photos, height = cap for
portrait ones (the aspect histogram shows the same two peaks). At the park, each camera model would give one
dot, and a model could learn "this size = this camera = this animal" (leakage, section 6).</p>
</details>
</div>""")

<callout: answer>

### Colour modes and colours

How a file stores colour is easy to forget and painful to discover during training: one grayscale (`L`) or
`RGBA` file in a batch of `RGB` arrays crashes the data loader, or silently gets a channel duplicated or
dropped. The next cell counts the modes, shows the photos that look grayscale, and
draws the typical RGB histogram of each class.

In [ ]:
# @title Count colour modes, show gray photos, plot RGB histograms { display-mode: "form" }
# Colour modes per class (always listing RGB / L / RGBA / CMYK), the grayscale-looking photos, then the mean
# R, G, B histogram per class.
_modes = COLOUR_MODES + sorted(set(image_stats["mode"]) - set(COLOUR_MODES))
mode_by_class = pd.crosstab(image_stats["mode"], image_stats["class"])
mode_by_class = mode_by_class.reindex(index=_modes, columns=CLASSES, fill_value=0)
image_stats["looks_gray"] = (image_stats["mode"] == "L") | (image_stats["colourfulness"] < 2.0)
print("colour modes: " + " · ".join(f"{m} {int(mode_by_class.loc[m].sum())}" for m in _modes))
_n_gray = int(image_stats["looks_gray"].sum())
print(f"photos that look grayscale (mode L, or RGB with colourfulness below 2): {_n_gray}")
N_GRAY = 6 if SMOKE else 12  # how many grayscale-looking photos to show (N-014); there may be none
show_rows(image_stats[image_stats["looks_gray"]].head(N_GRAY), "photos that look grayscale")

fig = plt.figure(figsize=(13, 4.6))
_gs = fig.add_gridspec(
    2, 4, width_ratios=[1.4, 1, 1, 1], hspace=0.65, wspace=0.3, left=0.05, right=0.99, top=0.86, bottom=0.12
)
ax_modes = fig.add_subplot(_gs[:, 0])
_bottom = np.zeros(len(_modes))
for c in CLASSES:
    ax_modes.bar(_modes, mode_by_class[c].to_numpy(), bottom=_bottom, color=CLASS_COLORS[c], label=c)
    _bottom += mode_by_class[c].to_numpy()
ax_modes.set_ylabel("photos")
ax_modes.set_title("Colour mode as stored in the file", fontsize=10)
ax_modes.legend(frameon=False, fontsize=8)

_edges = np.linspace(0, 256, HIST_BINS + 1)
_centres = (_edges[:-1] + _edges[1:]) / 2
for k, c in enumerate(CLASSES):
    ax = fig.add_subplot(_gs[k // 3, 1 + k % 3])
    _mean_hist = CHANNEL_HISTS[(image_stats["class"] == c).to_numpy()].mean(axis=0)
    for ch_idx, ch in enumerate("RGB"):
        ax.plot(_centres, _mean_hist[ch_idx], color=CHANNEL_COLORS[ch], lw=1.4, label=ch)
    ax.set_title(f"{c} ({int((image_stats['class'] == c).sum())} photos)", color=CLASS_COLORS[c], fontsize=10)
    ax.set_yticks([])
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(CLASS_COLORS[c])
        spine.set_linewidth(1.6)
    if k == 0:
        ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.text(0.67, 0.95, "Mean R, G, B histogram per class (pixel value 0–255)", ha="center", color=INK)
display(fig)
plt.close(fig)

💡 The photos above are the grayscale-looking ones (stored as `L`, or as `RGB` with almost no colour).

Read the histograms as *typical colours of a class*. If the curves differ per class, a model can cheat by
using colour instead of the animal: a dog on green grass might be called "sheep".

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Grayscale files in section 5, and what a histogram shows</div>
<p style="margin:6px 0 0 0;">Section 5 keeps the grayscale photos: files stored as <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">L</code> get a note in a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">flags</code> column, and every photo is
opened with <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">.convert("RGB")</code>, so a grayscale file becomes three equal channels instead of crashing a batch.</p>
<p style="margin:6px 0 0 0;">A strong blue channel suggests a pond, high values across all three a sandy savanna.</p>
</div>""")

<callout: go-deeper>

### Brightness and the extremes

The extremes of a statistic are the fastest route to broken or odd files. Numbers point, eyes decide:
`show_extremes(stat, n)` shows the `n` photos at both ends of any column of `image_stats`, with the median
photo in the middle for comparison.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · Look at the extremes</div>
<p style="margin:6px 0 0 0;"><strong>Look for:</strong> which class has the darkest photos, and are they broken or just dusk?</p>
<p style="margin:6px 0 0 0;">Set <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">STAT_TO_SHOW</code> (<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">brightness, contrast, colourfulness, aspect, megapixels, long_side</code>) and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">N_EXTREMES</code>
(how many photos per end), then run the cell. Try <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"brightness"</code> first, then <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"colourfulness"</code> (the low end
is where grayscale pictures hide) and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"aspect"</code>.</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
STAT_TO_SHOW = "brightness"
N_EXTREMES = 4

if not isinstance(N_EXTREMES, int) or N_EXTREMES < 1:
    raise SystemExit(f"N_EXTREMES must be a positive integer, not {N_EXTREMES!r}")
n_extremes = min(N_EXTREMES, 3) if SMOKE else N_EXTREMES  # smoke mode caps the gallery, same code (N-014)
show_extremes(STAT_TO_SHOW, n_extremes)

💡 The darkest photos are usually real (an animal in shade, a night shot with flash) and worth keeping: they
resemble the park's dusk frames. The brightest are often snow, sand or an over-exposed sky.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Which extremes are a cleaning job</div>
<p style="margin:6px 0 0 0;">A uniform grey rectangle or a photo of a screen at either end is a file for section 5's cleaning list.</p>
</div>""")

<callout: go-deeper>

### 🔴 Optional · Brightness per class as a violin

The brightness distribution tells you whether the dataset has night shots at all. To read a violin: the
wider the shape, the more photos at that value; the line marks the median.

In [ ]:
# @title Plot brightness per class as violins { display-mode: "form" }
# Brightness per class as violin plots, in class colours. The dotted lines mark "very dark" and "very bright".
fig, ax = plt.subplots(figsize=(9, 3.8))
_data = [image_stats.loc[image_stats["class"] == c, "brightness"].to_numpy() for c in CLASSES]
_violins = ax.violinplot(_data, widths=0.8, showmedians=True)
for body, c in zip(_violins["bodies"], CLASSES, strict=True):
    body.set_facecolor(CLASS_COLORS[c])
    body.set_alpha(0.85)
_violins["cmedians"].set_color(INK)
ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES)
for level in (40, 215):
    ax.axhline(level, color=INK, lw=0.8, ls=":")
ax.set_ylim(0, 255)
ax.set_ylabel("brightness (mean grey level)")
ax.set_title("Brightness per class")
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class covers most of the range, but the medians sit in the middle: daytime holiday photos. The
park's cameras also run at dusk and at night, which this data cannot teach.

### Across tasks: what resizing does to boxes and masks

Everything above was about the photo. The park's second and third jobs (count, outline) attach **boxes** and
**masks** to it, and the first thing every training pipeline does is resize.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · Resizing</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Classification:</strong> squashing a 4:3 photo into a 224×224 square distorts every photo the same way: usually
  harmless.</li>
<li style="margin:2px 0;"><strong>Detection:</strong> a box is in pixels, so rescale it with the image: x and width by <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">new_width / width</code>, y and
  height by <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">new_height / height</code>.</li>
<li style="margin:2px 0;"><strong>Segmentation:</strong> a mask holds class ids, not colours: smooth (bilinear) resizing invents in-between
  values that are no class, so masks need <strong>nearest-neighbour</strong> resizing.</li>
</ul>
</div>""")

<callout: across-tasks>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Crop, pad, and a forgotten rescale</div>
<p style="margin:6px 0 0 0;">Cropping or padding keeps the photo's shape at the cost of pixels or borders. Forget to rescale the boxes
and "all the boxes sit in the top-left corner".</p>
</div>""")

<callout: go-deeper>

**Q5.** 🟡 A 640×427 photo is resized to 224×224. Answer both parts, then run the cell to check them:

- **(a) Box:** the photo holds a box `(x, y, w, h) = (100, 50, 200, 150)` in pixels. What is the new box?
- **(b) Mask:** its mask holds only 0 (background) and 1 (animal). How many distinct values does it hold
  after a nearest-neighbour resize, and after a bilinear one?

In [ ]:
# @title Q5 Rescale a box, resize a mask two ways and show both as heatmaps { display-mode: "form" }
TARGET = 224
_w0, _h0, _box0 = 640, 427, (100, 50, 200, 150)  # the numbers of the question
_new0 = (_box0[0] * TARGET / _w0, _box0[1] * TARGET / _h0, _box0[2] * TARGET / _w0, _box0[3] * TARGET / _h0)
print(f"question: {_w0}×{_h0} → {TARGET}×{TARGET}, box {_box0} → ({', '.join(f'{v:.1f}' for v in _new0)})")

# One real photo from the Play Dataset, with its box and mask:
_polygon = annotations["segmentation"].apply(lambda s: isinstance(s, list))
_ann = annotations[(annotations["iscrowd"] == 0) & _polygon].iloc[0]
_sample = images.loc[images["id"] == _ann["image_id"]].iloc[0]  # not `_photo`: that is a helper above
_color = CLASS_COLORS[_ann["class"]]

with Image.open(_sample["path"]) as _im:
    original = _im.convert("RGB")
W, H = original.size
sx, sy = TARGET / W, TARGET / H
x, y, w, h = _ann["bbox"]
new_box = (x * sx, y * sy, w * sx, h * sy)

mask = Image.new("L", (W, H), 0)
for _poly in _ann["segmentation"]:
    ImageDraw.Draw(mask).polygon(list(zip(_poly[::2], _poly[1::2], strict=True)), fill=1)
squashed = original.resize((TARGET, TARGET), Image.BILINEAR)
mask_nearest = np.asarray(mask.resize((TARGET, TARGET), Image.NEAREST))
mask_bilinear = np.asarray(mask.point(lambda v: v * 255).resize((TARGET, TARGET), Image.BILINEAR)) / 255.0

print(f"photo {W}×{H} → {TARGET}×{TARGET}: scale x by {sx:.3f}, y by {sy:.3f}")
_old = ", ".join(f"{v:.0f}" for v in (x, y, w, h))
_new = ", ".join(f"{v:.1f}" for v in new_box)
print(f"box  (x, y, w, h) = ({_old})  →  ({_new})")
print(
    f"mask values: original {np.unique(np.asarray(mask)).tolist()} · "
    f"nearest {np.unique(mask_nearest).tolist()} · bilinear {len(np.unique(mask_bilinear))} distinct values"
)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.9), layout="constrained")
axes[0].imshow(original)
axes[0].add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=_color, lw=2))
axes[0].set_title(f"original {W}×{H} + box", fontsize=9)
axes[1].imshow(squashed)
axes[1].add_patch(Rectangle(new_box[:2], new_box[2], new_box[3], fill=False, edgecolor=_color, lw=2))
axes[1].set_title(f"resized {TARGET}×{TARGET} + rescaled box", fontsize=9)
axes[2].imshow(mask_nearest, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title(f"mask, nearest: {len(np.unique(mask_nearest))} values", fontsize=9)
_heat = axes[3].imshow(mask_bilinear, cmap="viridis", vmin=0, vmax=1)
axes[3].set_title(f"mask, bilinear: {len(np.unique(mask_bilinear))} values", fontsize=9)
for ax in axes:
    frame(ax, _color, width=2)
fig.colorbar(_heat, ax=axes[2:], fraction=0.025, pad=0.02, label="mask value (0 = background, 1 = animal)")
fig.suptitle(f"{_ann['class']}: what one resize does to the photo, its box and its mask (masks as heatmaps)",
             color=INK)
display(fig)
plt.close(fig)

# Zoom on 6×6 pixels at the animal's edge, with the value of every pixel written in.
_edge = np.argwhere((mask_bilinear > 0.3) & (mask_bilinear < 0.7))
if len(_edge):
    _r0, _c0 = (int(np.clip(v - 3, 0, TARGET - 6)) for v in _edge[len(_edge) // 2])
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.9))
    for ax, data, name in ((axes[0], mask_nearest, "nearest"), (axes[1], mask_bilinear, "bilinear")):
        patch = data[_r0 : _r0 + 6, _c0 : _c0 + 6]
        ax.imshow(patch, cmap="viridis", vmin=0, vmax=1)
        for (i, j), v in np.ndenumerate(patch):
            ink = "white" if v < 0.6 else "black"  # readable on dark and on light cells
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color=ink)
        ax.set_title(f"{name}: 6×6 pixels at the edge", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q5)</summary>
<p style="margin:6px 0 0 0;"><strong>(a)</strong> New box = <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">(x · 224/W, y · 224/H, w · 224/W, h · 224/H)</code>: here x and w
times 224/640 = 0.35, y and h times 224/427 = 0.525, giving <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">(35, 26.2, 70, 78.7)</code>, squashed like the photo.
<strong>(b)</strong> Nearest-neighbour keeps the two values (0 and 1); bilinear invents many in-between values along
every edge (the rim in the heatmap), so thin parts (legs, tails, a beak) suffer most.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Where the in-between values come from</div>
<p style="margin:6px 0 0 0;">Bilinear interpolation gives every new pixel a weighted average of the four nearest old pixels,
<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">v00 v10 v01 v11</code>:</p>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">new = (1−a)(1−b)·v00 + a(1−b)·v10 + (1−a)b·v01 + a·b·v11</code></p>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">a</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">b</code> (0 to 1) say how far the new pixel lies from them. At an edge some of the four are 0 and some
are 1, so the result is a fraction such as 0.25 or 0.5.</p>
<p style="margin:6px 0 0 0;"><strong>Why not just round the fractions?</strong> With 0 = background and 1 = dog, a pixel of 0.5 is neither. With
several class ids (0 background, 1 cat, 2 dog), an edge between background (0) and dog (2) averages to 1,
which is a <em>cat that is not there</em>. Nearest-neighbour copies one existing neighbour, so it can only produce
ids that exist: photos get a smooth filter, masks get nearest-neighbour.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a size, shape or mode that most photos share and a few do not → <strong>do</strong> ask what source
  made the others different, then keep, convert to RGB, or drop them.</li>
<li style="margin:2px 0;"><strong>When you read</strong> a statistic → <strong>do</strong> look at the min, median and max photo behind it.</li>
<li style="margin:2px 0;"><strong>When you resize</strong> photos for training → <strong>do</strong> rescale boxes by the same x and y factors, and resize
  masks with nearest-neighbour.</li>
</ul>
</div>""")

<callout: takeaway>

## 4. Labels & class distribution

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Count the photos of each class, see why a photo with several animals is a labelling problem, and
understand why the dog alert cares about imbalance more than about accuracy.</p>
</div>""")

<callout: objective>

This section looks at the **labels**, the one word per photo: how many photos of each class, and is "one
word per photo" the right shape for this data? Karpathy looks for this at the same point:

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-recipe" style="background:#E8F5E9;color:#1B5E20;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🧾 Recipe · Step 1, continued: imbalances and biases</div>
<p style="margin:6px 0 0 0;"><em>"Luckily, your brain is pretty good at this. One time I discovered that the data contained duplicate
examples. Another time I found corrupted images / labels. I look for data imbalances and biases."</em></p>
<p style="margin:6px 0 0 0;">Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">A Recipe for Training Neural
Networks</a> (2019).</p>
</div>""")

<callout: recipe>

Duplicates and corrupted files are section 5's job. Here we take the last sentence: imbalances and biases.

### Images per class

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Predict before you run. Section 1's checklist printed the per-class counts once; now you see them as a bar
chart, with each class's share of the dataset.</p>
</div>""")

<callout: questions>

**Q6.** 🟢 Which class do you expect to have the most photos, which the fewest, and roughly how many times
larger is the biggest class than the smallest? Write down your guess, then run the cell.

In [ ]:
# @title Q6 Photos per class as bars, with each class's share written on the bar { display-mode: "form" }
images_per_class = {c: int((images["class"] == c).sum()) for c in CLASSES}
RESULTS["images_per_class"] = images_per_class

_total = sum(images_per_class.values())
_largest = max(images_per_class, key=images_per_class.get)
_smallest = min(images_per_class, key=images_per_class.get)
_ratio = images_per_class[_largest] / max(1, images_per_class[_smallest])
print(f"{_total} photos · largest class {_largest} ({images_per_class[_largest]}) · smallest {_smallest} "
      f"({images_per_class[_smallest]}) · ratio {_ratio:.1f}×")

fig, ax = plt.subplots(figsize=(9, 3.8))
_order = sorted(CLASSES, key=images_per_class.get, reverse=True)
ax.bar(_order, [images_per_class[c] for c in _order], color=[CLASS_COLORS[c] for c in _order])
for i, c in enumerate(_order):
    ax.text(i, images_per_class[c], f"{100 * images_per_class[c] / _total:.0f} %", ha="center", va="bottom",
            fontsize=9)
ax.set_ylabel("photos")
ax.set_title("Images per class (one label per photo)")
ax.set_ylim(0, max(images_per_class.values()) * 1.15)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q6)</summary>
<p style="margin:6px 0 0 0;">On the full Play Dataset cat and dog are the largest classes (about 130 photos
each) and sheep the smallest (about 50): roughly 2–3×. People photograph pets, not sheep; that is what
Flickr uploads look like, not what the park's cameras will see. (The smoke sample is flat.)</p>
</details>
</div>""")

<callout: answer>

Look at the ratio, then ask whether the smallest class is the one you care about. Dog is common in this
dataset but will be rare in the paddock.

### Several animals in one photo: which label then?

`images` has one `class` per photo, yet `n_animals` is often above 1. That is fine while all the animals
are the same class. The Play Dataset keeps only COCO photos with **exactly one animal class**: a decision
someone took *for* you. Check what it hides.

**Q7.** 🟡 What share of the photos do you think hold more than one animal, and which class do you expect to
be the most "crowded"? Then run the cell: it counts animals per photo and shows the most crowded photos.

In [ ]:
# @title Q7 Animals per photo, stacked by class, and the most crowded photos { display-mode: "form" }
_max_n = int(images["n_animals"].max())
_several = images[images["n_animals"] > 1]
print(f"{len(_several)} of {len(images)} photos ({100 * len(_several) / len(images):.0f} %) hold more than "
      f"one animal; the most crowded photo has {_max_n}.")
print("mean animals per photo: " + " · ".join(
    f"{c} {images.loc[images['class'] == c, 'n_animals'].mean():.1f}" for c in CLASSES))

fig, ax = plt.subplots(figsize=(9, 3.8))
_bins = np.arange(0.5, min(_max_n, 10) + 1.5, 1)
_per_class = [images.loc[images["class"] == c, "n_animals"].clip(upper=10) for c in CLASSES]
ax.hist(_per_class, bins=_bins, stacked=True, color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.set_xticks(range(1, min(_max_n, 10) + 1))
ax.set_xticklabels([str(i) if i < 10 else "10+" for i in range(1, min(_max_n, 10) + 1)])
ax.set_xlabel("animals in the photo")
ax.set_ylabel("photos")
ax.set_title("Animals per photo")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

N_CROWDED = 3 if SMOKE else 6  # how many crowded photos to show (N-014)
_crowded = images.sort_values(["n_animals", "id"], ascending=[False, True]).head(N_CROWDED)
show_rows(_crowded, "the most crowded photos", lambda r: f"{r['n_animals']} × {r['class']}", cols=N_CROWDED)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q7)</summary>
<p style="margin:6px 0 0 0;">On the full Play Dataset about a third of the photos hold more than one animal;
sheep and birds are the crowded classes (flocks), cats and dogs are mostly alone. One label per photo works
only because they share a species, and it cannot say <em>how many</em>: thirty sheep and a lamb are both "sheep".</p>
</details>
</div>""")

<callout: answer>

"Which label then?" The park picks one of three before labelling a single frame:

- **one label, the main animal** (COCO's choice): cheap, but the labeller decides what "main" means;
- **one label per class present**: "sheep and dog" is the paddock alert, but needs a different loss and
  metric;
- **one box per animal** (detection, section 7): answers "which" and "how many", at a higher cost.

The Play Dataset went further and dropped every photo with two classes. For the paddock camera, where the
case that matters is a dog **among** sheep, that filter deletes exactly the frames the alert is for.

### Why the dog alert cares about imbalance

Imbalance is harmless in a bar chart and dangerous in a metric: the paddock camera sees sheep almost all
the time and a dog very rarely.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · A paddock day</div>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">DOG_SHARE</code> is the fraction of the paddock camera's frames that show a dog (the rest show sheep). Keep
<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">0.02</code> for Q8. Afterwards: edit the value (try <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">0.2</code>, then <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">0.5</code>), run this cell, then re-run the Q8 cell.</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
DOG_SHARE = 0.02

if not isinstance(DOG_SHARE, (int, float)) or not 0 <= DOG_SHARE <= 1:
    raise SystemExit(f"DOG_SHARE must be a number between 0 and 1, not {DOG_SHARE!r}")

**Q8.** 🟡 No model is trained here; this is a thought experiment about the *dataset*. Two lazy rules that
ignore the photo, one question each. **Accuracy** is the share of photos answered correctly; **recall on
dog** is, of the real dogs, how many the rule gets right. Guess both numbers for each rule, then run:

- **Rule A, on the Play Dataset:** always answer *the biggest class* (read it off the Q6 bars). What are
  its accuracy and its recall on dog?
- **Rule B, on a paddock week:** 1000 frames, 20 of them with a dog (`DOG_SHARE = 0.02`), and the rule
  always answers "sheep". What are its accuracy and its recall on dog?

In [ ]:
# @title Q8 A lazy model stamps "sheep" on every frame: accuracy and dog recall { display-mode: "form" }
# Rule A on the whole Play Dataset: always the biggest class.
_majority = max(images_per_class, key=images_per_class.get)
_true = images["class"].to_numpy()
acc_play = float((_true == _majority).mean())
recall_dog_play = float(_majority == "dog") if (_true == "dog").any() else float("nan")
_lines = [f"Rule A · Play Dataset, 'always {_majority}': accuracy {100 * acc_play:.0f} %, recall on dog "
          f"{100 * recall_dog_play:.0f} %"]

# Rule B on a paddock day: PADDOCK_FRAMES frames, a share DOG_SHARE of them show a dog, the rest sheep.
PADDOCK_FRAMES = 200 if SMOKE else 1000
_n_dog = int(round(DOG_SHARE * PADDOCK_FRAMES))
acc_day = 1 - _n_dog / PADDOCK_FRAMES  # "always sheep" is right on every sheep frame
recall_dog_day = 0.0 if _n_dog else float("nan")  # and wrong on every dog frame
RESULTS["accuracy"], RESULTS["recall_dog"] = acc_day, recall_dog_day  # "always sheep" on the paddock day
_lines.append(f"Rule B · paddock day, {_n_dog} dog frames in {PADDOCK_FRAMES}, 'always sheep': accuracy "
              f"{100 * acc_day:.0f} %, recall on dog {100 * recall_dog_day:.0f} %")

# The same day in miniature: 12 real photos, a share DOG_SHARE of them dogs, the rest sheep.
_N_STRIP = 12
_strip_dogs = min(int(round(DOG_SHARE * _N_STRIP)) or int(DOG_SHARE > 0), images_per_class["dog"])
_rng = random.Random(SEED)
_dog_idx = _rng.sample(sorted(images.index[images["class"] == "dog"]), _strip_dogs)
_sheep_idx = _rng.sample(sorted(images.index[images["class"] == "sheep"]),
                         min(_N_STRIP - _strip_dogs, images_per_class["sheep"]))
_strip_idx = _dog_idx + _sheep_idx
_rng.shuffle(_strip_idx)
_strip = images.loc[_strip_idx]

_cols = 6
_rows = -(-len(_strip) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(2.3 * _cols, 2.5 * _rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, (_, row) in zip(axes.ravel(), _strip.iterrows(), strict=False):
    ax.axis("on")
    ax.imshow(load_thumb(row["path"]))
    frame(ax, CLASS_COLORS[row["class"]])
    _right = row["class"] == "sheep"
    ax.text(0.04, 0.96, "sheep " + ("✔" if _right else "✘"), transform=ax.transAxes, ha="left", va="top",
            fontsize=10, fontweight="bold", color="white",
            bbox={"boxstyle": "round,pad=0.25", "facecolor": CLASS_COLORS["sheep"], "edgecolor": "none"})
    ax.set_title(f"true: {row['class']}", fontsize=8, color="#2E7D32" if _right else "#C62828")
_strip_right = int((_strip["class"] == "sheep").sum())
fig.suptitle(f"A lazy model stamps 'sheep' on {len(_strip)} paddock frames: {_strip_right} right, "
             f"{len(_strip) - _strip_right} wrong", color=INK)
fig.tight_layout()
_lines.append(f"This strip: accuracy {100 * _strip_right / len(_strip):.0f} %, recall on dog 0 %. "
              "The dog-in-the-paddock alert never fires.")
answer_box("Q8 · the lazy model's numbers and a strip of real frames (open after you guessed)", _lines, fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q8)</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Rule A, Play Dataset:</strong> accuracy is the biggest class's share (about 20 % on the full set, a sixth in
  the balanced smoke sample); recall on dog is 0 %, unless dog <em>is</em> the biggest class (then 100 % on dog,
  0 % on everything else).</li>
<li style="margin:2px 0;"><strong>Rule B, paddock week:</strong> "always sheep" is right on 980 of 1000 frames: <strong>98 %</strong> accuracy and <strong>0 %</strong>
  recall on dog, all 20 dogs missed.</li>
</ul>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-pitfall" style="background:#FDECEA;color:#8B1A10;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">⚠️ Pitfall · Accuracy hides a rare class</div>
<p style="margin:6px 0 0 0;"><strong>Accuracy</strong> is the share of all photos labelled correctly, so the big classes dominate it: the rarer the
dog, the better "always sheep" looks. For a rare class, watch its <strong>recall</strong>. A balanced-looking Play
Dataset says nothing about the imbalance of the <em>deployment</em>. Class weights and oversampling are next
week's topic.</p>
</div>""")

<callout: pitfall>

### ✍️ Pick the alert metric

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">ALERT_METRIC</code></div>
<p style="margin:6px 0 0 0;">The paddock alert fires when a dog is in front of the sheep camera. Missing a dog is worse than a false
alarm (the park story, section 0). Three candidates, all on the dog class:</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>accuracy:</strong> the share of all frames answered correctly (Q8);</li>
<li style="margin:2px 0;"><strong>recall on dog:</strong> of the real dogs, how many the alert catches (Q8);</li>
<li style="margin:2px 0;"><strong>precision on dog:</strong> of the frames the alert called "dog", how many really were dogs (it counts false
  alarms).</li>
</ul>
<p style="margin:6px 0 0 0;"><strong>Do:</strong> replace the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">...</code> with <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"accuracy"</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"precision"</code> or <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"recall"</code>. The check cell says whether it
agrees and continues with the reference either way.</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✍️ YOUR CODE
ALERT_METRIC = ...  # one of "accuracy", "precision", "recall": the number the paddock alert is judged on

In [ ]:
# @title Check ALERT_METRIC against the reference choice { display-mode: "form" }
_REFERENCE_ALERT_METRIC = "recall"
_WHY = {
    "accuracy": "accuracy is dominated by the big classes: 'always sheep' scores 98 % on a paddock week and "
                "catches no dog (Q8).",
    "precision": "precision on dog counts false alarms. A false alarm costs the farmer a look at the screen; "
                 "a missed dog costs sheep.",
}
if ALERT_METRIC is ...:
    print("✍️  ALERT_METRIC: not chosen yet, using the reference choice.")
elif ALERT_METRIC not in ("accuracy", "precision", "recall"):
    raise SystemExit(f"ALERT_METRIC must be 'accuracy', 'precision' or 'recall', not {ALERT_METRIC!r}")
elif ALERT_METRIC == _REFERENCE_ALERT_METRIC:
    print(f"✅ ALERT_METRIC = {ALERT_METRIC!r} matches the reference choice.")
else:
    print(f"❌ ALERT_METRIC = {ALERT_METRIC!r}: {_WHY[ALERT_METRIC]} Using the reference.")
alert_metric = _REFERENCE_ALERT_METRIC
RESULTS["alert_metric"] = alert_metric
print(f"The paddock alert is judged on {alert_metric} on dog.")

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (alert metric)</summary>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">"recall"</code>.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">The alert exists to catch the rare event, and recall on dog is the share of real dogs it catches.</li>
<li style="margin:2px 0;">Precision on dog would matter if false alarms were expensive; accuracy rewards ignoring the dog.</li>
<li style="margin:2px 0;">Next week, when a real model is trained, this is the number to put on the validation curve.</li>
</ul>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> classes of very different size → <strong>do</strong> read the ratio between the largest and the
  smallest, and ask whether the smallest is the class your use case depends on.</li>
<li style="margin:2px 0;"><strong>When you see</strong> photos with several animals → <strong>do</strong> decide between one label, one label per class or
  one box per animal before labelling a single frame.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a high accuracy and a rare class that matters → <strong>do</strong> compute that class's recall;
  accuracy will happily ignore it.</li>
<li style="margin:2px 0;"><strong>When you see</strong> the Play Dataset's balance → <strong>do</strong> remember the deployment has its own imbalance.</li>
</ul>
<p style="margin:6px 0 0 0;">Section 5 goes after the labels and photos that are <em>wrong</em>, not just rare.</p>
</div>""")

<callout: takeaway>

## 5. Common dataset errors & cleaning

Sections 2 to 4 asked whether the data fits the task; now: is it wrong?

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Find duplicates, flipped and rotated copies and suspicious labels in a dataset, and produce a cleaned
table you can split.</p>
</div>""")

<callout: objective>

A duplicate in both train and test makes the score lie, and a wrong label teaches the model a wrong rule.
Neither shows up in a bar chart, so this section looks for them on purpose.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-recipe" style="background:#E8F5E9;color:#1B5E20;border-left:6px solid #1B5E20;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🧾 Recipe · Step 1, continued: the two bugs Karpathy names</div>
<p style="margin:6px 0 0 0;"><em>"One time I discovered that the data contained duplicate examples. Another time I found corrupted
images / labels."</em></p>
<p style="margin:6px 0 0 0;">Andrej Karpathy, <a href="https://karpathy.github.io/2019/04/25/recipe/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">A Recipe for Training Neural
Networks</a> (2019).</p>
</div>""")

<callout: recipe>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · This is ordinary: two Kaggle stories</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Datasets built with care that still shipped duplicates</summary>
<p style="margin:6px 0 0 0;"><strong>PetFinder.my Pawpularity</strong> (2021) shipped near-identical photos with <em>different</em> scores, and <strong>Cassava
Leaf Disease</strong> (2020) identical leaf photos with two different labels. Both were built with care. Yours
will have duplicates too, unless you look.</p>
</details>
</div>""")

<callout: go-deeper>

### Exact duplicates: the same bytes twice

The cheapest check first. Two files that are byte-for-byte identical have the same **MD5 hash**, so one
pass over the files finds every exact copy, whatever its name. You run it in Q9 below.

In [ ]:
# @title Hash helpers (MD5, perceptual hash, flipped and rotated variants) { display-mode: "form" }
import hashlib
import shutil
from io import BytesIO

import imagehash
from PIL import Image, ImageOps

THUMB_FOR_HASH = 256  # perceptual hashes are computed on a thumbnail: fast, and the same for every file
EXTREME_SIDE_PX = (64, 6000)  # longest side below / above this is "extreme"


def file_md5(path) -> str:
    """MD5 of the file's bytes: identical files share it, a one-pixel change breaks it."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def phash(image: Image.Image) -> imagehash.ImageHash:
    """Perceptual hash of a PIL image (64 bits). `h1 - h2` is the number of bits that differ (0–64)."""
    return imagehash.phash(image)


def variants(image: Image.Image) -> dict[str, Image.Image]:
    """The image as stored plus its mirrored, flipped and rotated versions, keyed by what was done."""
    return {
        "original": image,
        "flip_lr": ImageOps.mirror(image),
        "flip_tb": ImageOps.flip(image),
        "rot90": image.transpose(Image.Transpose.ROTATE_90),
        "rot180": image.transpose(Image.Transpose.ROTATE_180),
        "rot270": image.transpose(Image.Transpose.ROTATE_270),
    }


def hash_bits(hashes: list[imagehash.ImageHash]) -> np.ndarray:
    """Stack ImageHash objects into an (n, 64) boolean matrix, so distances can be computed in one go."""
    return np.array([h.hash.flatten() for h in hashes], dtype=bool)


def hash_distances(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """(len(a), len(b)) matrix of Hamming distances between two (n, 64) hash matrices."""
    return (a[:, None, :] != b[None, :, :]).sum(axis=-1)


def exact_duplicate_groups(paths: list) -> list[list]:
    """Groups of paths whose bytes are identical (only groups with 2+ members), in first-seen order."""
    by_md5: dict[str, list] = {}
    for p in paths:
        by_md5.setdefault(file_md5(p), []).append(p)
    return [group for group in by_md5.values() if len(group) > 1]


def is_readable(path) -> bool:
    """True if PIL can open and fully decode the file (sections 9 and 10 use it)."""
    try:
        with Image.open(path) as im:
            im.draft("RGB", (THUMB_FOR_HASH, THUMB_FOR_HASH))  # a reduced decode still reads every byte
            im.load()
        return True
    except Exception:
        return False


def flag_odd_files(stats: pd.DataFrame) -> pd.DataFrame:
    """One row per flagged file of `stats` (rows of image_stats): id, class, flags. Empty if all is well."""
    rows = []
    for _, r in stats.iterrows():
        flags = []
        if not is_readable(r["path"]):
            flags.append("unreadable")
        if r["mode"] != "RGB":
            flags.append(f"mode {r['mode']}")
        if r["long_side"] < EXTREME_SIDE_PX[0] or r["long_side"] > EXTREME_SIDE_PX[1]:
            flags.append(f"extreme size {r['width']}×{r['height']}")
        if flags:
            rows.append({"id": r["id"], "class": r["class"], "flags": ", ".join(flags)})
    return pd.DataFrame(rows, columns=["id", "class", "flags"])

### Near-duplicates: the same picture, different bytes

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · Perceptual hashing</div>
<p style="margin:6px 0 0 0;">A <strong>perceptual hash</strong> (pHash) turns a photo into 64 bits that describe its coarse shapes: shrink it, drop
the colour, keep the big blobs, write 1 or 0 for each. A resized or re-compressed copy keeps the same big
blobs, so it gets almost the same bits.</p>
<p style="margin:6px 0 0 0;">The <strong>Hamming distance</strong> is how many of the 64 bits differ: 0 = same picture, about 32 = unrelated. What
counts as "the same picture" is a threshold you choose.</p>
</div>""")

<callout: theory>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · How the 64 bits are made</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">The recipe, step by step (the figure below shows it)</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">Shrink to 32×32 and drop the colour, so size and JPEG noise disappear (<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">imagehash</code> package).</li>
<li style="margin:2px 0;">Describe the picture as waves (a DCT), keep the 64 slowest ones (the big blobs), and write 1 where a wave
  is above the median, otherwise 0.</li>
</ul>
</details>
</div>""")

<callout: go-deeper>

In [ ]:
# @title How the 64 bits are made, and why a near copy gets almost the same ones { display-mode: "form" }
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle
from scipy.fft import dctn

HIGHLIGHT = "#E67E22"  # differing bits
_c0 = images["class"].iloc[0]
_pool = images[images["class"] != _c0]
_orig = load_thumb(images["path"].iloc[0], THUMB_FOR_HASH)
_unrelated = load_thumb((_pool if len(_pool) else images)["path"].iloc[-1], THUMB_FOR_HASH)
_buf = BytesIO()
_orig.resize((int(_orig.width * 0.7), int(_orig.height * 0.7))).save(_buf, "JPEG", quality=55)
_near = Image.open(_buf).convert("RGB")  # a smaller, re-compressed copy: different bytes, same picture

_grey = _orig.convert("L").resize((32, 32), Image.Resampling.LANCZOS)
_dct = dctn(np.asarray(_grey, dtype=float), type=2)
_bits = {k: phash(im).hash for k, im in (("orig", _orig), ("near", _near), ("other", _unrelated))}


def _draw_bits(ax, bits, other=None, title=""):
    """An 8×8 grid of 0/1; cells that differ from `other` get an orange frame."""
    ax.imshow(bits, cmap=ListedColormap(["#EAF1F2", "#5B8E96"]), vmin=0, vmax=1)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    for (r, c), b in np.ndenumerate(bits):
        ax.text(c, r, str(int(b)), ha="center", va="center", fontsize=7, color="white" if b else INK)
        if other is not None and b != other[r, c]:
            ax.add_patch(Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec=HIGHLIGHT, lw=2.5))
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 4, figsize=(10, 5.2))
axes[0, 0].imshow(_orig)
axes[0, 0].set_title("1. the photo", fontsize=9)
axes[0, 1].imshow(_grey, cmap="gray")
axes[0, 1].set_title("2. shrink to 32×32, grayscale", fontsize=9)
axes[0, 2].imshow(np.log1p(np.abs(_dct)), cmap="viridis")
axes[0, 2].add_patch(Rectangle((-0.5, -0.5), 8, 8, fill=False, ec=HIGHLIGHT, lw=2.5))
axes[0, 2].set_title("3. DCT: keep the top-left 8×8\n(the coarse shapes)", fontsize=9)
_draw_bits(axes[0, 3], _bits["orig"], title="4. above the median? 1 / 0\n= 64 bits")
for ax in axes[0, :3]:
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
_d_near, _d_other = int((_bits["orig"] != _bits["near"]).sum()), int((_bits["orig"] != _bits["other"]).sum())
_draw_bits(axes[1, 0], _bits["orig"], title="original")
_draw_bits(axes[1, 1], _bits["near"], _bits["orig"], f"re-compressed copy:\n{_d_near} bits differ")
_draw_bits(axes[1, 2], _bits["other"], _bits["orig"], f"unrelated photo:\n{_d_other} bits differ")
axes[1, 3].axis("off")
_note = "orange = a bit that differs\nHamming distance =\nnumber of orange cells\n\n"
_note += "near copy: small\nunrelated: about 32"
axes[1, 3].text(0, 0.5, _note, fontsize=10, va="center")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · The duplicate threshold</div>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">HASH_THRESHOLD</code> is the largest Hamming distance (0–64) at which two photos count as near-duplicates.
Keep the default <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">8</code> for Q9 and look at the closest pairs. Then edit the value (a strict <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">2</code>, a loose
<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">14</code>), run this cell, and re-run the Q9 cell below to see what enters the list. Set it back to <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">8</code>
before going on: later cells use it too.</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
HASH_THRESHOLD = 8

if not isinstance(HASH_THRESHOLD, int) or not 0 <= HASH_THRESHOLD <= 64:
    raise SystemExit(f"HASH_THRESHOLD must be a whole number between 0 and 64, not {HASH_THRESHOLD!r}")

**Q9.** 🟡 Two guesses, then run the cell below.

- Exact copies: COCO val2017 is a curated benchmark that thousands of papers evaluate on. How many
  byte-identical pairs are in the Play Dataset:
  none, a handful or many?
- Near copies: with your threshold, how many pairs do you expect to be flagged, and will the closest ones
  be true copies or two photos of the same kind of scene?

In [ ]:
# @title Q9 MD5 of every file (exact copies), then pHash of every photo + variants { display-mode: "form" }
exact_groups = exact_duplicate_groups(images["path"].tolist())
print(f"{len(images)} files · {len(exact_groups)} groups of byte-identical files")
for _group in exact_groups[:5]:
    print("  ", " = ".join(Path(p).name for p in _group))

VARIANT_NAMES = list(variants(Image.new("RGB", (4, 4))))
_per_variant: dict[str, list] = {v: [] for v in VARIANT_NAMES}
for _p in images["path"]:
    _thumb = load_thumb(_p, THUMB_FOR_HASH)
    for _v, _im in variants(_thumb).items():
        _per_variant[_v].append(phash(_im))
VARIANT_HASHES = {v: hash_bits(hs) for v, hs in _per_variant.items()}  # each (n_images, 64)
HASHES = VARIANT_HASHES["original"]

_dist = hash_distances(HASHES, HASHES)
np.fill_diagonal(_dist, 99)
nearest_distance = _dist.min(axis=1)  # each photo's distance to its closest other photo
_i, _j = np.where(np.triu(_dist <= HASH_THRESHOLD, k=1))
near_pairs = pd.DataFrame({"a": images["id"].to_numpy()[_i], "b": images["id"].to_numpy()[_j],
                           "distance": _dist[_i, _j]}).sort_values("distance").reset_index(drop=True)
print(f"pairs within {HASH_THRESHOLD} bits: {len(near_pairs)} · median nearest-neighbour distance: "
      f"{np.median(nearest_distance):.0f} bits")

fig, ax = plt.subplots(figsize=(9, 3.4))
_bins = np.arange(-0.5, 41, 2)
ax.hist([nearest_distance[(images["class"] == c).to_numpy()] for c in CLASSES], bins=_bins, stacked=True,
        color=[CLASS_COLORS[c] for c in CLASSES], label=CLASSES)
ax.axvline(HASH_THRESHOLD + 0.5, color=INK, ls=":", lw=1)
ax.text(HASH_THRESHOLD + 1, ax.get_ylim()[1] * 0.9, f"HASH_THRESHOLD = {HASH_THRESHOLD}", fontsize=9)
ax.set_xlabel("Hamming distance to the closest other photo (bits)")
ax.set_ylabel("photos")
ax.set_title("How close is each photo's nearest neighbour?")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

# The closest pairs, side by side, whether or not they pass the threshold: this is what "near" looks like.
N_PAIRS_TO_SHOW = 2 if SMOKE else 4
_upper = np.where(np.triu(np.ones_like(_dist, dtype=bool), k=1), _dist, 99)  # each pair once
_flat = np.argsort(_upper, axis=None)[:N_PAIRS_TO_SHOW]
_closest = list(zip(*np.unravel_index(_flat, _dist.shape), strict=True))
if len(near_pairs):
    display(near_pairs.head(10))
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in _closest])
_dist_of = {row_id: _dist[a, b] for a, b in _closest for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, "closest pairs by perceptual hash (each pair side by side)",
          lambda r: f"{r['class']} · {_dist_of[r['id']]} bits apart", cols=2)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q9)</summary>
<p style="margin:6px 0 0 0;">Exact copies: COCO val2017 was deduplicated by its authors, so expect none or
close to it. The park's re-exported cards will not be so tidy, so run the MD5 check on every folder you get.
But MD5 only sees identical bytes: re-save a photo at another JPEG quality and the hash changes completely,
which is why we need a hash that looks at the picture.</p>
<p style="margin:6px 0 0 0;">Near copies: unrelated pictures sit about 30 bits apart, far right of the threshold. Most closest pairs on
COCO are different photos of a similar scene (two zebras on grass), so the default catches one true pair on
the full Play Dataset: two frames of the same tabby in front of one brick wall (the smoke sample may lack
it). A loose threshold adds non-copies, a strict one misses re-compressed copies, so read the list by eye
before deleting.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-pitfall" style="background:#FDECEA;color:#8B1A10;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">⚠️ Pitfall · A hash distance is not a similarity score</div>
<p style="margin:6px 0 0 0;">Two burst frames of the same sheep can be 12 bits apart and two unrelated photos 10. The threshold only
separates "same picture, different file" from everything else; it does not rank how alike two photos are.</p>
</div>""")

<callout: pitfall>

### A memory card from the park: the guided hunt

The park's intern hands you a memory card of photos. Over the years five things happened to it:

1. The camera software saved one photo **twice**. Two files, identical bytes.
2. Someone exported a **smaller, lower-quality** copy for an e-mail. Different bytes, same picture.
3. Someone **mirrored** a photo and saved it. The animal now faces the other way: every pixel has moved.
4. Someone **rotated** a photo and saved it. Same story, turned on its side.
5. Every file got a neutral name (`card_017.jpg`), so nothing says which files belong together.

Find every pair without opening 40 photos one by one. **The answer key comes after the hunt.**

<details><summary>Where the card lives</summary>

The next cell writes the card into `data/cache/hunt5/` (seeded: everyone gets the same one). The fifth
thing, the neutral file names, is why you need a detector at all.
</details>

In [ ]:
# @title Plant the memory card (seeded, at runtime; nothing is redistributed) { display-mode: "form" }
HUNT_DIR = DATA_DIR / "cache" / "hunt5"
N_CARD_ORIGINALS = 12 if SMOKE else 40  # distinct photos on the card (N-014: fewer in smoke mode)
PLANT_KINDS = ["exact", "exact", "flipped", "rotated", "recompressed"]  # ≥ 4 planted copies, always present


def plant_hunt5(dest: Path, n_originals: int, seed: int) -> tuple[dict[str, Path], list[dict]]:
    """Write a 'memory card': n_originals distinct photos plus one planted copy per entry of PLANT_KINDS.

    Returns (files, key): files maps the card's neutral file names to paths; key lists the planted copies as
    {"copy": name, "source": name, "kind": kind}. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    chosen = images.sample(n=min(n_originals, len(images)), random_state=seed).sort_values("id")
    sources = [Path(p) for p in chosen["path"]]
    # (bytes, kind or None for an original, index of the source photo)
    entries: list[tuple[bytes, str | None, int]] = [(s.read_bytes(), None, i) for i, s in enumerate(sources)]
    planted: list[tuple[str, int]] = []
    source_idx = rng.sample(range(len(sources)), len(PLANT_KINDS))
    for kind, i in zip(PLANT_KINDS, source_idx, strict=True):
        with Image.open(sources[i]) as im:
            rgb = im.convert("RGB")
        buf = BytesIO()
        if kind == "exact":
            data = sources[i].read_bytes()
        elif kind == "flipped":
            ImageOps.mirror(rgb).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        elif kind == "rotated":
            rgb.transpose(Image.Transpose.ROTATE_90).save(buf, "JPEG", quality=92)
            data = buf.getvalue()
        else:  # recompressed: smaller and at a lower JPEG quality
            small = rgb.resize((max(8, int(rgb.width * 0.7)), max(8, int(rgb.height * 0.7))))
            small.save(buf, "JPEG", quality=55)
            data = buf.getvalue()
        entries.append((data, kind, i))
        planted.append((kind, i))
    order = list(range(len(entries)))
    rng.shuffle(order)
    names = {idx: f"card_{k:03d}.jpg" for k, idx in enumerate(order)}
    files, key = {}, []
    for idx, (data, kind, i) in enumerate(entries):
        path = dest / names[idx]
        path.write_bytes(data)
        files[names[idx]] = path
        if kind is not None:
            key.append({"copy": names[idx], "source": names[i], "kind": kind})
    return dict(sorted(files.items())), key

In [ ]:
# @title Write the memory card to disk and load its photos { display-mode: "form" }
hunt_files, HUNT5_KEY = plant_hunt5(HUNT_DIR, N_CARD_ORIGINALS, SEED)
hunt_images = {name: load_thumb(path, THUMB_FOR_HASH) for name, path in hunt_files.items()}
print(f"{len(hunt_files)} files on the card in {HUNT_DIR}/ · {len(HUNT5_KEY)} of them are copies of another "
      f"file on the card")

### ✍️ Write the flip detector yourself

A mirrored or rotated copy has moved every pixel, so its plain hash is far from the original's. Before the
shoot-out, write the one piece plain hashing lacks.

In [ ]:
# @title Reference solution — run this cell; open it only if you are stuck { display-mode: "form" }
def _ref_find_flipped_copies(image, candidates):
    flipped_hashes = [phash(im) for name, im in variants(image).items() if name != "original"]
    found = []
    for name, cand in candidates.items():
        h = phash(cand)
        if min(h - fh for fh in flipped_hashes) <= HASH_THRESHOLD:
            found.append(name)
    return sorted(found)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">find_flipped_copies</code></div>
<p style="margin:6px 0 0 0;">Fill in the function below: given one photo and a dictionary of candidate photos, return the names of the
candidates that are a flipped or rotated copy of it, by comparing the photo's hash with the hashes of the
candidates' flipped and rotated versions. You have <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">variants(image)</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">phash(image)</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">h1 - h2</code> for the
distance and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">HASH_THRESHOLD</code>. The <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">use()</code> line checks your answer against the reference and continues
with the reference either way.</p>
</div>""")

<callout: your-turn>

In [ ]:
# ✍️ YOUR CODE
def find_flipped_copies(image, candidates):
    """Return the names of the candidates that are a flipped or rotated copy of `image`.

    image: a PIL image. candidates: dict {name: PIL image} (the image itself is not among them).
    Returns a sorted list of names whose pHash is within HASH_THRESHOLD bits of the pHash of one of
    image's flipped or rotated variants (every variant except "original").

    e.g. ["card_017.jpg", "card_031.jpg"]
    """
    ...


_flip_source = next(k["source"] for k in HUNT5_KEY if k["kind"] == "flipped")
_others = {name: im for name, im in hunt_images.items() if name != _flip_source}
flipped_copies = use(find_flipped_copies, hunt_images[_flip_source], _others,
                     reference=_ref_find_flipped_copies)
print(f"flipped/rotated copies of {_flip_source}: {flipped_copies}")

**Q10.** 🟡 The shoot-out: plain hashing (MD5 plus pHash) against *variant* hashing, the idea you just
wrote. Which of the first four things that happened to the card (twice, smaller, mirrored, rotated) does
each one catch? Predict, then run.

In [ ]:
# @title Copy finder: plain hashing vs variant hashing { display-mode: "form" }
KIND_OF_VARIANT = {"original": "recompressed", "flip_lr": "flipped", "flip_tb": "flipped",
                   "rot90": "rotated", "rot180": "rotated", "rot270": "rotated"}


def find_copies(files: dict[str, Path], threshold: int, use_variants: bool) -> pd.DataFrame:
    """Pairs of files on a card that are copies of each other: columns a, b, kind, distance.

    kind is "exact" (same bytes), "recompressed" (hashes within `threshold`), or, with use_variants=True,
    "flipped" / "rotated" (a flipped or rotated variant of a hashes within `threshold` of b).
    """
    names = list(files)
    md5s = {n: file_md5(p) for n, p in files.items()}
    thumbs = {n: load_thumb(p, THUMB_FOR_HASH) for n, p in files.items()}
    plain = hash_bits([phash(thumbs[n]) for n in names])
    variant_names = VARIANT_NAMES if use_variants else ["original"]
    # distances[v][i, j]: hash of variant v of file i vs the plain hash of file j
    distances = {v: hash_distances(hash_bits([phash(variants(thumbs[n])[v]) for n in names]), plain)
                 for v in variant_names}
    rows = []
    for i, a in enumerate(names):
        for j in range(i + 1, len(names)):
            b = names[j]
            if md5s[a] == md5s[b]:
                rows.append({"a": a, "b": b, "kind": "exact", "distance": 0})
                continue
            best_v = min(variant_names, key=lambda v: distances[v][i, j])
            best_d = int(distances[best_v][i, j])
            if best_d <= threshold:
                rows.append({"a": a, "b": b, "kind": KIND_OF_VARIANT[best_v], "distance": best_d})
    return pd.DataFrame(rows, columns=["a", "b", "kind", "distance"])


def score_hunt(found: pd.DataFrame, key: list[dict]) -> int:
    """How many planted copies in `key` appear as a pair (either order) in `found`."""
    pairs = {frozenset((r["a"], r["b"])) for _, r in found.iterrows()}
    return sum(frozenset((k["copy"], k["source"])) in pairs for k in key)

In [ ]:
# @title Q10 Plain hashing, then variant hashing, on the same card and threshold { display-mode: "form" }
plain_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=False)
variant_found = find_copies(hunt_files, HASH_THRESHOLD, use_variants=True)
print(f"plain hashing   : {len(plain_found)} pairs → {dict(Counter(plain_found['kind']))}")
print(f"variant hashing : {len(variant_found)} pairs → {dict(Counter(variant_found['kind']))}")
display(variant_found)

Why, on one photo and five kinds of copy: the pHash distance to the original, plain vs variant.

In [ ]:
# @title One photo, five kinds of copy: pHash distance, plain vs variant hashing { display-mode: "form" }
_src = load_thumb(images["path"].iloc[0], THUMB_FOR_HASH)


def _reencode(im, quality):
    buf = BytesIO()
    im.save(buf, "JPEG", quality=quality)
    return Image.open(buf).convert("RGB")


_copies = {
    "original": _src,
    "exact copy": _src,
    "resized": _src.resize((int(_src.width * 0.6), int(_src.height * 0.6))),
    "flipped": ImageOps.mirror(_src),
    "re-encoded": _reencode(_src, 25),
    "rotated": _src.transpose(Image.Transpose.ROTATE_90),
}
_h0 = phash(_src)
FOUND_COLOR, MISSED_COLOR = "#1E7B6E", "#B03A2E"
fig, axes = plt.subplots(1, len(_copies), figsize=(2.3 * len(_copies), 3.3))
for ax, (name, im) in zip(axes, _copies.items(), strict=True):
    ax.imshow(im)
    frame(ax, INK, 2)
    ax.set_title(name, fontsize=9)
    if name == "original":
        ax.set_xlabel("distance to the\noriginal, in bits:", fontsize=8)
        continue
    d_plain = phash(im) - _h0
    d_variant = min(phash(v) - _h0 for v in variants(im).values())
    for k, (label, d) in enumerate((("plain", d_plain), ("variants", d_variant))):
        verdict = "found" if d <= HASH_THRESHOLD else "missed"
        ax.text(0.5, -0.07 - 0.12 * k, f"{label}: {d} → {verdict}", transform=ax.transAxes, ha="center",
                va="top", fontsize=8, color=FOUND_COLOR if d <= HASH_THRESHOLD else MISSED_COLOR)
fig.suptitle(f"Does the copy land within HASH_THRESHOLD = {HASH_THRESHOLD} bits of its original?", color=INK)
fig.subplots_adjust(bottom=0.25, top=0.82, wspace=0.08)
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q10)</summary>
<p style="margin:6px 0 0 0;">Plain hashing catches the byte-identical copies (MD5) and the smaller,
re-compressed one (pHash survives both), but files the mirrored and rotated copies as new photos: a flip
moves the hash by dozens of bits. Variant hashing catches them too, because one of the six variants of the
source hashes like the copy. The price: six hashes per photo instead of one, still seconds for thousands.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title The memory card's answer key: which file is a copy of which { display-mode: "form" }
hunt5_found = score_hunt(variant_found, HUNT5_KEY)
RESULTS["hunt5"] = {"planted": len(HUNT5_KEY), "found": int(hunt5_found),
                    "kinds": dict(Counter(k["kind"] for k in HUNT5_KEY))}
answer_box("The memory card's answer key", ["answer key:"]
           + [f"  {k['copy']}: {k['kind']} copy of {k['source']}" for k in HUNT5_KEY]
           + [f"Variant hashing found {hunt5_found} of the {len(HUNT5_KEY)} planted copies."])

### 🔴 Optional · Suspicious labels: each class's least typical photos, then embeddings

A wrong label cannot be found by hashing. Without a model, sort each class by how *unlike the rest of its
class* a photo is, and hand the top of each list to a human. The score (atypicality) is the sum of
absolute differences between the photo's colour histogram (section 3) and its class's average histogram.
Most flagged photos will be fine (unusual but correct); the wrong ones are far likelier to be here than in
a random sample.

In [ ]:
# @title Distance of each photo to its class's mean histogram; list the farthest { display-mode: "form" }
N_ATYPICAL = 1 if SMOKE else 2  # photos shown per class (N-014); 6 classes × 2 = 12 photos
_features = CHANNEL_HISTS.reshape(len(image_stats), -1)
_class_mean = {c: _features[(image_stats["class"] == c).to_numpy()].mean(axis=0) for c in CLASSES}
image_stats["atypicality"] = [float(np.abs(f - _class_mean[c]).sum())
                              for f, c in zip(_features, image_stats["class"], strict=True)]

fig, ax = plt.subplots(figsize=(9, 3.4))
for k, c in enumerate(CLASSES):
    _vals = image_stats.loc[image_stats["class"] == c, "atypicality"].to_numpy()
    _jitter = np.random.RandomState(SEED).uniform(-0.25, 0.25, len(_vals))
    ax.scatter(k + _jitter, _vals, s=14, color=CLASS_COLORS[c], alpha=0.8)
ax.set_xticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES)
ax.set_ylabel("distance to the class's mean histogram")
ax.set_title("How typical is each photo of its class? (high = look at me)")
fig.tight_layout()
display(fig)
plt.close(fig)

least_typical = (image_stats.sort_values("atypicality", ascending=False).groupby("class", sort=False)
                 .head(N_ATYPICAL).sort_values(["class", "atypicality"], ascending=[True, False]))
show_rows(least_typical, "least typical photos per class: does the label fit?",
          lambda r: f"{r['class']} · {r['atypicality']:.1f}", cols=6)

**Q11.** 🔴 Look at the gallery. For each photo, is the label right? Is the animal even visible? Note the
ones you would send back to the annotator.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q11)</summary>
<p style="margin:6px 0 0 0;">Colour cannot tell a cat from a dog, so this list finds photos that are
<em>unusual</em>, not <em>wrong</em>: close-ups, night shots, an animal filling a tenth of the frame. Real label errors in
COCO val are rare at image level (more common in the masks, section 8), so an honest answer is "nothing to
send back, two to keep an eye on". On a crowd-sourced dataset, this list is where the wrong labels live.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title Embedding helpers (precomputed vectors, or colour histograms as a stand-in) { display-mode: "form" }
EMBEDDINGS_NAME = "coco_animals_embeddings.npz"  # npz keys: ids (n,), embeddings (n, d); matched by image id


def load_embeddings() -> tuple[np.ndarray, str]:
    """(vectors aligned with `images`, source). Source is "precomputed" or "colour histograms (stand-in)".

    Reads data/coco_animals_embeddings.npz, fetching it from DATA_BASE_URL outside smoke mode. Never runs a
    model. Falls back to the colour histograms when the file is missing or does not cover every image.
    """
    path = DATA_DIR / EMBEDDINGS_NAME
    if not path.exists() and not SMOKE:
        download_file(DATA_BASE_URL + EMBEDDINGS_NAME, path, headers=_github_headers())
    if path.exists():
        try:
            z = np.load(path)
            by_id = dict(zip(z["ids"].tolist(), z["embeddings"], strict=True))
            if all(i in by_id for i in images["id"]):
                return np.stack([by_id[i] for i in images["id"]]).astype(np.float32), "precomputed"
        except Exception:
            pass
    return CHANNEL_HISTS.reshape(len(images), -1).astype(np.float32), "colour histograms (stand-in)"


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    """Cosine similarity between every pair of rows (after centring the columns)."""
    centred = vectors - vectors.mean(axis=0)
    norm = np.linalg.norm(centred, axis=1, keepdims=True) + 1e-9
    unit = centred / norm
    return unit @ unit.T

#### Nearest pairs and outliers in embedding space

An **embedding** is a list of numbers that stands for a photo, chosen so that photos with a similar
*content* get similar numbers (here precomputed; if the file is missing, colour histograms stand in, and
the plot titles say which). **Similarity** runs from 1 (same direction) down to 0 and below. Two cases:

- **Nearest pair**: two photos with similarity near 1, e.g. two frames of the same sheep seconds apart.
  Keep one, or keep both on the same side of the split.
- **Outlier**: a photo with low similarity to the rest of its own class, e.g. a "dog" photo that looks like
  no other dog. Look at it, then relabel or drop it.

Real embeddings would find photos of the same *thing* (the same dog from another angle) and photos whose
content does not match their label. Teams use them this way at scale: nobody can check a million photos by
eye, so the computer lists the closest pairs and the oddest photos and a person looks at those few.

**Q12.** 🔴 Will the closest pairs be the same ones the hash found, and will the outliers overlap with the
colour-based list from Q11? Guess, then run.

In [ ]:
# @title Q12 Nearest pairs and per-class outliers in embedding space { display-mode: "form" }
embeddings, embedding_source = load_embeddings()
print(f"vectors: {embeddings.shape[0]} × {embeddings.shape[1]} from {embedding_source}")
RESULTS["embedding_source"] = embedding_source

_sim = cosine_similarity_matrix(embeddings)
np.fill_diagonal(_sim, -1)
_i, _j = np.unravel_index(np.argsort(-np.triu(_sim, k=1), axis=None)[:N_PAIRS_TO_SHOW], _sim.shape)
_rows = pd.concat([image_stats.iloc[[a, b]] for a, b in zip(_i, _j, strict=True)])
_sim_of = {row_id: _sim[a, b] for a, b in zip(_i, _j, strict=True) for row_id in images["id"].iloc[[a, b]]}
show_rows(_rows, "Nearest pairs (sim 1 = identical): copies or same scene? Keep one",
          lambda r: f"{r['class']} · sim {_sim_of[r['id']]:.2f}", cols=4)

_class_arr = image_stats["class"].to_numpy()
image_stats["class_similarity"] = [float(_sim[k, (_class_arr == c) & (np.arange(len(_sim)) != k)].mean())
                                   for k, c in enumerate(_class_arr)]
embedding_outliers = (image_stats.sort_values("class_similarity").groupby("class", sort=False)
                      .head(N_ATYPICAL).sort_values(["class", "class_similarity"]))
show_rows(embedding_outliers, "Outliers (low sim to their own class): wrong label? Look, relabel or drop",
          lambda r: f"{r['class']} · sim {r['class_similarity']:.2f}", cols=6)
_overlap = len(set(embedding_outliers["id"]) & set(least_typical["id"]))
print(f"{_overlap} of {len(embedding_outliers)} embedding outliers are also in the colour-based list")

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q12)</summary>
<p style="margin:6px 0 0 0;">The hash finds copies of the same <em>picture</em>; the embedding finds photos of the
same <em>thing</em>. The two pair lists overlap on exact and near copies and differ elsewhere, and the two outlier
lists overlap only partly: colour and content are different notions of "typical".</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Embeddings, and how to get real ones</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Why embeddings find what hashes cannot</summary>
<p style="margin:6px 0 0 0;">With real embeddings the closest pairs are <em>semantic</em> near-duplicates (the same dog, another angle), which
a hash cannot find, and the outliers are photos whose content does not match the label. With the stand-in
they are colour matches. To get real embeddings, compute them once on a GPU (any pretrained image model),
save <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">ids</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">embeddings</code> to <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">data/coco_animals_embeddings.npz</code>, and the Q12 cell picks them up.</p>
</details>
</div>""")

<callout: go-deeper>

### Clean the dataset

Time to act. The rule applied below is the one most teams use: keep the first photo of every duplicate
group (exact, near, flipped or rotated), drop files that do not open, and keep everything else. The result
is `clean_images`, the table the rest of the notebook uses.

<details><summary>What happens to grayscale and odd-size photos</summary>

Section 3 left the grayscale and odd-size photos for this section. Photos stored in another mode (L,
CMYK…) or with an extreme size stay in the table, with a note in a `flags` column; every photo is
converted to RGB when it is loaded, so none of them breaks a later section.
</details>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-pitfall" style="background:#FDECEA;color:#8B1A10;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">⚠️ Pitfall · Split after cleaning, not before</div>
<p style="margin:6px 0 0 0;">If you split first and deduplicate each half on its own, a pair with one photo in train and one in test
survives, and the test score reports how well the model <em>memorised</em> that photo. Deduplicate the whole
table, then split (section 6).</p>
</div>""")

<callout: pitfall>

In [ ]:
# @title Find the duplicate groups with variant hashing, then build the cleaned table { display-mode: "form" }
_ids = images["id"].to_numpy()
_drop: dict[int, str] = {}
for _group in exact_groups:  # exact: keep the first file of each group
    _group_ids = [int(images.loc[images["path"] == str(p), "id"].iloc[0]) for p in _group]
    for _gid in sorted(_group_ids)[1:]:
        _drop[_gid] = "exact duplicate"
_best = np.full(_dist.shape, 99)
for _v in VARIANT_NAMES:  # variant hashing across the whole Play Dataset: min distance over the six variants
    _best = np.minimum(_best, hash_distances(VARIANT_HASHES[_v], HASHES))
np.fill_diagonal(_best, 99)
_pi, _pj = np.where(np.triu(np.minimum(_best, _best.T) <= HASH_THRESHOLD, k=1))
for _a, _b in zip(_pi, _pj, strict=True):  # near/flipped/rotated: keep the lower id
    _drop.setdefault(int(max(_ids[_a], _ids[_b])), "near-duplicate (incl. flipped/rotated)")
_odd = flag_odd_files(image_stats)  # files that do not open are dropped silently
_unreadable = _odd.loc[_odd["flags"].str.contains("unreadable"), "id"]
_drop.update({int(i): "unreadable" for i in _unreadable})

clean_images = images[~images["id"].isin(_drop)].copy()
# the non-fatal flags (grayscale, extreme size) stay in a column that sections 9 and 10 read
clean_images["flags"] = clean_images["id"].map(_odd.set_index("id")["flags"]).fillna("")
removed = pd.DataFrame({"id": list(_drop), "reason": list(_drop.values())})
RESULTS["n_clean"] = int(len(clean_images))
RESULTS["n_removed"] = int(len(removed))
print(f"Play Dataset: removed {len(removed)}, kept {len(clean_images)} "
      f"(reasons: {dict(Counter(removed['reason']))})")

# The same rule on the memory card: one file per duplicate group.
_card_drop = set()
for _, _r in variant_found.sort_values(["a", "b"]).iterrows():
    if _r["a"] not in _card_drop:
        _card_drop.add(_r["b"])
print(f"memory card : removed {len(_card_drop)}, kept {len(hunt_files) - len(_card_drop)}")

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · The same errors, three shapes</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Classification:</strong> the same dog photo is stored twice, once labelled "dog", once "sheep". Whichever
  copy lands in the test set, one of them is wrong.</li>
<li style="margin:2px 0;"><strong>Detection:</strong> a mirrored copy of a photo of two sheep whose boxes were not mirrored (<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">x' = W − x − w</code>):
  both boxes now point at grass (section 7).</li>
<li style="margin:2px 0;"><strong>Segmentation:</strong> a copied photo whose mask outlines the other animal, or a mask that is swollen or
  shifted while the label is right (section 8).</li>
</ul>
<p style="margin:6px 0 0 0;">Deduplicate once for all three; the label checks differ per task.</p>
</div>""")

<callout: across-tasks>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a new folder of photos → <strong>do</strong> MD5 for exact copies, then pHash with a threshold you
  have looked at for near copies.</li>
<li style="margin:2px 0;"><strong>When you see</strong> that mirrored or rotated copies could exist → <strong>do</strong> hash the six variants of each photo.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a photo far from the rest of its class → <strong>do</strong> look at it, then relabel or drop it.</li>
<li style="margin:2px 0;"><strong>When you see</strong> the checks are done → <strong>do</strong> report "removed N, kept M", and split after cleaning.</li>
</ul>
</div>""")

<callout: takeaway>

## 6. Splits & leakage

Which photos does the model never see, and does the score on them mean anything?

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Choose a split that does not leak, and know what k-fold, stratified, leave-one-out and leave-one-group-out
(LOGO; here a group = a camera) cross-validation each buy you.</p>
</div>""")

<callout: objective>

The next decision decides whether next week's test score means anything: **which photos does the model
never see?**

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · What the held-out part should look like</div>
<p style="margin:6px 0 0 0;">The held-out part has to look like the photos the park's cameras take <em>after</em> training. If a test photo
has a near-twin in training, the score reports memory, not recognition.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · Data leakage</div>
<p style="margin:6px 0 0 0;"><strong>Leakage</strong> is any way information about the test photos reaches the model during training: a duplicate in
both train and test (section 5), or a near-twin from the same moment (a burst, the same tree). The score
then measures memory.</p>
<p style="margin:6px 0 0 0;"><strong>Rule:</strong> split by whatever will be <em>new</em> at deployment (a camera, a day, a farm), not by photo.</p>
</div>""")

<callout: theory>

### Random vs stratified split on the Play Dataset

A random split shuffles the photos and cuts them 80 / 20. A **stratified** split cuts *each class* 80 / 20,
so train and test keep the class shares of the whole table. A **class share** = a class's photos / all
photos in that set (30 cats among 100 = 30 %).

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · One bad shuffle, in numbers</div>
<p style="margin:6px 0 0 0;">Picture 100 cats and 20 sheep, and a random test set of 24 photos (20 %). On average it holds 20 cats and
4 sheep (shares 83 % and 17 %). A bad shuffle gives 22 cats and 2 sheep (92 % and 8 %): the cat share moved
by 9 points, the sheep share was cut in half.</p>
</div>""")

<callout: go-deeper>

**Q13.** 🟡 On the Play Dataset's random 80 / 20 split, for each class compare its share in the test set
with its share overall. Which class is off by the biggest fraction of itself: the biggest or the smallest?
Does the stratified split fix that completely, or only mostly? Decide, then run.

In [ ]:
# @title Q13 Random vs stratified 80 / 20 split and the class share of each test set { display-mode: "form" }
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    LeaveOneGroupOut,
    LeaveOneOut,
    StratifiedKFold,
    train_test_split,
)

TEST_SIZE = 0.2
_ids = clean_images["id"].to_numpy()
_y = clean_images["class"].to_numpy()
_tr_rand, _te_rand = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED)
_tr_strat, _te_strat = train_test_split(_ids, test_size=TEST_SIZE, random_state=SEED, stratify=_y)


def class_shares(ids) -> pd.Series:
    """Share of each class among the given image ids of clean_images (in CLASSES order, sums to 1)."""
    counts = clean_images.set_index("id").loc[list(ids), "class"].value_counts()
    return counts.reindex(CLASSES).fillna(0) / max(len(ids), 1)


shares = pd.DataFrame(
    {
        "whole table": class_shares(_ids),
        "random · test": class_shares(_te_rand),
        "stratified · test": class_shares(_te_strat),
    }
)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, col in zip(axes, ["random · test", "stratified · test"], strict=True):
    x = np.arange(len(CLASSES))
    ax.bar(x - 0.2, shares["whole table"], width=0.4, color="#DDE5E7", label="whole table")
    ax.bar(x + 0.2, shares[col], width=0.4, color=[CLASS_COLORS[c] for c in CLASSES], label=col)
    ax.set_xticks(x, CLASSES)
    ax.set_title(f"{col} ({len(_te_rand)} photos)")
    ax.set_ylabel("share of photos")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()
_drift = (shares.sub(shares["whole table"], axis=0).abs() * 100).round(1)
print("largest class-share drift from the whole table, in percentage points:")
print(f"  random split: {_drift['random · test'].max():.1f} ({_drift['random · test'].idxmax()})   "
      f"stratified split: {_drift['stratified · test'].max():.1f} ({_drift['stratified · test'].idxmax()})")
_whole = shares["whole table"].replace(0, np.nan)
_rel = shares.sub(shares["whole table"], axis=0).abs().div(_whole, axis=0) * 100
print("largest drift relative to the class's own share, in %:")
print(f"  random split: {_rel['random · test'].max():.0f} ({_rel['random · test'].idxmax()})   "
      f"stratified split: {_rel['stratified · test'].max():.0f} ({_rel['stratified · test'].idxmax()})")
RESULTS["split_drift_random_pp"] = float(_drift["random · test"].max())
RESULTS["split_drift_stratified_pp"] = float(_drift["stratified · test"].max())

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q13)</summary>
<p style="margin:6px 0 0 0;">The smallest class moves the most relative to its own size: one photo can halve
a small share (grey bars = whole table, coloured = test set), and if that class is the dog, the alert is
scored on a handful of photos. Stratifying fixes it up to rounding (you cannot put 10.6 sheep in test), but
says nothing about <em>what is in</em> the photos: that is the rest of this section.</p>
</details>
</div>""")

<callout: answer>

### Cross-validation: four ways to reuse one dataset

One split gives one score that moves by several points with a few hundred photos. **Cross-validation**
rotates the test part around the table so every photo is tested once, and reports the mean (and spread)
over the folds.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · k-fold, stratified k-fold, leave-one-out, leave-one-group-out</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>k-fold:</strong> k pieces (<em>folds</em>), each tested once, trained on the other k−1. <em>Park:</em> 5 folds of 100 photos.</li>
<li style="margin:2px 0;"><strong>stratified k-fold:</strong> every fold keeps the class shares; use it when classes are imbalanced.</li>
<li style="margin:2px 0;"><strong>leave-one-out (LOO):</strong> k = the number of photos; each photo is tested alone. <em>Park:</em> 100 trainings.</li>
<li style="margin:2px 0;"><strong>leave-one-group-out (LOGO; here a group = a camera):</strong> each fold holds out one whole camera. The only
  scheme that tests a <em>new</em> camera.</li>
</ul>
</div>""")

<callout: theory>

The next cell draws the four schemes on 12 toy photos from 3 cameras (`D` = a dog, the rare class), three
folds each: magenta photos are tested, grey-blue ones are trained on.

In [ ]:
# @title Diagram: which photos are tested in k-fold, stratified k-fold, LOO and LOGO { display-mode: "form" }
SPLIT_COLORS = {"train": "#8FA8AD", "test": "#B5446E"}  # train: neutral grey-blue · test: the held-out part
_dogs = {2, 3, 6}  # toy photos 0-11, four per camera (A: 0-3, B: 4-7, C: 8-11); the dog is the rare class
_cv_folds = {  # the test photos of three folds per scheme
    "k-fold (a fold may miss the dog)": [{2, 3, 7, 10}, {0, 5, 8, 11}, {1, 4, 6, 9}],
    "stratified k-fold (each fold has one dog)": [{0, 2, 4, 9}, {1, 3, 7, 10}, {5, 6, 8, 11}],
    "LOO (12 folds, the first three)": [{0}, {1}, {2}],
    "LOGO (one camera per fold)": [{0, 1, 2, 3}, {4, 5, 6, 7}, {8, 9, 10, 11}],
}
fig, axes = plt.subplots(len(_cv_folds), 1, figsize=(8.5, 7.2))
for ax, (name, folds) in zip(axes, _cv_folds.items(), strict=True):
    for i, test in enumerate(folds):
        for p in range(12):
            x = p + 0.35 * (p // 4)  # a small gap between cameras
            ax.add_patch(mpl.patches.FancyBboxPatch(
                (x + 0.06, i + 0.08), 0.88, 0.84, boxstyle="round,pad=0.02",
                facecolor=SPLIT_COLORS["test" if p in test else "train"], edgecolor="none"))
            if p in _dogs:
                ax.text(x + 0.5, i + 0.5, "D", ha="center", va="center", fontsize=8, color="white",
                        fontweight="bold")
    ticks, labels = [0.5, 1.5, 2.5], ["fold 1", "fold 2", "fold 3"]
    if name.startswith("LOO"):  # N-051: say in the picture that folds 4-12 exist but are not drawn
        for p in range(12):
            ax.text(p + 0.35 * (p // 4) + 0.5, 3.3, "⋮", ha="center", va="center", fontsize=11, color=INK)
        ticks, labels = ticks + [3.3], labels + ["folds 4–12\nnot shown"]
    ax.set_xlim(0, 12.7)
    ax.set_ylim(3.6, 0)
    ax.set_xticks([])
    ax.set_yticks(ticks, labels, fontsize=8)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(name, loc="left", fontsize=9, color=INK)
axes[-1].set_xticks([2, 6.35, 10.7], ["camera A", "camera B", "camera C"], fontsize=8)
axes[-1].tick_params(length=0)
fig.suptitle("Toy table of 12 photos: every row is one fold, test photos in magenta", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Read the picture: plain k-fold can leave a fold without the dog, stratified k-fold gives every fold one,
and a leave-one-out fold holds a single photo, so some class has **0 %** of it. LOGO holds out a whole
camera.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · When leave-one-out is worth it</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">LOO on a very small dataset, and the scikit-learn names</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">LOO suits a <strong>very small</strong> dataset (a few dozen photos) where every training example counts.</li>
<li style="margin:2px 0;">Otherwise it costs n trainings, and each fold scores one photo, 0 or 1, so the mean is noisy and says
  nothing per class. On a slice of 40 photos: 40 folds of 1 test photo, 40 trainings for one score.</li>
<li style="margin:2px 0;">Plain k-fold lets a small class dip in some fold; stratified k-fold holds it steady.</li>
<li style="margin:2px 0;">LOGO needs groups: the Play Dataset has none (COCO photos come from thousands of Flickr accounts), the
  park's cameras do. scikit-learn: <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">LeaveOneGroupOut</code>, or <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">GroupKFold</code> with k = the number of groups.</li>
<li style="margin:2px 0;">The next (collapsed) cell counts the folds of each scheme on the Play Dataset, with LOO on a slice of 40
  photos.</li>
</ul>
</details>
</div>""")

<callout: go-deeper>

In [ ]:
# @title Count the folds of k-fold, stratified k-fold and leave-one-out { display-mode: "form" }
K_FOLDS = 5
LOO_N = 12 if SMOKE else 40  # leave-one-out on a slice: a "very small dataset" (N-014: same code, fewer rows)


def summarise_folds(name: str, splitter, X, y, groups=None) -> dict:
    """One row per CV scheme: folds, test-fold size and the smallest class share across the test folds."""
    sizes, min_share = [], 1.0
    for _, test_idx in splitter.split(X, y, groups):
        sizes.append(len(test_idx))
        counts = pd.Series(y[test_idx]).value_counts(normalize=True).reindex(sorted(set(y))).fillna(0)
        min_share = min(min_share, float(counts.min()))
    return {"scheme": name, "folds": len(sizes), "test fold size": f"{min(sizes)}–{max(sizes)}",
            "lowest class share in a test fold": round(min_share, 3)}


_loo_ids = _ids[:LOO_N]
cv_rows = [
    summarise_folds("kfold", KFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("stratified_kfold", StratifiedKFold(K_FOLDS, shuffle=True, random_state=SEED), _ids, _y),
    summarise_folds("loo", LeaveOneOut(), _loo_ids, _y[:LOO_N]),
]
cv_schemes = [r["scheme"] for r in cv_rows]

### The woodland trail cameras

The park has just hung **trail cameras in its woodland reserve**. A camera fires on motion and takes a
**burst** of three frames a second apart, every night, from the same tree, for years.

- **Stand-in data:** a slice of the Caltech Camera Traps dataset (CCT20): eight cameras, five animals
  (opossum, raccoon, coyote, bobcat, rabbit), whole bursts kept together.
- **First run:** the next cell downloads the photos and shrinks them to 512 px, so **it takes a minute**;
  afterwards it is instant.

In [ ]:
# @title Trail-camera helpers (manifest, download + resize to 512 px, burst gallery) { display-mode: "form" }
import re

CCT_DIR = DATA_DIR / "cache" / "cct20"
CCT_SIDE = 512
CCT_CLASSES = ["opossum", "raccoon", "coyote", "bobcat", "rabbit"]
CCT_COLORS = dict(zip(CCT_CLASSES, [CLASS_COLORS[c] for c in CLASSES], strict=False))
SMOKE_CCT_LOCATIONS = 3
SMOKE_CCT_IMAGES = 30


def select_cct_rows(table: pd.DataFrame) -> pd.DataFrame:
    """The slice to use: everything, or in smoke mode whole bursts from 3 locations up to 30 photos."""
    if not SMOKE:
        return table
    keep = []
    locations = sorted(table["location"].unique())[:SMOKE_CCT_LOCATIONS]
    per_location = SMOKE_CCT_IMAGES // SMOKE_CCT_LOCATIONS
    for loc in locations:
        n = 0
        at_loc = table[table["location"] == loc].sort_values(["seq_id", "frame_num"])
        for _, burst in at_loc.groupby("seq_id", sort=True):
            if n + len(burst) > per_location:
                continue
            keep.append(burst)
            n += len(burst)
    return pd.concat(keep).reset_index(drop=True)


def cct_original_size(path: Path) -> tuple[int, int] | None:
    """The camera's (width, height), stored in the 512 px copy's JPEG comment; None for an older copy."""
    with Image.open(path) as im:
        found = re.match(rb"original (\d+)x(\d+)", im.info.get("comment", b""))
    return (int(found[1]), int(found[2])) if found else None


def fetch_cct_image(row: dict) -> tuple[str, Path | None]:
    """Download one trail-camera photo and store a 512 px copy; returns (file_name, path or None)."""
    dest = CCT_DIR / row["file_name"]
    if dest.exists() and dest.stat().st_size > 0 and cct_original_size(dest):
        return row["file_name"], dest
    raw = download_file(row["url"], CCT_DIR / "raw" / row["file_name"])
    if raw is None:
        return row["file_name"], None
    with Image.open(raw) as im:
        width, height = im.size  # the boxes in the manifest are in these pixels
        im.draft("RGB", (CCT_SIDE, CCT_SIDE))
        small = im.convert("RGB")
        small.thumbnail((CCT_SIDE, CCT_SIDE))
        small.save(dest, "JPEG", quality=90, comment=f"original {width}x{height}".encode())
    raw.unlink(missing_ok=True)
    return row["file_name"], dest


def download_cct(table: pd.DataFrame, workers: int = 16) -> pd.DataFrame:
    """Fetch every photo of `table` (parallel, cached), return the rows whose photo is on disk with a path."""
    CCT_DIR.mkdir(parents=True, exist_ok=True)
    paths, done = {}, 0
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for name, path in pool.map(fetch_cct_image, table.to_dict("records")):
            done += 1
            if path is not None:
                paths[name] = str(path)
            print(f"\r  trail-camera photos ready: {done}/{len(table)}", end="", flush=True)
    print()
    out = table[table["file_name"].isin(paths)].copy()
    out["path"] = out["file_name"].map(paths)
    return out.sort_values(["location", "seq_id", "frame_num"]).reset_index(drop=True)


def draw_cct_boxes(ax, row, thumb) -> None:
    """Draw the animal's box(es) from the manifest on a thumbnail, in the class colour (none when empty)."""
    original = cct_original_size(Path(row["path"]))
    if not isinstance(row["bbox"], str) or original is None:
        return
    scale = thumb.width / original[0]
    for x, y, w, h in json.loads(row["bbox"]):
        ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False, lw=2,
                                           edgecolor=CCT_COLORS[row["label"]]))


def show_bursts(table: pd.DataFrame, n_bursts: int, size: int = 256) -> None:
    """One row per burst: its frames side by side, framed in the class colour, titled with the location.

    Picks bursts of different animals at different cameras, with a box on every frame, where the table
    allows it.
    """
    candidates = [b for _, b in table.groupby("seq_id", sort=True) if len(b) > 1]
    candidates.sort(key=lambda b: not b["bbox"].notna().all())  # fully boxed bursts first (stable sort)
    bursts, seen_labels, seen_locations = [], set(), set()
    for b in candidates:
        label, location = b["label"].iloc[0], b["location"].iloc[0]
        if label not in seen_labels and location not in seen_locations:
            bursts.append(b)
            seen_labels.add(label)
            seen_locations.add(location)
    bursts += [b for b in candidates if not any(b is k for k in bursts)][: n_bursts - len(bursts)]
    bursts = bursts[:n_bursts]
    cols = max(len(b) for b in bursts)
    fig, axes = plt.subplots(len(bursts), cols, figsize=(2.6 * cols, 2.1 * len(bursts)), squeeze=False)
    for r, burst in enumerate(bursts):
        for c in range(cols):
            ax = axes[r, c]
            if c < len(burst):
                row = burst.iloc[c]
                thumb = load_thumb(row["path"], size)
                ax.imshow(thumb)
                draw_cct_boxes(ax, row, thumb)
                frame(ax, CCT_COLORS[row["label"]])
                ax.set_title(f"camera {row['location']} · {row['label']} · frame {row['frame_num']}",
                             fontsize=8)
            else:
                ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title Load the trail-camera slice, download and shrink the photos, show bursts { display-mode: "form" }
cct_all = pd.read_csv(fetch_manifest("cct20_slice.csv"))
cct = download_cct(select_cct_rows(cct_all))
if len(cct) < len(select_cct_rows(cct_all)):
    print(f"  ⚠️ {len(select_cct_rows(cct_all)) - len(cct)} photos could not be downloaded; run again.")
CCT_LOCATIONS = sorted(cct["location"].unique().tolist())
print(f"{len(cct)} photos · {cct['seq_id'].nunique()} bursts · cameras {CCT_LOCATIONS}")
print("photos per class: " + " · ".join(f"{c} {int((cct['label'] == c).sum())}" for c in CCT_CLASSES))
show_bursts(cct, n_bursts=3)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Why a burst is a leak</div>
<p style="margin:6px 0 0 0;">The frames of a burst differ by a second and a few steps; the tree, the ground and the exposure are
identical. A test frame whose siblings are in train is a photo the model has almost seen.</p>
</div>""")

<callout: go-deeper>

### Leakage, measured without a model

The **sibling rate** is the share of test photos that have a burst sibling (another frame with the same
`seq_id`) in training. 0 % means no leak; no training is needed to see it.

1. A burst of three frames from camera A meets a random split: frames 1 and 3 go to train, frame 2 to test.
2. Frame 2 has two siblings in training, so it counts toward the sibling rate.
3. The model has seen this tree and this raccoon: frame 2 is easy, the score too good.
4. A split **by camera** keeps all three frames on one side: nothing leaks.

In [ ]:
# @title Diagram: three bursts under a random split and a split by camera { display-mode: "form" }
_demo = {  # three cameras, one burst of three frames each; where does every frame land?
    "random split": [["train", "test", "train"], ["train", "train", "test"], ["test", "train", "train"]],
    "split by camera": [["train"] * 3, ["train"] * 3, ["test"] * 3],
}
fig, axes = plt.subplots(1, 2, figsize=(9, 2.9))
for ax, (name, grid) in zip(axes, _demo.items(), strict=True):
    n_test = sum(f == "test" for burst in grid for f in burst)
    n_leaky = sum(f == "test" for burst in grid if "train" in burst for f in burst)
    for i, burst in enumerate(grid):
        for j, where in enumerate(burst):
            ax.add_patch(mpl.patches.FancyBboxPatch((j + 0.08, i + 0.08), 0.84, 0.84,
                                                    boxstyle="round,pad=0.02",
                                                    facecolor=SPLIT_COLORS[where], edgecolor="none"))
            ax.text(j + 0.5, i + 0.5, f"frame {j + 1}\n{where}", ha="center", va="center", fontsize=8,
                    color="white")
    ax.set_xlim(0, 3)
    ax.set_ylim(3, 0)
    ax.set_xticks([])
    ax.set_yticks([0.5, 1.5, 2.5], ["camera A", "camera B", "camera C"])
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(f"{name}: sibling rate {n_leaky / n_test:.0%}", fontsize=10, color=INK)
fig.suptitle("One burst per camera: every frame is coloured by the set it lands in", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

**Q14.** 🟡 Take 100 trail-camera photos in bursts of three and a random 80 / 20 split (20 photos in test).
For one test photo, each of its two siblings lands in training with probability 0.8. Predict the sibling
rate of the random split: about 0 %, about 20 %, or most of the test photos? And of a split by camera?

In [ ]:
# @title Q14 Test photos with a burst sibling in training: random vs by camera { display-mode: "form" }
def sibling_rate(train: pd.DataFrame, test: pd.DataFrame) -> float:
    """Share of test photos whose seq_id (burst) also appears in the training table (0 = no leak)."""
    if len(test) == 0:
        return 0.0
    return float(test["seq_id"].isin(set(train["seq_id"])).mean())


_tr, _te = train_test_split(cct.index.to_numpy(), test_size=TEST_SIZE, random_state=SEED)
sibling_random = sibling_rate(cct.loc[_tr], cct.loc[_te])

_gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
_tr_g, _te_g = next(_gss.split(cct, groups=cct["location"]))
sibling_camera = sibling_rate(cct.iloc[_tr_g], cct.iloc[_te_g])
_held_out = sorted(cct.iloc[_te_g]["location"].unique().tolist())

RESULTS["sibling_rate_random"] = sibling_random
RESULTS["sibling_rate_by_camera"] = sibling_camera
print(f"random split    : {sibling_random:5.1%} of the {len(_te)} test photos have a burst sibling in train")
print(f"split by camera : {sibling_camera:5.1%} of the {len(_te_g)} test photos (held out: {_held_out})")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["random split", "split by camera"], [sibling_random, sibling_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.set_ylabel("test photos with a burst sibling in train")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

The same random split on real photos: a test frame (magenta) next to its siblings in training (grey-blue).

In [ ]:
# @title Q14 on real photos: a test frame next to its siblings in training { display-mode: "form" }
def show_leaks(table: pd.DataFrame, train_idx, test_idx, n_examples: int = 2, size: int = 256) -> None:
    """For a few test photos that have siblings in training, show the whole burst: test vs train frames."""
    in_test = set(test_idx)
    train_bursts = set(table.loc[train_idx, "seq_id"])
    leaked = table.loc[test_idx]
    leaked = leaked[leaked["seq_id"].isin(train_bursts)].drop_duplicates("seq_id").head(n_examples)
    if leaked.empty:
        print("no test photo has a sibling in training in this split.")
        return
    bursts = [table[table["seq_id"] == s] for s in leaked["seq_id"]]
    cols = max(len(b) for b in bursts)
    fig, axes = plt.subplots(len(bursts), cols, figsize=(2.6 * cols, 2.3 * len(bursts)), squeeze=False)
    for r, burst in enumerate(bursts):
        for c in range(cols):
            ax = axes[r, c]
            if c >= len(burst):
                ax.axis("off")
                continue
            row = burst.iloc[c]
            part = "test" if burst.index[c] in in_test else "train"
            thumb = load_thumb(row["path"], size)
            ax.imshow(thumb)
            draw_cct_boxes(ax, row, thumb)
            frame(ax, SPLIT_COLORS[part])
            ax.set_title(f"camera {row['location']} · frame {row['frame_num']} · {part}", fontsize=8)
    fig.suptitle("Random split: the test frame and its siblings in training", color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


show_leaks(cct, _tr, _te)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q14)</summary>
<p style="margin:6px 0 0 0;">Random split: <strong>most test photos</strong>, about 96 of 100. Split by camera: <strong>0 %</strong>,
because a burst never spans two cameras.</p>
<p style="margin:6px 0 0 0;">A test frame is safe only if <em>both</em> siblings are in test too, probability about 0.2 × 0.2 = 4 %; the rate
is a bit lower on the real slice, where some bursts are shorter than three frames.</p>
<p style="margin:6px 0 0 0;"><strong>In the field:</strong> Beery et al. (2018) report an error of <strong>20.8 % at cameras seen in training and 41.1 % at
cameras never seen</strong>: it doubles at the first new tree. A random split would have reported the 20.8 % and
the park would have found the 41.1 % in the field.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Where this data comes from</div>
<p style="margin:6px 0 0 0;">The trail-camera photos are a slice of the Caltech Camera Traps data, introduced in Beery, Van Horn and
Perona (2018), <em>Recognition in Terra Incognita</em>, the paper behind the two error rates above. It tests the
same trained model at cameras it has seen and at cameras it has never seen. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">data/cct20_slice.csv</code> lists
the photos. The five woodland classes borrow the park's colours, so each class keeps one colour.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-pitfall" style="background:#FDECEA;color:#8B1A10;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">⚠️ Pitfall · LOO leaks like a random split</div>
<p style="margin:6px 0 0 0;">LOO sounds strict (train on everything but one photo), but on burst data it is the <em>worst</em> case:</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">Hold out frame 2 of a burst.</li>
<li style="margin:2px 0;">Frames 1 and 3, same tree and animal, are in training: the sibling rate is as high as it can be.</li>
</ol>
<p style="margin:6px 0 0 0;"><strong>Fix:</strong> LOGO with the camera as the group: every fold holds out one whole camera, so the sibling rate is
zero. The unit you leave out must be the unit that will be new.</p>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">More on this pitfall</summary>
<p style="margin:6px 0 0 0;">"As high as it can be" is the share of photos that belong to a burst of more than one frame: nearly all of
them. With LOGO the score is the one a new camera will see.</p>
</details>
</div>""")

<callout: pitfall>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · Splitting for boxes and masks</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Classification:</strong> stratify by the image label; group by camera, day or source.</li>
<li style="margin:2px 0;"><strong>Detection:</strong> the unit is still the <em>image</em>, never the box. A paddock photo with 3 sheep and 1 horse
  cannot have its sheep in train and its horse in test. One photo carries several labels, so plain
  stratification does not apply: <em>iterative stratification</em> balances the box counts per class across folds
  (section 7).</li>
<li style="margin:2px 0;"><strong>Segmentation:</strong> the same, with pixel shares instead of box counts. A rare class with a few large masks,
  the aviary's birds, can end up entirely in one fold if you do not check (section 8).</li>
</ul>
<p style="margin:6px 0 0 0;">In all three: split once, then never look at the test set again.</p>
</div>""")

<callout: across-tasks>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> classes of very different sizes → <strong>do</strong> stratify the split, so every class is in the
  test set and scored.</li>
<li style="margin:2px 0;"><strong>When you see</strong> photos from the same moment, camera or day (a burst, one tree) → <strong>do</strong> split by that
  group and measure the sibling rate (it should be 0 %) before any model.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a tiny dataset → <strong>do</strong> use LOO only if every photo counts, and LOGO if the photos come
  in groups.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a test score that looks too good → <strong>do</strong> ask what train and test share.</li>
</ul>
</div>""")

<callout: takeaway>

### 🔴 Optional · Split the trail cameras yourself

Three steps for after class or a fast group: check the LOO pitfall fold by fold, write a split by camera,
and see what the leak does to a score.

#### LOO vs LOGO, fold by fold

The next cell checks that claim fold by fold: the mean sibling rate of LOO and LOGO on the trail-camera
photos.

In [ ]:
# @title LOO vs LOGO (camera): sibling rate per fold { display-mode: "form" }
def mean_sibling_rate(splitter, table: pd.DataFrame, groups=None) -> float:
    """Mean over the folds of the share of test photos with a burst sibling in the fold's training part."""
    rates = [sibling_rate(table.iloc[tr], table.iloc[te]) for tr, te in splitter.split(table, groups=groups)]
    return float(np.mean(rates))


loo_sibling = mean_sibling_rate(LeaveOneOut(), cct)
logo_sibling = mean_sibling_rate(LeaveOneGroupOut(), cct, groups=cct["location"])
cv_schemes += ["logo"]
RESULTS["cv_schemes"] = cv_schemes
RESULTS["sibling_rate_loo"] = loo_sibling
RESULTS["sibling_rate_logo"] = logo_sibling
print(f"LOO  : {len(cct)} folds, sibling rate {loo_sibling:.1%}")
print(f"LOGO : {cct['location'].nunique()} folds (one per camera), sibling rate {logo_sibling:.1%}")
print(f"cross-validation schemes run in this notebook: {cv_schemes}")

#### ✍️ Choose and run a split

Before you choose the held-out cameras, the next cell counts each class's photos per camera: a class with
no photos at your test cameras gets no score.

In [ ]:
# @title Photos per class per camera (to keep every class in test) { display-mode: "form" }
display(pd.crosstab(cct["location"], cct["label"]).reindex(columns=CCT_CLASSES, fill_value=0))

In [ ]:
# @title Reference solution — peek only if you are stuck { display-mode: "form" }
def _ref_split_by_camera(table, test_locations):
    in_test = table["location"].isin(test_locations)
    return {"train": sorted(table.loc[~in_test, "file_name"]),
            "test": sorted(table.loc[in_test, "file_name"])}

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">split_by_camera</code></div>
<p style="margin:6px 0 0 0;">Fill in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">split_by_camera(table, test_locations)</code>:</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Input:</strong> the table <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">cct</code> (columns <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">file_name</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">location</code>) and a list of cameras to hold out.</li>
<li style="margin:2px 0;"><strong>Output:</strong> <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">{"train": [...], "test": [...]}</code> of sorted <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">file_name</code> values: held-out cameras in test,
  all others in train.</li>
<li style="margin:2px 0;"><strong>Then</strong> choose <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">TEST_LOCATIONS</code> (cameras that keep every class in test, see the table above). <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">use()</code>
  checks your function and continues with the reference either way.</li>
</ul>
</div>""")

<callout: your-turn>

In [ ]:
# ✍️ YOUR CODE
def split_by_camera(table, test_locations):
    """Split the trail-camera table by camera.

    table: a DataFrame with columns file_name and location (cct). test_locations: list of location ids.
    Returns {"train": sorted file_names of every other camera, "test": sorted file_names of test_locations}.

    e.g. {"train": ["59a30dbc-….jpg", "5a2c84a7-….jpg", …], "test": ["5965aa3e-….jpg", …]}
    """
    ...


TEST_LOCATIONS = [CCT_LOCATIONS[0], CCT_LOCATIONS[-1]]  # choose: any cameras from CCT_LOCATIONS
_bad = [loc for loc in TEST_LOCATIONS if loc not in CCT_LOCATIONS]
if _bad or not TEST_LOCATIONS or len(set(TEST_LOCATIONS)) >= len(CCT_LOCATIONS):
    raise SystemExit(f"TEST_LOCATIONS must be a non-empty subset of {CCT_LOCATIONS} that leaves cameras for "
                     f"training; unknown: {_bad}")

my_split = use(split_by_camera, cct, TEST_LOCATIONS, reference=_ref_split_by_camera)
_train_t = cct[cct["file_name"].isin(my_split["train"])]
_test_t = cct[cct["file_name"].isin(my_split["test"])]
print(f"train: {len(_train_t)} photos from cameras {sorted(_train_t['location'].unique().tolist())}")
print(f"test : {len(_test_t)} photos from cameras {sorted(_test_t['location'].unique().tolist())}")
print(f"sibling rate of this split: {sibling_rate(_train_t, _test_t):.1%}")
_split_shares = pd.DataFrame({
    "train": _train_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
    "test": _test_t["label"].value_counts(normalize=True).reindex(CCT_CLASSES).fillna(0),
}).round(2)
display(_split_shares.T)
_absent = [c for c in CCT_CLASSES if _split_shares.loc[c, "test"] == 0]
if _absent:
    print(f"⚠️ no {', '.join(_absent)} photos in the test cameras: those classes get no score. "
          "Try other cameras.")

💡 A split by camera has no siblings across the cut, but it can leave a class without test photos. You
trade a clean score for a less balanced one: choose the held-out cameras with both in mind.

<details><summary>An example</summary>

Not every animal walks past every camera. Holding out cameras that never see a coyote leaves the test set
without one (and holding out all the cameras that do leaves training without one).

</details>

#### A ruler to see the gap

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · The nearest-neighbour ruler</div>
<p style="margin:6px 0 0 0;">No network is trained in this notebook. To <em>see</em> the gap the two splits create, we use a <strong>ruler, not
training</strong>: a 1-nearest-neighbour classifier on each photo's colour histogram (the 96 numbers of section 3).
It copies the label of the most similar training photo, has nothing to learn, and runs in seconds on a CPU.
It measures the same thing twice: with k-fold (random, leaks across bursts) and with LOGO (one camera out).</p>
</div>""")

<callout: go-deeper>

**Q15.** 🔴 With 5 animals, guessing scores 20 %. Will the ruler score higher with k-fold or with LOGO, and
roughly by how much? Predict, then run.

In [ ]:
# @title Q15 Colour-histogram ruler: 1-NN accuracy, k-fold vs LOGO (one camera out) { display-mode: "form" }
from sklearn.neighbors import KNeighborsClassifier

RULER_THUMB = 64
_features = np.stack([measure_image(p, RULER_THUMB)[1].ravel() for p in cct["path"]])
_labels = cct["label"].to_numpy()


def ruler_accuracy(splitter, groups=None) -> float:
    """Mean 1-NN accuracy over the folds of `splitter` on the colour-histogram features."""
    scores = []
    for tr, te in splitter.split(_features, _labels, groups):
        knn = KNeighborsClassifier(n_neighbors=1).fit(_features[tr], _labels[tr])
        scores.append(float((knn.predict(_features[te]) == _labels[te]).mean()))
    return float(np.mean(scores))


_n_cameras = cct["location"].nunique()
ruler_kfold = ruler_accuracy(KFold(min(K_FOLDS, _n_cameras), shuffle=True, random_state=SEED))
ruler_by_camera = ruler_accuracy(GroupKFold(_n_cameras), groups=cct["location"])  # k = cameras: LOGO
RESULTS["ruler_kfold"] = ruler_kfold
RESULTS["ruler_by_camera"] = ruler_by_camera
print(f"ruler accuracy, k-fold (random)  : {ruler_kfold:.1%}")
print(f"ruler accuracy, LOGO (by camera) : {ruler_by_camera:.1%}")
print(f"chance level with {len(CCT_CLASSES)} classes: {1 / len(CCT_CLASSES):.0%}")

fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(["k-fold (random)", "LOGO (by camera)"], [ruler_kfold, ruler_by_camera],
       color=[CLASS_COLORS["cat"], CLASS_COLORS["zebra"]])
ax.axhline(1 / len(CCT_CLASSES), color=INK, linestyle="--", linewidth=1, label="chance")
ax.set_ylabel("1-NN ruler accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q15)</summary>
<p style="margin:6px 0 0 0;">K-fold scores higher, by a large gap: on the full slice the ruler roughly
doubles its accuracy when its neighbours may be burst siblings (same animal, same tree), while under LOGO
it must find a similar photo at another camera and falls towards chance. Read the direction and the size of
the gap, not the digits (they depend on the slice and the seed). A real model scores higher on both and the
gap stays: it is the 20.8 % → 41.1 % of the paper.</p>
</details>
</div>""")

<callout: answer>

## 7. Same data, boxes

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">See what imbalance, object size, position and crowding mean once the park wants a <em>headcount</em>, and find
the one box an annotation team forgot.</p>
</div>""")

<callout: objective>

### What a label costs

The park's second job, **count** the sheep, zebras and birds every day, needs a **box per animal**
instead of one label per photo, and the same photos now tell a different story. Boxes (and, in section 8,
masks) are richer labels, and richer labels cost more. The cell below counts them in the crudest unit
there is: clicks a labeller had to make.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · One label, one box, every pixel</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Classification:</strong> one label per image. One click.</li>
<li style="margin:2px 0;"><strong>Detection:</strong> one box per animal. Two clicks per box.</li>
<li style="margin:2px 0;"><strong>Segmentation:</strong> one click per vertex of a polygon, many vertices per animal.</li>
</ul>
<p style="margin:6px 0 0 0;">The table below counts exactly these clicks on the Play Dataset.</p>
</div>""")

<callout: across-tasks>

In [ ]:
# @title Clicks per task: 1 per image label, 2 per box, 1 per mask vertex { display-mode: "form" }
_polygons = annotations["segmentation"].apply(lambda s: s if isinstance(s, list) else [])
_vertices = _polygons.apply(lambda polys: sum(len(p) // 2 for p in polys))
_n_crowd = int((annotations["iscrowd"] == 1).sum())
annotation_cost = pd.DataFrame(
    {
        "unit": ["image labels", "boxes", "mask vertices"],
        "count": [len(images), len(annotations), int(_vertices.sum())],
        "clicks": [len(images), 2 * len(annotations), int(_vertices.sum())],
    }
).set_index("unit")
annotation_cost["clicks per photo"] = (annotation_cost["clicks"] / len(images)).round(1)
_label_clicks = annotation_cost.loc["image labels", "clicks"]
annotation_cost["× classification"] = (annotation_cost["clicks"] / _label_clicks).round(1)
if _n_crowd:
    print(f"({_n_crowd} crowd regions are stored as pixel masks, not polygons, and count 0 vertices here.)")
display(annotation_cost)

💡 Same photos, same animals, and the click count climbs with every task: outlining every animal costs
far more clicks than labelling the photo, so ask for masks only when the park really needs outlines.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · What a richer label changes</div>
<p style="margin:6px 0 0 0;">This section looks at the boxes of the photos in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">clean_images</code>. Classification cannot say "how many" or
"where". Detection needs a decision for every animal (is that blur a bird?), and imbalance is now counted
in <em>boxes</em> (this section). Segmentation gives every pixel a class,
so imbalance is counted in <em>pixels</em>, and birds end up with a few per cent of them (section 8). Each task
costs several times more than the one before, which is why real datasets have many more image labels than
boxes, and many more boxes than masks.</p>
</div>""")

<callout: go-deeper>

### From labels to boxes

`annotations` holds one row per animal, with its box as `[x, y, w, h]` in pixels (COCO format: the
top-left corner, then the width and height).

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · What changes when the unit is a box</div>
<p style="margin:6px 0 0 0;">A classifier sees one example per photo. A detector sees one example per <strong>box</strong>, so a photo with twelve
sheep weighs twelve times a photo with one cat.</p>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">The four things that follow, each with a chart below</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Imbalance</strong> is counted in boxes, not photos (a paddock photo shows a flock).</li>
<li style="margin:2px 0;"><strong>Size</strong>: a bird on the far side of the pond is a 12×8 px speck.</li>
<li style="margin:2px 0;"><strong>Position</strong>: animals always in the middle means the sheep at the fence edge is missed.</li>
<li style="margin:2px 0;"><strong>Crowding</strong>: two sheep behind each other may get one box (🔴 optional below).</li>
</ul>
</details>
</div>""")

<callout: theory>

The collapsed cell defines `show_boxes`, a gallery that draws each photo's boxes in its class colour.

In [ ]:
# @title Box helpers (draw boxes on thumbnails) { display-mode: "form" }
def box_area_bucket(w: float, h: float) -> str:
    """COCO's size buckets from a box's width and height in pixels: small < 32², medium < 96², else large."""
    area = w * h
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"


def show_boxes(rows_df: pd.DataFrame, boxes_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384) -> None:
    """Gallery of the photos in `rows_df` with their boxes from `boxes_df` drawn in the class colour.

    `caption(row) -> str` writes the title above each photo; by default the class and the number of boxes.
    """
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], size)
        scale = thumb.width / row["width"]
        ax.axis("on")
        ax.imshow(thumb)
        own = boxes_df[boxes_df["image_id"] == row["id"]]
        for _, b in own.iterrows():
            x, y, w, h = b["bbox"]
            ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False,
                                               lw=2, edgecolor=CLASS_COLORS[b["class"]]))
        frame(ax, CLASS_COLORS[row["class"]])
        text = caption(row) if caption else f"{row['class']} · {len(own)} boxes"
        ax.set_title(text, fontsize=9)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

Four photos first, so you have seen what the next questions count: a flock of sheep, a lone pet, a flock of
small birds and one big horse.

In [ ]:
# @title Four photos with their boxes: sheep flock, lone pet, bird flock, big horse { display-mode: "form" }
_ann = annotations[annotations["image_id"].isin(clean_images["id"])]
_imgs = clean_images.set_index("id")
_per_photo = _ann.groupby("image_id").agg(n=("class", "size"), cls=("class", "first"),
                                          area=("area", "median"))


def _pick(classes, column, largest=True, alone=False):
    """The photo of one of `classes` with the largest (or smallest) `column`, or None."""
    c = _per_photo[_per_photo["cls"].isin(classes)]
    c = c[c["n"] == 1] if alone else c
    return None if c.empty else (c[column].idxmax() if largest else c[column].idxmin())


_picks = [_pick(["sheep", "zebra"], "n"), _pick(["cat", "dog"], "area", alone=True),
          _pick(["bird"], "n"), _pick(["horse"], "area", alone=True)]
_picks = list(dict.fromkeys(i for i in _picks if i is not None))  # skip missing ones, no repeats
show_boxes(_imgs.loc[_picks].reset_index(), _ann, "same photos, new unit: one box per animal",
           caption=lambda r: f"{r['class']} · {int(_per_photo.at[r['id'], 'n'])} box(es)")

### Boxes per class vs photos per class

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Section 4 ranked the classes by photos: cat and dog on top, sheep at the bottom. Now the same photos are
counted by animal. <em>Example:</em> one photo of 12 sheep adds 1 to the sheep's photo count and 12 to its box
count.</p>
</div>""")

<callout: questions>

**Q16.** 🟡 Will the ranking by *boxes* be the same as the ranking by *photos*? Name the class you expect to
climb most, then run the cell. It needs your first function of the section.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">count_boxes_per_class</code></div>
<p style="margin:6px 0 0 0;">Write the function in the next cell: given an annotations table (one row per animal, with a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">class</code>
column), return a dict <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">{class: number of boxes}</code> with <strong>every</strong> class in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">CLASSES</code> as a key, counts as
plain <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">int</code>s. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">use()</code> checks it against the reference and continues with the reference either way.</p>
</div>""")

<callout: your-turn>

In [ ]:
# @title Reference solution — peek only if you are stuck { display-mode: "form" }
def _ref_count_boxes_per_class(annotations):
    counts = annotations["class"].value_counts()
    return {c: int(counts.get(c, 0)) for c in CLASSES}

In [ ]:
# ✍️ YOUR CODE
def count_boxes_per_class(annotations):
    """Number of boxes (rows) per class.

    Inputs: `annotations`, a DataFrame with one row per animal and a `class` column.
    Output: a dict {class: int} with every class of CLASSES as a key (0 when a class has no box).

    e.g. {"cat": 12, "dog": 9, "bird": 30, "horse": 7, "sheep": 25, "zebra": 14}
    """
    ...


boxes = annotations[annotations["image_id"].isin(clean_images["id"])].reset_index(drop=True)
boxes_per_class = use(count_boxes_per_class, boxes, reference=_ref_count_boxes_per_class)

In [ ]:
# @title Q16 Boxes per class next to photos per class, both ranked { display-mode: "form" }
photos_per_class = {c: int((clean_images["class"] == c).sum()) for c in CLASSES}
RESULTS["boxes_per_class"] = boxes_per_class
RESULTS["n_boxes"] = int(len(boxes))

_by_photos = sorted(CLASSES, key=lambda c: -photos_per_class[c])
_by_boxes = sorted(CLASSES, key=lambda c: -boxes_per_class[c])
print(f"{len(clean_images)} photos · {len(boxes)} boxes")
print("ranked by photos: " + " > ".join(f"{c} {photos_per_class[c]}" for c in _by_photos))
print("ranked by boxes:  " + " > ".join(f"{c} {boxes_per_class[c]}" for c in _by_boxes))
for c in CLASSES:
    move = _by_photos.index(c) - _by_boxes.index(c)
    if move:
        print(f"  {c}: rank {_by_photos.index(c) + 1} by photos → rank {_by_boxes.index(c) + 1} by boxes "
              f"({'↑' if move > 0 else '↓'}{abs(move)})")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=False)
for ax, counts, order, what in zip(axes, [photos_per_class, boxes_per_class], [_by_photos, _by_boxes],
                                   ["photos", "boxes"], strict=True):
    ax.bar(order, [counts[c] for c in order], color=[CLASS_COLORS[c] for c in order])
    ax.set_title(f"{what} per class (ranked)")
    ax.set_ylabel(what)
    for lab in ax.get_xticklabels():
        lab.set_color(CLASS_COLORS[lab.get_text()])
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q16)</summary>
<p style="margin:6px 0 0 0;">No. The flock classes climb because one photo holds many animals: on the full set
bird goes from 4th by photos to 1st by boxes and sheep from last to 2nd, while the lone cat and dog drop to
the bottom.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Reading counts on a small subset</div>
<p style="margin:6px 0 0 0;">A detector is not short of sheep the way a classifier is. On a small subset, compare shapes, not digits.</p>
</div>""")

<callout: go-deeper>

### Box sizes: small, medium, large

A box's **area** is its width × height in pixels. COCO calls a box **small** under 32×32 = 1,024 px² (a
bird far away on the pond), **medium** under 96×96 = 9,216 px² and **large** above (a horse next to the
fence). The plots use the box's **side** = √(w·h), the side of a square with the same area.

**Q17.** 🟢 Which of the six classes do you expect to have the smallest boxes? Pick one, then run the cell:
it shows the smallest, a typical (median) and the largest box of the set.

In [ ]:
# @title Q17 Box size per class, COCO buckets, smallest · median · largest box { display-mode: "form" }
boxes["w"] = boxes["bbox"].map(lambda b: b[2])
boxes["h"] = boxes["bbox"].map(lambda b: b[3])
boxes["box_area"] = boxes["w"] * boxes["h"]
boxes["side"] = np.sqrt(boxes["box_area"])  # the side of a square of the same area, in pixels
boxes["bucket"] = [box_area_bucket(w, h) for w, h in zip(boxes["w"], boxes["h"], strict=True)]
bucket_share = (boxes.groupby("class")["bucket"].value_counts(normalize=True).unstack(fill_value=0)
                .reindex(index=CLASSES, columns=["small", "medium", "large"], fill_value=0))
RESULTS["small_box_share"] = {c: float(bucket_share.loc[c, "small"]) for c in CLASSES}
print("share of each class's boxes that is small / medium / large:")
display((bucket_share * 100).round(0).astype(int).T)

fig, ax = plt.subplots(figsize=(8, 3.8))
_sides = [boxes.loc[boxes["class"] == c, "side"].clip(lower=4).to_numpy() for c in CLASSES]
# Violins are drawn on log10(side) so the KDE is smooth on the log axis; ticks are relabelled in pixels.
_parts = ax.violinplot([np.log10(v) for v in _sides], showmedians=True, showextrema=False, widths=0.8)
for body, c in zip(_parts["bodies"], CLASSES, strict=True):
    body.set_facecolor(CLASS_COLORS[c])
    body.set_alpha(0.75)
_parts["cmedians"].set_color(INK)
_parts["cmedians"].set_linewidth(2.5)
_tick_names = [f"{c}\nmedian {np.median(v):.0f} px" for c, v in zip(CLASSES, _sides, strict=True)]
ax.set_xticks(range(1, len(CLASSES) + 1), _tick_names, fontsize=8)
for lab, c in zip(ax.get_xticklabels(), CLASSES, strict=True):
    lab.set_color(CLASS_COLORS[c])
ax.set_yticks(np.log10([4, 10, 32, 100, 300, 1000]), ["4", "10", "32", "100", "300", "1000"])
for thr, name in [(32, "small ↓"), (96, "large ↑")]:
    ax.axhline(np.log10(thr), color=INK, linestyle="--", linewidth=1)
    ax.text(0.5, np.log10(thr), f" {thr} px · {name}", va="bottom", fontsize=8, color=INK)
ax.set_ylabel("box side √(w·h), px (log)")
ax.set_title("Box size per class (bar = median)")
fig.tight_layout()
display(fig)
plt.close(fig)

# The smallest, a typical (median) and the largest box of the whole set: the photo with the box in bold, then
# the box cut out and enlarged (pixels shown as blocks), so a "10 px bird" is seen as the speck it is.
_by_area = boxes.sort_values(["box_area", "id"]).reset_index(drop=True)
_picks = {"smallest": _by_area.iloc[0], "median": _by_area.iloc[len(_by_area) // 2],
          "largest": _by_area.iloc[-1]}
_photo_of = clean_images.set_index("id")
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for col, (tag, b) in enumerate(_picks.items()):
    photo = _photo_of.loc[b["image_id"]]
    thumb = load_thumb(photo["path"], 384)
    scale = thumb.width / photo["width"]
    ax = axes[0, col]
    ax.imshow(thumb)
    for _, o in boxes[boxes["image_id"] == b["image_id"]].iterrows():
        ox, oy, ow, oh = o["bbox"]
        ax.add_patch(mpl.patches.Rectangle((ox * scale, oy * scale), ow * scale, oh * scale, fill=False,
                                           lw=3 if o["id"] == b["id"] else 1,
                                           edgecolor=CLASS_COLORS[o["class"]],
                                           ls="-" if o["id"] == b["id"] else ":"))
    frame(ax, CLASS_COLORS[b["class"]])
    ax.set_title(f"{tag}: {b['class']}, {b['w']:.0f}×{b['h']:.0f} px\n= {b['box_area']:,.0f} px² "
                 f"({b['bucket']})", fontsize=9)
    x, y, w, h = b["bbox"]
    pad = 0.25 * max(w, h) + 4
    x0, y0 = max(0.0, x - pad), max(0.0, y - pad)
    full = Image.open(photo["path"]).convert("RGB")
    crop = full.crop((int(x0), int(y0), int(x + w + pad), int(y + h + pad)))
    ax = axes[1, col]
    ax.imshow(crop, interpolation="nearest")
    ax.add_patch(mpl.patches.Rectangle((x - int(x0), y - int(y0)), w, h, fill=False, lw=2,
                                       edgecolor=CLASS_COLORS[b["class"]]))
    frame(ax, CLASS_COLORS[b["class"]])
    ax.set_title("the box up close", fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q17)</summary>
<p style="margin:6px 0 0 0;">Bird, by a wide margin: a large share of its boxes is <em>small</em>, while cat, dog and
horse are mostly <em>large</em>. The <strong>small share</strong> is the number to watch: small boxes are what every detector
misses first, and here they are the far-away birds the headcount needs.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Why the violin is on a log scale</div>
<p style="margin:6px 0 0 0;">Sizes range from a speck to a full frame, so the violins are drawn on log10 of the side and the ticks are
relabelled in pixels.</p>
</div>""")

<callout: go-deeper>

### Where the animals are: a box-centre heatmap

A box's **centre** is `(x + w/2, y + h/2)`. Dividing by the photo's width and height puts every centre on
the same 0–1 square, so a sheep at the left fence has a centre near x = 0.05 whatever the photo's size.
The cell below draws one heatmap per class: dark = many centres there.

In [ ]:
# @title Box-centre heatmap, one per class in its own colour { display-mode: "form" }
# Box centres on the unit square, one heatmap per class in its own colour (white = no boxes there).
_size = clean_images.set_index("id")[["width", "height"]]
_img_w = boxes["image_id"].map(_size["width"])
_img_h = boxes["image_id"].map(_size["height"])
boxes["cx"] = (boxes["bbox"].map(lambda b: b[0] + b[2] / 2) / _img_w).clip(0, 1)
boxes["cy"] = (boxes["bbox"].map(lambda b: b[1] + b[3] / 2) / _img_h).clip(0, 1)
HEAT_BINS = 6 if SMOKE else 12  # coarser cells with few boxes (N-014)
_edge_share = float(((boxes["cx"] < 0.2) | (boxes["cx"] > 0.8) | (boxes["cy"] < 0.2) | (boxes["cy"] > 0.8))
                    .mean())
RESULTS["edge_box_share"] = _edge_share

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    own = boxes[boxes["class"] == c]
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    heat, _, _ = np.histogram2d(own["cy"], own["cx"], bins=HEAT_BINS, range=[[0, 1], [0, 1]])
    ax.imshow(heat, cmap=cmap, extent=[0, 1, 1, 0], vmin=0)
    ax.set_title(f"{c} · {len(own)} boxes", fontsize=9, color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
axes[0].set_ylabel("box centre y (0 = top)")
fig.suptitle(f"Where the box centres are · {_edge_share:.0%} of all boxes are in the outer 20 % of the frame",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 The centre is darker in every panel: photographers point the camera at the animal. A park camera does
not, so a detector that learned "animals live in the middle" misses the sheep at the fence.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Why bird and sheep spread out</div>
<p style="margin:6px 0 0 0;">Bird and sheep boxes spread out more because a flock fills the frame: the one way this dataset resembles
the paddock.</p>
</div>""")

<callout: go-deeper>

### Overlap: IoU of two boxes

Two sheep standing behind each other give two boxes that share pixels. Overlap is measured with **IoU**
(intersection over union): the shared area divided by the area covered by either box. We call two boxes
*overlapping* when their IoU is above 0.1. **What it is for:** the 🔴 crowding part below counts overlapping
boxes with it, and section 8's first mask check uses it to ask "does this mask still match its box?".

In [ ]:
# @title IoU helper: box_iou_dice (used here and in section 8) { display-mode: "form" }
def box_iou_dice(a: tuple, b: tuple) -> tuple[float, float]:
    """IoU and Dice of two boxes given as (x, y, w, h)."""
    ix = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area_a, area_b = a[2] * a[3], b[2] * b[3]
    return inter / (area_a + area_b - inter), 2 * inter / (area_a + area_b)

In [ ]:
# @title IoU of two boxes, drawn: shared area ÷ covered area { display-mode: "form" }
OVERLAP_IOU = 0.1  # two boxes "overlap" when their IoU is above this

_a = (20, 20, 60, 40)
_cases = [("two sheep, one behind the other", (50, 35, 60, 40)),
          ("two sheep, corners just touch", (70, 55, 60, 40))]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, (what, _b) in zip(axes, _cases, strict=True):
    iou = box_iou_dice(_a, _b)[0]
    _ix0, _iy0 = max(_a[0], _b[0]), max(_a[1], _b[1])
    _iw = max(0, min(_a[0] + _a[2], _b[0] + _b[2]) - _ix0)
    _ih = max(0, min(_a[1] + _a[3], _b[1] + _b[3]) - _iy0)
    _inter = _iw * _ih
    _union = _a[2] * _a[3] + _b[2] * _b[3] - _inter
    ax.add_patch(mpl.patches.Rectangle(_a[:2], *_a[2:], color=CLASS_COLORS["sheep"], alpha=0.35))
    ax.add_patch(mpl.patches.Rectangle(_b[:2], *_b[2:], color=CLASS_COLORS["zebra"], alpha=0.35))
    ax.add_patch(mpl.patches.Rectangle((_ix0, _iy0), _iw, _ih, facecolor="none", edgecolor=INK, hatch="////",
                                       lw=1.5))
    ax.set_xlim(10, 140)
    ax.set_ylim(105, 10)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(f"{what}\nIoU = {_inter:,} ÷ {_union:,} = {iou:.2f} → "
                 f"{'overlapping' if iou > OVERLAP_IOU else 'not counted'}", fontsize=9)
fig.suptitle("IoU = hatched area (shared) ÷ coloured area (covered by either box)", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · How boxes and masks are scored next week</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Counting error, IoU, Dice and mAP</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Counting error</strong> (the daily headcount) = |predicted count − true count|. It is the number the farmer
  reads, and it hides compensating mistakes: one missed sheep plus one bush called sheep is a perfect count.</li>
<li style="margin:2px 0;"><strong>IoU</strong> (segmentation) = overlap ÷ union of the predicted and the true mask, per class, then averaged.
  <strong>Dice</strong> = 2 × overlap ÷ (area A + area B): the same overlap counted more kindly (Dice ≥ IoU always).</li>
<li style="margin:2px 0;"><strong>mAP</strong> (the research metric for boxes): a box counts as correct when its IoU with a true box is ≥ 0.5;
  precision and recall are traced as the confidence threshold moves, the area under that curve is the
  class's average precision, and its mean over classes is mAP. Tiny far-away birds hurt it most.</li>
</ul>
<p style="margin:6px 0 0 0;">As with the dog alert (section 4): the overall number hides the rare class, the per-class number shows it.</p>
</details>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · What handles each of these later (next week and beyond)</div>
<p style="margin:6px 0 0 0;">Each finding (box imbalance, small boxes, a wide range of sizes, photos with many boxes) has a standard
answer on the model side, so the data report can say "we saw this, the training recipe handles it".
None of them fixes a photo whose annotation is wrong: that is the data side, and it is yours.</p>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Names to look up next week</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Imbalance in boxes</strong> (bird 1st, cat last) → a <strong>weighted</strong> or <strong>focal loss</strong>, which down-weights
  the easy, frequent boxes.</li>
<li style="margin:2px 0;"><strong>Small boxes</strong> (far-away birds) → <strong>tiling</strong>: cut the photo into overlapping crops, so a 10 px bird
  becomes a 40 px bird.</li>
<li style="margin:2px 0;"><strong>A wide range of sizes</strong> (a 10 px bird and a 400 px horse in one set) → <strong>multi-scale</strong> training, so
  the detector looks at several resolutions at once.</li>
<li style="margin:2px 0;"><strong>Photos with several boxes</strong> (a "sheep" photo holding 12 boxes) → <strong>iterative stratification</strong>, the
  split of section 6 that balances box counts per class, not only photos.</li>
</ul>
</details>
</div>""")

<callout: across-tasks>

### 🔴 Optional · Crowding: overlapping boxes per class

Which box is which is hard for a detector when boxes pile up, and easy for the annotator to get wrong (one
box for two sheep, or none for the hidden one).

**Q18.** 🔴 Which class will have the most overlapping pairs of boxes per photo? *Example:* a photo of 4
sheep has 6 pairs of boxes (4×3÷2); if two sheep stand behind each other, 1 of those 6 pairs overlaps.
Decide, then run the cell: it measures every pair of boxes in every photo.

In [ ]:
# @title Q18 Boxes per photo per class, and crowding (overlapping box pairs) { display-mode: "form" }
def overlap_pairs(boxes_df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """One row per photo: n_boxes, n_pairs (all pairs) and n_overlapping (pairs with IoU > threshold)."""
    rows = []
    for image_id, own in boxes_df.groupby("image_id"):
        bb = [tuple(b) for b in own["bbox"]]
        pairs = [(i, j) for i in range(len(bb)) for j in range(i + 1, len(bb))]
        n_over = sum(box_iou_dice(bb[i], bb[j])[0] > threshold for i, j in pairs)
        rows.append({"image_id": image_id, "class": own["class"].iloc[0], "n_boxes": len(bb),
                     "n_pairs": len(bb) * (len(bb) - 1) // 2, "n_overlapping": int(n_over)})
    return pd.DataFrame(rows)


crowding = overlap_pairs(boxes, OVERLAP_IOU)
_per_class = crowding.groupby("class").agg(photos=("image_id", "size"), boxes_per_photo=("n_boxes", "mean"),
                                           crowded_photos=("n_overlapping", lambda s: float((s > 0).mean())),
                                           overlapping_pairs=("n_overlapping", "sum")).reindex(CLASSES)
_per_class["boxes_per_photo"] = _per_class["boxes_per_photo"].round(1)
_per_class["crowded_photos"] = (_per_class["crowded_photos"] * 100).round(0).astype(int).astype(str) + " %"
RESULTS["crowded_photo_share"] = float((crowding["n_overlapping"] > 0).mean())
_n_crowded = int((crowding["n_overlapping"] > 0).sum())
print("boxes_per_photo = boxes ÷ photos · crowded_photos = share of photos with ≥ 1 overlapping pair")
print(f"{_n_crowded} of {len(crowding)} photos have at least one pair of boxes with IoU > {OVERLAP_IOU} "
      f"({RESULTS['crowded_photo_share']:.0%}).")
display(_per_class.T)

N_CROWDED_BOXES = 2 if SMOKE else 4  # most crowded photos to show with their boxes (N-014)
_most = crowding.sort_values(["n_overlapping", "n_boxes", "image_id"], ascending=[False, False, True]).head(
    N_CROWDED_BOXES)
_rows = clean_images.set_index("id").loc[_most["image_id"]].reset_index()
_over_of = dict(zip(_most["image_id"], _most["n_overlapping"], strict=True))
show_boxes(_rows, boxes, "the most crowded photos (every box drawn)",
           caption=lambda r: f"{r['n_animals']} × {r['class']} · {_over_of[r['id']]} overlapping pairs",
           cols=N_CROWDED_BOXES)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q18)</summary>
<p style="margin:6px 0 0 0;">The flock classes: sheep and zebra on the full set (herds stand close), then bird.
Cats and dogs are mostly alone, so they have almost no pairs; on a small subset the leader can change, so
read the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">crowded_photos</code> row. Try counting the crowded photos' animals <em>without</em> the boxes: that is the
annotator's job, and where boxes pile up a missing or doubled box is easy to make.</p>
</details>
</div>""")

<callout: answer>

### Guided hunt: the box that is not there

The park's annotation team returned boxes for its first batch of paddock photos (`park_boxes`). In three
steps:

1. The team draws a box around every animal and sends the table back.
2. On the day, the keeper counted the animals in each photo by eye (`n_animals` in `clean_images`).
3. One box was dropped from a photo of a small flock, and a detector trained on it is told that animal is
   *background*. Which photo is decided at runtime from `SEED`; nobody can look at hundreds of photos.

In [ ]:
# @title Plant the hunt (one box removed, seeded) { display-mode: "form" }
def plant_hunt8(boxes_df: pd.DataFrame, table: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, dict]:
    """Return (park_boxes, key): `boxes_df` minus one box of a photo with ≥ 3 animals of one class.

    Sheep photos are preferred (the paddock story); otherwise the class with the most such photos. With no
    photo of 3 or more animals, the most crowded photo is used, so one box is always planted.
    """
    rng = random.Random(seed)
    counts = boxes_df.groupby("image_id").size()
    class_of = table.set_index("id")["class"]
    candidates = [int(i) for i in counts[counts >= 3].index]
    sheep = [i for i in candidates if class_of.get(i) == "sheep"]
    if sheep:
        candidates = sheep
    if not candidates:
        candidates = [int(counts.idxmax())]
    image_id = rng.choice(sorted(candidates))
    own = boxes_df[boxes_df["image_id"] == image_id]
    ann_id = int(rng.choice(sorted(own["id"])))
    park = boxes_df[boxes_df["id"] != ann_id].reset_index(drop=True)
    return park, {"image_id": image_id, "ann_id": ann_id, "class": str(class_of.get(image_id)),
                  "n_animals": int(counts[image_id])}


park_boxes, HUNT8_KEY = plant_hunt8(boxes, clean_images, SEED)
print(f"park_boxes: {len(park_boxes)} boxes for {park_boxes['image_id'].nunique()} photos "
      f"(boxes had {len(boxes)}). One box is missing. Find the photo.")

In [ ]:
# @title What a missing box looks like (toy drawing: headcount 3, boxes 2) { display-mode: "form" }
fig, ax = plt.subplots(figsize=(6, 2))
for k, x in enumerate([2, 5, 8]):
    ax.add_patch(mpl.patches.Ellipse((x, 1), 2.0, 1.2, color=CLASS_COLORS["sheep"], alpha=0.8))
    drawn = k < 2
    ax.add_patch(mpl.patches.Rectangle((x - 1.3, 0.1), 2.6, 1.8, fill=False, lw=2 if drawn else 1.5,
                                       ls="-" if drawn else "--",
                                       edgecolor=CLASS_COLORS["sheep"] if drawn else INK))
    if not drawn:
        ax.text(x, 1.0, "no box", ha="center", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 10)
ax.set_ylim(0, 2)
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)
ax.set_title("the toy version: keeper counted 3 · team drew 2 boxes · difference 1", fontsize=9)
fig.tight_layout()
display(fig)
plt.close(fig)

**Q19.** 🟡 You have two numbers for every photo: the keeper's headcount (`n_animals`) and the number of
boxes the team drew for it (the rows of `park_boxes` with that `image_id`). What simple comparison of
the two points straight at the photo with a missing box? Say it in one sentence, then run the cell.

In [ ]:
# @title Q19 Keeper's headcount vs the team's box count per photo { display-mode: "form" }
box_counts = park_boxes.groupby("image_id").size()
headcount = clean_images.set_index("id")["n_animals"]
missing_boxes = (headcount - box_counts.reindex(headcount.index, fill_value=0))
suspects = missing_boxes[missing_boxes != 0].sort_index()
hunt8_found = int(HUNT8_KEY["image_id"] in suspects.index)
RESULTS["hunt8"] = {"planted": 1, "found": hunt8_found}
print(f"{len(suspects)} photo(s) where the headcount and the box count differ:")
for image_id, diff in suspects.items():
    print(f"  photo {image_id}: headcount {headcount[image_id]}, boxes {int(box_counts.get(image_id, 0))} "
          f"({'missing' if diff > 0 else 'extra'} {abs(int(diff))})")

_rows = clean_images.set_index("id").loc[list(suspects.index)].reset_index()
show_boxes(_rows, park_boxes, "the suspect photo(s): one animal has no box",
           caption=lambda r: f"{r['class']} · headcount {r['n_animals']} · "
                             f"{int(box_counts.get(r['id'], 0))} boxes")
answer_box("The answer key: the photo whose box was removed",
           [f"planted: photo {HUNT8_KEY['image_id']} ({HUNT8_KEY['n_animals']} × {HUNT8_KEY['class']}, "
            f"one box removed) · found by the comparison: {'✅' if hunt8_found else '❌'}"])

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q19)</summary>
<p style="margin:6px 0 0 0;">Subtract the number of boxes per photo from the keeper's headcount: one photo has
one box fewer than animals, and the gallery shows the animal without a frame. Two counts of the same thing
from two sources cost nothing, which is why annotation teams are asked for a per-image count. Section 8
repeats the idea between masks and boxes, and the Challenge hides another missing box among other faults.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-pitfall" style="background:#FDECEA;color:#8B1A10;border-left:6px solid #8B1A10;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">⚠️ Pitfall · Counting different things across tasks</div>
<p style="margin:6px 0 0 0;">The same class has a different share depending on the unit:</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Sheep</strong> is about 9 % of the photos but about 22 % of the boxes.</li>
<li style="margin:2px 0;"><strong>Bird</strong> is rare by photos, common by boxes, and rare again by pixels (section 8).</li>
</ul>
<p style="margin:6px 0 0 0;">Whenever you report imbalance, size or a split, say what the unit is: photo, box or pixel. The park's three
jobs count three different things from the same photos.</p>
</div>""")

<callout: pitfall>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a class ranking → <strong>do</strong> say whether you counted photos or boxes; the order can flip
  (bird 4th by photos, 1st by boxes).</li>
<li style="margin:2px 0;"><strong>When you see</strong> many boxes under 32×32 px (far-away birds) → <strong>do</strong> report the small share and plan for
  tiling or a higher input resolution.</li>
<li style="margin:2px 0;"><strong>When you see</strong> box centres piled in the middle → <strong>do</strong> expect misses at the frame edges, and test on
  photos where the animal is off-centre.</li>
<li style="margin:2px 0;"><strong>When you see</strong> the keeper's headcount and the team's box count disagree on a photo → <strong>do</strong> open it:
  a box is missing or doubled.</li>
</ul>
</div>""")

<callout: takeaway>

## 8. Same data, masks

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">See what imbalance and object size mean once the park wants an <em>outline</em> of every animal, and find five
ugly masks hidden among the real ones.</p>
</div>""")

<callout: objective>

A **mask** is the exact pixels an animal covers. `annotations` holds one in its `segmentation` column, and
`pycocotools` turns it into a 0/1 array the size of the photo.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · How a mask is stored: polygon and RLE</div>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Two storage formats, and why RLE is handy</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>polygon</strong>: the x, y points an annotator clicked around the animal;</li>
<li style="margin:2px 0;"><strong>RLE</strong> (run-length encoding, used for crowds): "300 background pixels, then 25 animal pixels, then …".</li>
</ul>
<p style="margin:6px 0 0 0;">The cell that builds the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">masks</code> table turns every mask into one compressed RLE and counts its pixels.
RLE is small and fast: the union and the area of masks are computed without building the full 0/1 array.</p>
</details>
<p style="margin:6px 0 0 0;">The collapsed cell above the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">masks</code> table defines the mask tools: decode and encode RLE, measure a mask's
shape, and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">show_masks</code>, a gallery that paints each mask in its class colour over the photo, with its
annotated box dashed.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · What changes when the unit is a pixel</div>
<p style="margin:6px 0 0 0;">A segmentation model makes one decision per <strong>pixel</strong>: which class is this pixel, or is it background?</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Imbalance is counted in pixels, background included.</strong> Most of every photo is grass, sky and water.</li>
<li style="margin:2px 0;"><strong>Size is everything.</strong> A bird of 10×10 px is 100 decisions; a horse filling the frame is 100,000. The
  horse adds about 1,000 times more pixels to the training error than the bird, so a model mostly learns
  horses.</li>
</ul>
</div>""")

<callout: theory>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Mask quality, thin parts, and where the animals sit</div>
<p style="margin:6px 0 0 0;">A box is four numbers; a mask is dozens of clicks, so a bad mask looks like any other row. Legs, tails,
beaks and far-away birds are few pixels wide, and they are the first to vanish when the photo is shrunk to
the network's input size. A big horse outweighs a whole flock of far-away birds. The optional part at the
end of this section measures thin parts, and draws where the animal pixels sit in the frame.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title Mask helpers (decode, encode, measure, draw) { display-mode: "form" }
import cv2
from pycocotools import mask as mask_utils

MIN_JUDGED_AREA = 32**2  # masks under COCO's "small" limit (32×32 px) are too few pixels to judge their shape
THIN_PX = 7  # parts narrower than this many photo pixels count as thin (about 2 px at 224×224)
BLOB_SHARE = 0.005  # pieces and holes under 0.5 % of the mask (or 16 px) are rasterisation noise, not shape


def to_rle(segmentation, height: int, width: int) -> dict:
    """A COCO segmentation (polygon list, uncompressed or compressed RLE) as one compressed RLE."""
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation["counts"], list):
        return mask_utils.frPyObjects(segmentation, height, width)
    return segmentation


def decode(rle: dict) -> np.ndarray:
    """RLE → 0/1 uint8 array of the photo's size (C-ordered, so OpenCV accepts it)."""
    return np.ascontiguousarray(mask_utils.decode(rle))


def encode(mask: np.ndarray) -> dict:
    """0/1 array → compressed RLE, the way COCO stores masks."""
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))


def thin_part(mask: np.ndarray) -> np.ndarray:
    """The pixels of `mask` in parts narrower than THIN_PX: what an opening with a THIN_PX disc removes."""
    disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (THIN_PX, THIN_PX))
    return mask & (1 - cv2.morphologyEx(mask, cv2.MORPH_OPEN, disc))


def mask_shape(mask: np.ndarray, bbox) -> dict:
    """Shape numbers of one mask (0/1 array) against its annotated box (x, y, w, h).

    box_iou: IoU of the mask's own tight box with the annotated box · fill: mask area / box area ·
    pieces, holes: separate parts and enclosed gaps · solidity: mask area / area of its convex hull (a rubber
    band around it; 1.0 = no dents at all) · thin: share of the mask's pixels in thin parts.
    """
    area = int(mask.sum())
    if area == 0:
        return {"box_iou": 0.0, "fill": 0.0, "pieces": 0, "holes": 0, "solidity": 0.0, "thin": 0.0}
    mx, my, mw, mh = cv2.boundingRect(mask)
    crop = np.pad(mask[my:my + mh, mx:mx + mw], 1)
    min_blob = max(16, BLOB_SHARE * area)
    _, _, parts, _ = cv2.connectedComponentsWithStats(crop, connectivity=8)
    pieces = int((parts[1:, cv2.CC_STAT_AREA] >= min_blob).sum())
    _, labels, gaps, _ = cv2.connectedComponentsWithStats(1 - crop, connectivity=4)
    outside = labels[0, 0]  # the background that touches the border is not a hole
    holes = sum(int(gaps[i, cv2.CC_STAT_AREA] >= min_blob) for i in range(1, len(gaps)) if i != outside)
    contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = np.zeros_like(crop)
    cv2.fillPoly(hull, [cv2.convexHull(np.vstack(contours))], 1)
    w, h = bbox[2], bbox[3]
    return {"box_iou": box_iou_dice((mx, my, mw, mh), tuple(bbox))[0],
            "fill": area / max(w * h, 1),
            "pieces": pieces,
            "holes": holes,
            "solidity": area / max(int(hull.sum()), 1),
            "thin": int(thin_part(crop).sum()) / area}


def show_masks(rows_df: pd.DataFrame, masks_df: pd.DataFrame, title: str, caption=None, cols: int = 4,
               size: int = 384, mark_thin: bool = False, highlight: dict | None = None) -> None:
    """Gallery of the photos in `rows_df` with every mask of `masks_df` (column `rle`) painted in its class
    colour and its annotated box dashed. `mark_thin=True` paints the thin parts dark.
    `highlight={image_id: mask id}` draws that one mask bright with a solid box and fades the others.

    `caption(row) -> str` writes the title above each photo; by default the class and the number of masks.
    """
    n = len(rows_df)
    if n == 0:
        print(f"{title}: no photos to show.")
        return
    cols = min(cols, n)
    n_rows = -(-n // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(3.4 * cols, 3.2 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, row) in zip(axes.ravel(), rows_df.iterrows(), strict=False):
        thumb = load_thumb(row["path"], size)
        scale = thumb.width / row["width"]
        overlay = np.zeros((thumb.height, thumb.width, 4))
        own = masks_df[masks_df["image_id"] == row["id"]]
        flagged = (highlight or {}).get(row["id"])
        if flagged is not None:  # the flagged mask goes last, so it is painted on top
            own = own.assign(_top=own["id"] == flagged).sort_values("_top")
        for _, m in own.iterrows():
            full = decode(m["rle"])
            small = cv2.resize(full, thumb.size, interpolation=cv2.INTER_NEAREST) > 0  # nearest: section 3
            alpha = 0.6 if flagged is None else (0.8 if m["id"] == flagged else 0.15)
            overlay[small] = (*mpl.colors.to_rgb(CLASS_COLORS[m["class"]]), alpha)
            if mark_thin:
                thin = cv2.resize(thin_part(full), thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                overlay[thin] = (*mpl.colors.to_rgb(INK), 0.95)
        ax.axis("on")
        ax.imshow(thumb)
        ax.imshow(overlay)
        for _, m in own.iterrows():
            x, y, w, h = m["bbox"]
            if flagged is not None and m["id"] != flagged:
                continue  # with a highlight, only the suspect's box is drawn
            ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False,
                                               lw=2.5 if flagged is not None else 1.5,
                                               ls="-" if flagged is not None else "--",
                                               edgecolor=CLASS_COLORS[m["class"]]))
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(row) if caption else f"{row['class']} · {len(own)} masks", fontsize=9)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

The next cell turns every mask of `clean_images` into one compressed RLE (the `rle` column of a new table,
`masks`) and counts its pixels.

In [ ]:
# @title Build `masks`: one RLE per mask, with its pixel count { display-mode: "form" }
# One row per mask of the photos in clean_images (crowd regions included), as RLE, with its pixel count.
_size = clean_images.set_index("id")[["width", "height"]]
masks = boxes[["id", "image_id", "class", "bbox", "iscrowd", "segmentation"]].copy()
masks["rle"] = [to_rle(seg, int(_size.at[i, "height"]), int(_size.at[i, "width"]))
                for seg, i in zip(masks["segmentation"], masks["image_id"], strict=True)]
masks["mask_area"] = [int(mask_utils.area(r)) for r in masks["rle"]]
masks = masks.drop(columns="segmentation")
print(f"{len(masks)} masks on {masks['image_id'].nunique()} photos "
      f"({int(masks['iscrowd'].sum())} of them crowd regions: one mask for a whole group)")
masks.head(3)

Four photos with their masks painted, before any number: a close-up pet, a flock of small birds, horses
with thin legs, and a flock of sheep.

In [ ]:
# @title Four photos with their masks: close-up pet, small birds, thin legs, a flock { display-mode: "form" }
_mimgs = clean_images.set_index("id")
_mper = masks.groupby("image_id").agg(n=("class", "size"), cls=("class", "first"),
                                      px=("mask_area", "sum"))
_mper["share"] = _mper["px"] / (_mimgs.loc[_mper.index, "width"] * _mimgs.loc[_mper.index, "height"])
_mpicks = []
for _cls, _how in [(["cat", "dog"], "largest share"), (["bird"], "most masks"),
                   (["horse"], "largest share"), (["sheep", "zebra"], "most masks")]:
    _c = _mper[_mper["cls"].isin(_cls) & ~_mper.index.isin(_mpicks)]
    if len(_c):
        _key = _c["share"].idxmax() if _how == "largest share" else _c["n"].idxmax()
        _mpicks.append(_key)
show_masks(_mimgs.loc[_mpicks].reset_index(), masks, "same photos, third unit: every pixel of every animal",
           caption=lambda r: f"{r['class']} · animals cover {_mper.at[r['id'], 'share']:.0%} of the photo")

### Pixels per class, background included

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">Every pixel that is not an animal is <em>background</em>, a class like any other for the model.</p>
</div>""")

<callout: questions>

**Q20.** 🟢 What share of all pixels do you expect to be background: a quarter, a half, or most? Is bird
frequent or rare by pixels? Guess both, then run.

In [ ]:
# @title Q20 Pixels per class including background, next to the photo and box shares { display-mode: "form" }
photo_rle = {}  # one merged mask per photo, reused by the heatmaps below
_animal_px = dict.fromkeys(CLASSES, 0)
for image_id, own in masks.groupby("image_id"):
    photo_rle[image_id] = mask_utils.merge(list(own["rle"]), intersect=False)
    _animal_px[own["class"].iloc[0]] += int(mask_utils.area(photo_rle[image_id]))
_total_px = int((clean_images["width"] * clean_images["height"]).sum())
pixels_per_class = {c: _animal_px[c] / _total_px for c in CLASSES}
pixels_per_class["background"] = (_total_px - sum(_animal_px.values())) / _total_px
RESULTS["pixels_per_class"] = pixels_per_class

BACKGROUND_COLOR = "#D9DEDF"  # neutral grey: background is not an animal, so it gets no class colour
_units = {
    "photos": {c: photos_per_class[c] / sum(photos_per_class.values()) for c in CLASSES},
    "boxes": {c: boxes_per_class[c] / sum(boxes_per_class.values()) for c in CLASSES},
    "pixels": {c: _animal_px[c] / max(sum(_animal_px.values()), 1) for c in CLASSES},
}
print(f"{_total_px / 1e6:.0f} million pixels · background {pixels_per_class['background']:.1%} · "
      f"all animals together {1 - pixels_per_class['background']:.1%}")
print("share of ALL pixels:  " + " · ".join(f"{c} {pixels_per_class[c]:.1%}" for c in CLASSES))
print("among animals only, by photos / boxes / pixels:")
for c in CLASSES:
    print(f"  {c:6s} {_units['photos'][c]:5.0%}  {_units['boxes'][c]:5.0%}  {_units['pixels'][c]:5.0%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
_x = np.arange(len(CLASSES))
for k, (unit, alpha) in enumerate([("photos", 0.3), ("boxes", 0.6), ("pixels", 1.0)]):
    ax.bar(_x + (k - 1) * 0.27, [_units[unit][c] for c in CLASSES], width=0.27, alpha=alpha,
           color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white", label=unit)
ax.set_xticks(_x, CLASSES)
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("Share among the animals: photos (pale) · boxes · pixels (solid)")
_legend = [mpl.patches.Patch(color=INK, alpha=a, label=u) for u, a in [("photos", 0.3), ("boxes", 0.6),
                                                                         ("pixels", 1.0)]]
ax.legend(handles=_legend, frameon=False, fontsize=8)

ax = axes[1]
_left = 0.0
for name in ["background", *CLASSES]:
    share = pixels_per_class[name]
    ax.barh(0, share, left=_left, color=BACKGROUND_COLOR if name == "background" else CLASS_COLORS[name],
            edgecolor="white")
    if share > 0.06:
        ax.text(_left + share / 2, 0, f"{name}\n{share:.0%}", ha="center", va="center", fontsize=8, color=INK)
    _left += share
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("All pixels, background included")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q20)</summary>
<p style="margin:6px 0 0 0;">Most of them: about 85 % of all pixels are background (the smoke subset gives about
80 %). Bird is rare again: about 1 % of all pixels (6–7 % of the animal pixels), because most birds are far
away, while cat and dog come back (a pet photo is a close-up). Photos, boxes, pixels: three rankings of one
dataset, so "bird is rare" needs its unit.</p>
</details>
</div>""")

<callout: answer>

### How big is a mask, and how much of its box does it fill?

A mask sits inside its box, so a second number comes for free: the **fill ratio**, mask area divided by box
area. A box around a curled-up cat is mostly cat; a box around a horse is mostly the air between its legs.
The cell plots mask size and fill ratio per class, then shows the photos with the lowest, a typical
(median) and the highest fill ratio.

**Q21.** 🟡 What fraction of its box does a typical animal fill: about 25 %, 50 % or 80 %? Which class do
you expect to fill its box least? Guess, then run.

In [ ]:
# @title Q21 Shape numbers of every mask, then mask area and fill ratio per class { display-mode: "form" }
animals = masks[masks["iscrowd"] == 0].reset_index(drop=True)
_shape = pd.DataFrame([mask_shape(decode(r), b)
                       for r, b in zip(animals["rle"], animals["bbox"], strict=True)])
animals = pd.concat([animals, _shape], axis=1)
_photo_px = animals["image_id"].map(_size["width"] * _size["height"])
animals["photo_share"] = animals["mask_area"] / _photo_px
_fill = animals.groupby("class")["fill"].median().reindex(CLASSES)
RESULTS["median_fill"] = {c: float(_fill[c]) for c in CLASSES}
print(f"{len(animals)} animal masks measured · median fill ratio " + " · ".join(
    f"{c} {_fill[c]:.0%}" for c in CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, col, label, log in [(axes[0], "photo_share", "mask area, share of the photo (log)", True),
                            (axes[1], "fill", "fill ratio = mask area / box area", False)]:
    data = [animals.loc[animals["class"] == c, col].clip(lower=1e-5).to_numpy() for c in CLASSES]
    if log:  # violins are drawn on log10(value) so the KDE is smooth on a log axis; ticks are relabelled
        data = [np.log10(v) for v in data]
    parts = ax.violinplot(data, showmedians=True, showextrema=False, widths=0.8)
    for body, c in zip(parts["bodies"], CLASSES, strict=True):
        body.set_facecolor(CLASS_COLORS[c])
        body.set_alpha(0.75)
    parts["cmedians"].set_color(INK)
    ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES)
    if log:
        _ticks = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1]
        ax.set_yticks(np.log10(_ticks), [f"{100 * t:g}%" for t in _ticks])
    else:
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
    ax.set_ylabel(label)
    for lab in ax.get_xticklabels():
        lab.set_color(CLASS_COLORS[lab.get_text()])
axes[0].set_title("How big is a mask?")
axes[1].set_title("How much of its box does it fill?")
fig.tight_layout()
display(fig)
plt.close(fig)

N_EXTREME_MASKS = 1 if SMOKE else 2  # photos at each end of the fill ratio to show (N-014)
_judged = (animals[animals["mask_area"] >= MIN_JUDGED_AREA].sort_values(["fill", "id"])
           .drop_duplicates("image_id").reset_index(drop=True))
_ends = pd.concat([_judged.head(N_EXTREME_MASKS), _judged.iloc[[len(_judged) // 2]],
                   _judged.tail(N_EXTREME_MASKS)]).drop_duplicates("image_id")
_fill_of = dict(zip(_ends["image_id"], _ends["fill"], strict=True))
_tags = ["lowest"] * N_EXTREME_MASKS + ["median"] + ["highest"] * N_EXTREME_MASKS
_tag_of = dict(zip(_ends["image_id"], _tags, strict=False))
show_masks(clean_images.set_index("id").loc[list(_ends["image_id"])].reset_index(), masks,
           "lowest · typical (median) · highest fill ratio",
           caption=lambda r: f"{_tag_of[r['id']]}: {r['class']}, fill {_fill_of[r['id']]:.0%}",
           cols=len(_ends))

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q21)</summary>
<p style="margin:6px 0 0 0;">About half, in a narrow band of roughly 40–60 % for every class: near 45 % for bird
and horse (wings, legs, necks), near 55 % for cat and sheep (compact shapes). The low end of the gallery
shows spread wings and diagonal poses, the high end close-ups cut by the photo's edge. That band is what
makes fill a quality check: a mask filling 98 % of its box is a rectangle, one filling 5 % lost the animal.
(Crowd regions are left out: they outline a group, not an animal.)</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-across-tasks" style="background:#EDE7F6;color:#311B92;border-left:6px solid #311B92;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔀 Across tasks · What handles each of these later (next week and beyond)</div>
<p style="margin:6px 0 0 0;">Imbalance and tiny animals have a model-side answer. None of it fixes a wrong mask: that is the data side.</p>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">Names to look up next week</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Pixel imbalance</strong> (background about 85 %, bird about 1 %) → a <strong>class-weighted</strong> cross-entropy or a
  <strong>Dice loss</strong>, which scores each class's overlap on its own scale, and reporting <strong>mean IoU over
  classes</strong> (section 7), so that birds count as much as background.</li>
<li style="margin:2px 0;"><strong>Tiny animals and thin parts</strong> → a higher input resolution, or <strong>crops / tiles</strong> of the photo,
  so that a 7 px leg is still 7 px for the network; boundary-aware losses weigh the outline more.</li>
<li style="margin:2px 0;"><strong>Everything in the centre</strong> → random crops and shifts when training, so the edge of the frame is
  not always background.</li>
</ul>
</details>
</div>""")

<callout: across-tasks>

### Guided hunt: five ugly masks

The park paid an annotation team per mask, and the masks came back as `park_masks`: `masks` with **five
masks spoiled**, picked from `SEED`, one of each kind a team really delivers:

- **swollen**: the outline drawn too generously;
- **shifted**: the right outline, about 15 px off;
- **a chunk missing**: a piece of the body cut out;
- **another animal's mask** pasted onto this one;
- **a crude polygon**: five or six clicks in a hurry.

They hide among hundreds of real masks. **Your job is not to look through hundreds of masks.** It is to
know which check catches which kind (Q22), so that a script does the looking and you read its short list.

In [ ]:
# @title Plant the hunt (five ugly masks, seeded) { display-mode: "form" }
HUNT9_KINDS = ["another animal", "chunk missing", "shifted", "swollen", "crude polygon"]
SHIFT_PX = 15


def _paste(mask: np.ndarray, height: int, width: int) -> np.ndarray:
    """`mask` placed at the same coordinates on an empty canvas of another photo's size."""
    out = np.zeros((height, width), np.uint8)
    h, w = min(height, mask.shape[0]), min(width, mask.shape[1])
    out[:h, :w] = mask[:h, :w]
    return out


def _crude_polygon(mask: np.ndarray, max_vertices: int = 6) -> list[float]:
    """A polygon of at most `max_vertices` points around the mask (its convex hull, simplified)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(np.vstack(contours))
    eps = 0.02 * cv2.arcLength(hull, True)
    poly = cv2.approxPolyDP(hull, eps, True)
    while len(poly) > max_vertices:
        eps *= 1.5
        poly = cv2.approxPolyDP(hull, eps, True)
    return [float(v) for v in poly.reshape(-1)]


def plant_hunt9(animals_df: pd.DataFrame, table: pd.DataFrame,
                seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (park_masks, key): the masks of `animals_df` with one ugly mask of each of HUNT9_KINDS.

    Each kind spoils a different photo's mask, picked in a seeded order among masks large enough to judge,
    and the mask is changed through pycocotools (decode → OpenCV → encode). With no mask fitting a kind's
    condition (a small subset), the first free one is used, so five masks are always planted.
    """
    rng = random.Random(seed)
    size = table.set_index("id")[["width", "height"]]
    pool = animals_df[animals_df["mask_area"] >= MIN_JUDGED_AREA].set_index("id")
    order = sorted(pool.index)
    rng.shuffle(order)
    used, new_rle, key = set(), {}, []

    def fits(kind: str, ann_id: int):
        """The spoiled mask as a 0/1 array (or an RLE), plus a note, or None when this mask does not suit."""
        a = pool.loc[ann_id]
        x, y, w, h = a["bbox"]
        W, H = int(size.at[a["image_id"], "width"]), int(size.at[a["image_id"], "height"])
        if kind == "another animal":  # prefer a neighbour in the same photo: a duplicated mask, as in COCO
            others = pool[pool["image_id"] != a["image_id"]].index
            same = pool[(pool["image_id"] == a["image_id"]) & (pool.index != ann_id)].index
            for b in [*sorted(same), *[i for i in order if i in others]]:
                donor = _paste(decode(pool.at[b, "rle"]), H, W)
                if donor.any() and box_iou_dice(cv2.boundingRect(donor), tuple(a["bbox"]))[0] < 0.3:
                    where = "next to it" if b in same else "from another photo"
                    return donor, f"the mask of a {pool.at[b, 'class']} {where}"
            return None
        mask = decode(a["rle"])
        if kind == "chunk missing":
            dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
            r = float(dist.max())
            if r < max(15, 0.15 * a["mask_area"] ** 0.5):
                return None
            cy, cx = np.unravel_index(int(dist.argmax()), dist.shape)
            out = mask.copy()
            cv2.circle(out, (int(cx), int(cy)), int(0.8 * r), 0, -1)
            return out, f"a disc of radius {int(0.8 * r)} px cut out of the body"
        if kind == "shifted":
            if x + w + SHIFT_PX > W or y + h + SHIFT_PX > H or max(w, h) > 300:
                return None
            move = np.float32([[1, 0, SHIFT_PX], [0, 1, SHIFT_PX]])
            moved = cv2.warpAffine(mask, move, (W, H), flags=cv2.INTER_NEAREST)
            return moved, f"moved {SHIFT_PX} px right and down"
        if kind == "swollen":
            d = max(6, round(0.08 * (w * h) ** 0.5))
            if x < d or y < d or x + w + d > W or y + h + d > H:
                return None
            disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * d + 1, 2 * d + 1))
            return cv2.dilate(mask, disc), f"grown by {d} px all round"
        if kind == "crude polygon":
            if a["solidity"] > 0.85:  # pick an animal with clear dents, so the shortcut shows
                return None
            poly = _crude_polygon(mask)
            return to_rle([poly], H, W), f"{len(poly) // 2} clicked points"
        raise ValueError(kind)

    for kind in HUNT9_KINDS:
        free = [i for i in order if pool.at[i, "image_id"] not in used and i not in new_rle]
        spoiled = next(((i, out) for i in free if (out := fits(kind, i)) is not None), None)
        if spoiled is None:  # nothing fits the condition: give the first free mask another photo's mask
            i, j = free[0], free[1]
            height, width = decode(pool.at[i, "rle"]).shape
            pasted = _paste(decode(pool.at[j, "rle"]), height, width)
            spoiled = (i, (pasted, "another photo's mask (fallback)"))
        ann_id, (out, note) = spoiled
        new_rle[ann_id] = out if isinstance(out, dict) else encode(out)
        used.add(pool.at[ann_id, "image_id"])
        key.append({"kind": kind, "ann_id": int(ann_id), "image_id": int(pool.at[ann_id, "image_id"]),
                    "class": pool.at[ann_id, "class"], "how": note})
    park = animals_df[["id", "image_id", "class", "bbox", "iscrowd"]].copy()
    park["rle"] = [new_rle.get(i, r) for i, r in zip(animals_df["id"], animals_df["rle"], strict=True)]
    park["mask_area"] = [int(mask_utils.area(r)) for r in park["rle"]]
    return park, pd.DataFrame(key)


park_masks, HUNT9_KEY = plant_hunt9(animals, clean_images, SEED)
print(f"park_masks: {len(park_masks)} animal masks on {park_masks['image_id'].nunique()} photos. "
      "Five of them are ugly. Find them.")

First, what the five kinds look like. The next picture spoils one toy horse in the five ways: the grey line
is the real outline, the dashed box is the box the team drew from it, the coloured shape is what they sent.

In [ ]:
# @title The five kinds of ugly mask, drawn on one toy horse { display-mode: "form" }
def toy_horse(height: int = 60, width: int = 84) -> np.ndarray:
    """A toy horse mask (an oval body and four legs), 0/1, used by the two toy pictures of the hunt."""
    yy, xx = np.mgrid[0:height, 0:width]
    horse = ((xx - 40) / 24) ** 2 + ((yy - 32) / 13) ** 2 <= 1
    for lx in (24, 32, 46, 54):
        horse |= (xx >= lx) & (xx < lx + 5) & (yy >= 38) & (yy < 56)
    return horse.astype(np.uint8)


_horse = toy_horse()
_kx, _ky, _kw, _kh = cv2.boundingRect(_horse)  # the team's box, drawn from the real outline
_kyy, _kxx = np.mgrid[0 : _horse.shape[0], 0 : _horse.shape[1]]
_poly = np.array(_crude_polygon(_horse), np.int32).reshape(-1, 1, 2)
_bite = ((_kxx - 40) ** 2 + (_kyy - 30) ** 2 <= 36).astype(np.uint8)  # a disc inside the body
_other = (((_kxx - 66) / 13) ** 2 + ((_kyy - 18) / 9) ** 2 <= 1).astype(np.uint8)  # another animal
_kinds = {
    "swollen\n(drawn too generously)": cv2.dilate(_horse, np.ones((7, 7), np.uint8)),
    "shifted\n(right outline, moved)": np.roll(_horse, (4, 9), axis=(0, 1)),
    "chunk missing\n(a piece cut out)": _horse & (1 - _bite),
    "another animal's mask\n(pasted here)": _other,
    "crude polygon\n(six quick clicks)": cv2.fillPoly(np.zeros_like(_horse), [_poly], 1),
}
_outline, _ = cv2.findContours(_horse, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.7))
for ax, (kind, spoiled) in zip(axes, _kinds.items(), strict=True):
    layer = np.zeros((*_horse.shape, 4))
    layer[spoiled > 0] = (*mpl.colors.to_rgb(CLASS_COLORS["horse"]), 0.85)
    ax.imshow(layer)
    for c in _outline:
        ax.plot(c[:, 0, 0], c[:, 0, 1], color="#9AA5A8", lw=1)
    ax.add_patch(mpl.patches.Rectangle((_kx - 0.5, _ky - 0.5), _kw, _kh, fill=False, lw=1.5, ls="--",
                                       edgecolor=INK))
    ax.set_xlim(-0.5, _horse.shape[1] - 0.5)
    ax.set_ylim(_horse.shape[0] - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    ax.set_title(kind, fontsize=9)
fig.suptitle("Five ways a delivered mask goes wrong (grey: real outline · dashed: the team's box)", color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

Then the five checks, all computed from the mask and its annotated box, each with the rule that makes a mask
a suspect. The picture after the list draws each check on one toy mask.

1. **mask–box agreement** (the only check that uses IoU): draw the tightest box around the mask and compare
   it with the team's box; their IoU (section 7) is 1 when they match. Suspect when it is below `IOU_MIN`
   (0.9), **the one value you can change**.
2. **fill ratio**: mask area / box area, from Q21. Suspect outside 15–95 %.
3. **pieces**: separate parts of the mask. Suspect with more than 3.
4. **holes**: gaps fully inside the mask. Suspect with any hole.
5. **solidity**: mask area / area of its **convex hull**, the "rubber band" around the whole mask. Suspect
   above 0.97: a real animal has dents (between the legs, under the neck).

<details><summary>What is normal for a real animal</summary>

Box agreement is almost always above 0.95 on real COCO masks, because the box was made from the mask. An
animal is usually one piece (two or three when something stands in front of it) and almost never has a
hole. It has dents (between the legs, under the neck), so its solidity is rarely close to 1. Only masks
of at least 32×32 px are judged: below that, a few pixels of rounding change every number.

</details>

In [ ]:
# @title The five checks drawn on one toy mask { display-mode: "form" }
# A toy "horse" (body, four legs, a hole, a separate piece) and its sloppily drawn box, measured with the same
# mask_shape() that the hunt uses, so the numbers in the titles are the numbers the checks will compute.
_H, _W = 60, 84
_yy, _xx = np.mgrid[0:_H, 0:_W]
_toy = toy_horse(_H, _W).astype(bool)  # the body and four legs
_toy |= (_xx >= 68) & (_xx < 76) & (_yy >= 30) & (_yy < 38)  # a separate piece, away from the body
_hole = (_xx - 40) ** 2 + (_yy - 28) ** 2 <= 16
_toy = (_toy & ~_hole).astype(np.uint8)
_tx, _ty, _tw, _th = cv2.boundingRect(_toy)  # the mask's own tight box
_drawn = (_tx + 5, _ty + 3, _tw, _th)  # the annotator's box, a bit off
_s = mask_shape(_toy, _drawn)


def _layer(mask, color, alpha):
    out = np.zeros((_H, _W, 4))
    out[mask > 0] = (*mpl.colors.to_rgb(color), alpha)
    return out


_n_parts, _parts_lab = cv2.connectedComponents(_toy, connectivity=8)
_part_sizes = {k: int((_parts_lab == k).sum()) for k in range(1, _n_parts)}
_main_part = max(_part_sizes, key=_part_sizes.get)
_contours, _ = cv2.findContours(_toy, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
_hull = cv2.convexHull(np.vstack(_contours))[:, 0, :]

fig, axes = plt.subplots(1, 5, figsize=(13, 2.9))
for ax in axes:
    ax.imshow(_layer(_toy, CLASS_COLORS["horse"], 0.35))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
ax = axes[0]  # 1 · mask-box agreement: the mask's own box (solid) against the drawn box (dashed)
ax.add_patch(mpl.patches.Rectangle((_tx - 0.5, _ty - 0.5), _tw, _th, fill=False, lw=2, edgecolor=INK))
ax.add_patch(mpl.patches.Rectangle((_drawn[0] - 0.5, _drawn[1] - 0.5), _drawn[2], _drawn[3], fill=False, lw=2,
                                   ls="--", edgecolor=CLASS_COLORS["sheep"]))
ax.set_title(f"1 · mask-box agreement\nsolid: mask's own box · dashed: drawn\nIoU = {_s['box_iou']:.2f}",
             fontsize=8)
ax = axes[1]  # 2 · fill: mask pixels out of the drawn box's pixels
ax.imshow(_layer(_toy, CLASS_COLORS["horse"], 0.9))
ax.add_patch(mpl.patches.Rectangle((_drawn[0] - 0.5, _drawn[1] - 0.5), _drawn[2], _drawn[3], fill=False, lw=2,
                                   ls="--", edgecolor=CLASS_COLORS["sheep"]))
ax.set_title(f"2 · fill = mask ÷ drawn box\n{int(_toy.sum()):,} ÷ {_drawn[2] * _drawn[3]:,} "
             f"= {_s['fill']:.2f}", fontsize=8)
ax = axes[2]  # 3 · pieces: each connected part, the main one in colour, the others dark
for k in _part_sizes:
    ax.imshow(_layer(_parts_lab == k, CLASS_COLORS["horse"] if k == _main_part else INK, 0.9))
ax.set_title(f"3 · pieces\nseparate parts (dark = extra)\n= {_s['pieces']}", fontsize=8)
ax = axes[3]  # 4 · holes: the enclosed gaps
ax.imshow(_layer(_toy, CLASS_COLORS["horse"], 0.9))
ax.imshow(_layer(_hole, INK, 1.0))
ax.annotate("hole", xy=(40, 28), xytext=(40, 6), ha="center", fontsize=8, color=INK,
            arrowprops={"arrowstyle": "->", "color": INK})
ax.set_title(f"4 · holes\ngaps fully inside the mask\n= {_s['holes']}", fontsize=8)
ax = axes[4]  # 5 · solidity: mask area over the area of its convex hull (the rubber band)
ax.add_patch(mpl.patches.Polygon(_hull, closed=True, fill=False, lw=2, ls="--", edgecolor=INK))
ax.imshow(_layer(_toy, CLASS_COLORS["horse"], 0.9))
ax.set_title(f"5 · solidity = mask ÷ rubber band\n(dashed: the rubber band)\n= {_s['solidity']:.2f}",
             fontsize=8)
for ax in axes:
    ax.set_xlim(-0.5, _W - 0.5)
    ax.set_ylim(_H - 0.5, -0.5)
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">IOU_MIN</code>, the threshold of check 1 only</div>
<p style="margin:6px 0 0 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">IOU_MIN</code> tunes <strong>check 1 (mask–box agreement) and nothing else</strong>; checks 2 to 5 keep their fixed rules.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>First run:</strong> leave it at 0.9 and answer Q22.</li>
<li style="margin:2px 0;"><strong>After the answer key:</strong> come back, set it to 0.97, rerun this cell, the Q22 cell and the key, and count
  how many <em>real</em> masks join the suspects (a stricter check costs more human looking time). Then try 0.5
  and see which planted mask you lose.</li>
</ul>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
IOU_MIN = 0.9  # suspect when the mask's own box overlaps the annotated box less than this (IoU, 0–1)

if not (isinstance(IOU_MIN, (int, float)) and 0 <= IOU_MIN <= 1):
    raise SystemExit("IOU_MIN is a number between 0 and 1.")

**Q22.** 🟡 Match each kind (first picture) to the check you expect to catch it (the numbered list):
write five pairs, e.g. *shifted → mask–box agreement*. Some kinds trip more than one check, and one check
may catch none.

Then run the next cell and read its output in this order:

1. **The count per check:** how many masks each check flags. Most of them are real COCO masks, not planted.
2. **The gallery of the strongest suspects** (the masks that tripped the most checks). In each photo the
   **bright mask with the solid box** is the suspect; the faint ones are the other animals. Its title gives
   the checks that fired and its numbers. For each photo, guess: planted, or a real COCO mask?
3. **The answer key** below: compare it with your five pairs and your guesses.

In [ ]:
# @title Q22 Measure every judged mask, flag the suspects, score the hunt { display-mode: "form" }
FILL_RANGE = (0.15, 0.95)  # suspect when mask area / box area is outside this range
MAX_PIECES = 3  # suspect with more separate pieces than this
MAX_HOLES = 0  # suspect with more holes than this
SOLIDITY_MAX = 0.97  # suspect when the mask fills more than this share of its rubber band (no dents at all)

_judge = park_masks[park_masks["mask_area"] >= MIN_JUDGED_AREA].reset_index(drop=True)
park_shape = pd.concat([_judge, pd.DataFrame([mask_shape(decode(r), b)
                                              for r, b in zip(_judge["rle"], _judge["bbox"], strict=True)])],
                       axis=1)
CHECKS = {
    "box IoU": park_shape["box_iou"] < IOU_MIN,
    "fill": (park_shape["fill"] < FILL_RANGE[0]) | (park_shape["fill"] > FILL_RANGE[1]),
    "pieces": park_shape["pieces"] > MAX_PIECES,
    "holes": park_shape["holes"] > MAX_HOLES,
    "solidity": park_shape["solidity"] > SOLIDITY_MAX,
}
park_shape["checks"] = [", ".join(name for name, hit in CHECKS.items() if hit.iloc[i])
                        for i in range(len(park_shape))]
mask_suspects = park_shape[park_shape["checks"] != ""].copy()
mask_suspects["n_checks"] = mask_suspects["checks"].str.count(",") + 1
mask_suspects = mask_suspects.sort_values(["n_checks", "box_iou", "id"], ascending=[False, True, True])

print(f"{len(_judge)} masks judged · {len(mask_suspects)} suspects "
      f"({len(mask_suspects) / max(len(_judge), 1):.1%}). Suspects per check:")
for name, hit in CHECKS.items():
    print(f"  {name:9s} {int(hit.sum())}")
_cols = ["id", "image_id", "class", "checks", "box_iou", "fill", "pieces", "holes", "solidity"]
display(mask_suspects[_cols].head(12).round(2).set_index("id"))

N_SUSPECTS_SHOW = 4 if SMOKE else 8  # suspect photos to show (N-014)
_show = mask_suspects.drop_duplicates("image_id").head(N_SUSPECTS_SHOW).set_index("image_id")


def _suspect_caption(r) -> str:
    s = _show.loc[r["id"]]
    return (f"photo {r['id']} · {s['class']} · {s['checks']}\n"
            f"box IoU {s['box_iou']:.2f} · fill {s['fill']:.2f}")


show_masks(clean_images.set_index("id").loc[list(_show.index)].reset_index(), park_masks,
           f"the {len(_show)} strongest suspects: bright mask + solid box = suspect, faint = other animals",
           caption=_suspect_caption, highlight=dict(zip(_show.index, _show["id"], strict=True)))

The answer key, closed until you open it: the five planted masks, the checks that fired, and each next to
the real mask it replaced.

In [ ]:
# @title The answer key: planted masks, checks that fired, original vs planted { display-mode: "form" }
# The key: each planted mask, the checks that fired on it, and the original mask (top) vs the planted one.
_checks_by_id = park_shape.set_index("id")["checks"]
HUNT9_KEY["checks"] = HUNT9_KEY["ann_id"].map(_checks_by_id).fillna("")
HUNT9_KEY["found"] = HUNT9_KEY["checks"] != ""
hunt9_found = int(HUNT9_KEY["found"].sum())
RESULTS["hunt9"] = {"planted": len(HUNT9_KEY), "found": hunt9_found}
RESULTS["mask_suspects_real"] = int((~mask_suspects["id"].isin(HUNT9_KEY["ann_id"])).sum())
_lines = [f"{'✅' if k['found'] else '❌'} {k['kind']:15s} photo {k['image_id']} ({k['class']}, {k['how']}) "
          f"→ {k['checks'] or 'no check fired'}" for _, k in HUNT9_KEY.iterrows()]
_lines.append(f"found {hunt9_found} of {len(HUNT9_KEY)} planted · plus {RESULTS['mask_suspects_real']} "
              "real COCO masks among the suspects")

_by_image = clean_images.set_index("id")
fig, axes = plt.subplots(2, len(HUNT9_KEY), figsize=(3.2 * len(HUNT9_KEY), 6.2), squeeze=False)
for col, (_, k) in enumerate(HUNT9_KEY.iterrows()):
    photo = _by_image.loc[k["image_id"]]
    thumb = load_thumb(photo["path"], 320)
    scale = thumb.width / photo["width"]
    color = mpl.colors.to_rgb(CLASS_COLORS[k["class"]])
    for row, source in enumerate([masks, park_masks]):
        ax = axes[row, col]
        full = decode(source.loc[source["id"] == k["ann_id"], "rle"].iloc[0])
        small = cv2.resize(full, thumb.size, interpolation=cv2.INTER_NEAREST) > 0
        overlay = np.zeros((thumb.height, thumb.width, 4))
        overlay[small] = (*color, 0.6)
        ax.imshow(thumb)
        ax.imshow(overlay)
        x, y, w, h = source.loc[source["id"] == k["ann_id"], "bbox"].iloc[0]
        ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale, fill=False, lw=1.5,
                                           ls="--", edgecolor=CLASS_COLORS[k["class"]]))
        frame(ax, CLASS_COLORS[k["class"]] if row == 0 else INK)
        ax.set_title(f"{k['class']} · original" if row == 0 else f"{k['kind']}\n{k['checks'] or '—'}",
                     fontsize=9)
fig.suptitle("the five planted masks (top: the real COCO mask, bottom: what the team sent)", color=INK)
fig.tight_layout()
answer_box("The answer key: the five planted masks, the checks that fired, original vs planted", _lines, fig)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q22)</summary>
<p style="margin:6px 0 0 0;">No check catches everything; together they catch all five, plus a few dozen real
masks (about 5 % of those judged: a list a person can look through in ten minutes). The box check does most
of the work because COCO built its boxes <em>from</em> the masks, so a mask that moved, grew or belongs elsewhere
stops matching its box. The other two kinds need a shape check. Expect:</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><em>another animal</em> → box IoU (the pasted mask sits around a different animal);</li>
<li style="margin:2px 0;"><em>shifted</em> → box IoU (15 px of a 100–300 px box is a large share);</li>
<li style="margin:2px 0;"><em>swollen</em> → box IoU, often fill too (it can fill more than its box);</li>
<li style="margin:2px 0;"><em>chunk missing</em> → holes (the box stays intact, the cut-out leaves a hole);</li>
<li style="margin:2px 0;"><em>crude polygon</em> → solidity (a few points leave no dents, so it fills its rubber band).</li>
</ul>
<p style="margin:6px 0 0 0;">Pieces catches none of the five, but flags real animals broken up by a fence or a branch. On a different
seed, a kind may trip one check more or less: read the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">checks</code> column of your own run.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · COCO's own masks are not clean</div>
<p style="margin:6px 0 0 0;">Some of your real suspects are genuine annotation errors, not false alarms. A 2024 re-annotation study
found that about <strong>2.4 % of COCO's val masks are poor</strong>, duplicated mask pairs among them (<em>COCO-ReM</em>,
ECCV 2024). In the Play Dataset, the masks with the highest solidity are polygons of 5 to 14 clicked
points: real crude polygons.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a class called "rare" or "common" → <strong>do</strong> name the unit (photo, box or pixel); bird is
  rare by pixels (about 1 %) and common by boxes.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a mostly-background dataset → <strong>do</strong> report per-class IoU or Dice, not pixel accuracy:
  "all background" already scores about 85 %.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a mask that does not match its box, or has a hole, or is rubber-band-tight → <strong>do</strong>
  open the photo: it is shifted, swollen, cut or crudely clicked.</li>
</ul>
</div>""")

<callout: takeaway>

### 🔴 Optional · More about masks

Two more things a mask can tell you: where in the frame the animal pixels are, and which masks have parts
too thin to survive a resize.

#### Where the animal pixels are: one heatmap per class

Section 7 drew where the box *centres* are. A mask says more: which pixels of the frame are animal. The
next cell shrinks every photo's merged mask to the same small square and averages them per class, so a
dark spot means "in many of this class's photos, this spot is animal".

In [ ]:
# @title Mean mask per class: where the animal pixels are { display-mode: "form" }
# Mean mask per class on a common square: the share of the class's photos in which each spot is animal.
HEAT_RES = 24 if SMOKE else 48  # coarser grid with few photos (N-014)
_class_of = clean_images.set_index("id")["class"]
heat = {c: np.zeros((HEAT_RES, HEAT_RES)) for c in CLASSES}
for image_id, rle in photo_rle.items():
    small = cv2.resize(decode(rle).astype(np.float32), (HEAT_RES, HEAT_RES), interpolation=cv2.INTER_AREA)
    heat[_class_of[image_id]] += small

fig, axes = plt.subplots(1, len(CLASSES), figsize=(2.3 * len(CLASSES), 2.8))
for ax, c in zip(axes, CLASSES, strict=True):
    n_photos = max(int((_class_of == c).sum()), 1)
    cmap = mpl.colors.LinearSegmentedColormap.from_list(c, ["white", CLASS_COLORS[c]])
    ax.imshow(heat[c] / n_photos, cmap=cmap, vmin=0, vmax=1, extent=[0, 1, 1, 0])
    ax.set_title(f"{c} · {heat[c].sum() / n_photos / HEAT_RES**2:.0%} of a photo", fontsize=9,
                 color=CLASS_COLORS[c])
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=7)
    ax.grid(False)
fig.suptitle("Where the animal pixels are (same scale for every class: white = never, full colour = always)",
             color=INK)
fig.tight_layout()
display(fig)
plt.close(fig)

💡 Every class is darkest in the centre, but by very different amounts: cat and dog spread a large blob
over the middle and lower half (close-ups), while bird stays pale everywhere because its pixels are few and
scattered. A model trained on this learns that the edges of the frame are background, which the paddock
camera will contradict every day.

#### Thin parts

A leg, a tail, a beak or a far-away bird is only a few pixels wide. What happens when the photo is shrunk:

1. A COCO photo is about 640 px wide; the network takes 224×224, about a third. A horse's leg of 7 px
   becomes 2 px.
2. A mask resized with nearest neighbour (section 3) samples one pixel in three, so anything under 3 px
   (a thin tail, a far-away bird's leg) can fall between the samples and vanish.
3. The network never sees that part: the label says "no tail" for a horse that has one.

The next cell marks, in every mask, the pixels in parts narrower than `THIN_PX` = 7 px.

**Q23.** 🔴 Which class has the most masks with thin parts, and is it for the same reason as the
runner-up? Pick one, then run.

In [ ]:
# @title Q23 Share of each class's masks with thin parts, and the thinnest masks { display-mode: "form" }
THIN_SHARE = 0.05  # a mask "has thin parts" when at least this share of its pixels is thin
_big = animals["mask_area"] >= MIN_JUDGED_AREA
_has_thin = animals["thin"] >= THIN_SHARE
thin_by_class = pd.DataFrame({
    "all masks": _has_thin.groupby(animals["class"]).mean(),
    f"masks ≥ {int(MIN_JUDGED_AREA**0.5)}² px": _has_thin[_big].groupby(animals.loc[_big, "class"]).mean(),
}).reindex(CLASSES).fillna(0)
RESULTS["thin_mask_share"] = {c: float(thin_by_class.loc[c, "all masks"]) for c in CLASSES}
display((thin_by_class * 100).round(0).astype(int).astype(str).add(" %").T)

fig, ax = plt.subplots(figsize=(8, 3.2))
_x = np.arange(len(CLASSES))
for k, (col, alpha) in enumerate([(thin_by_class.columns[0], 1.0), (thin_by_class.columns[1], 0.45)]):
    ax.bar(_x + (k - 0.5) * 0.38, thin_by_class[col], width=0.38, alpha=alpha,
           color=[CLASS_COLORS[c] for c in CLASSES], edgecolor="white", label=col)
ax.set_xticks(_x, CLASSES)
for lab in ax.get_xticklabels():
    lab.set_color(CLASS_COLORS[lab.get_text()])
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title(f"Masks with ≥ {THIN_SHARE:.0%} of their pixels thinner than {THIN_PX} px: all (solid), "
             "large enough to judge (pale)", fontsize=10)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
display(fig)
plt.close(fig)

N_THIN_SHOW = 2 if SMOKE else 4  # photos with the most thin pixels to show (N-014)
animals["thin_px"] = animals["thin"] * animals["mask_area"]
_most_thin = animals.sort_values(["thin_px", "id"], ascending=[False, True]).drop_duplicates("image_id").head(
    N_THIN_SHOW)
show_masks(clean_images.set_index("id").loc[list(_most_thin["image_id"])].reset_index(), masks,
           f"the photos with the most thin pixels (dark = narrower than {THIN_PX} px)", mark_thin=True,
           cols=N_THIN_SHOW)

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q23)</summary>
<p style="margin:6px 0 0 0;">Bird, by far, on all masks (about two thirds on the full set), but not because
birds have long legs: most birds are so small that the <em>whole bird</em> is a thin part. Among the masks large
enough to judge, horse leads (legs), then bird, sheep and zebra, and cat almost never has one: tiny objects
and thin parts of big ones are two problems behind one number. Thin parts are so few pixels that a model
can get all of them wrong and still score well.</p>
<p style="margin:6px 0 0 0;"><strong>When you see</strong> thin parts or tiny animals → <strong>do</strong> resize masks with nearest neighbour, keep a higher
input resolution or tile, and check that the thin parts survive.</p>
</details>
</div>""")

<callout: answer>

## 9. Data-centric AI in practice: the Challenge

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">See the problems planted in a new batch of park photos found with the tools of sections 3 to 8 alone, and
(optional) run the hunt yourself.</p>
</div>""")

<callout: objective>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-theory" style="background:#E3F2FD;color:#0D47A1;border-left:6px solid #0D47A1;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">📘 Theory · Data-centric AI</div>
<p style="margin:6px 0 0 0;">Two ways to make a model better. <strong>Model-centric:</strong> keep the data, change the model. <strong>Data-centric:</strong> keep
the model, change the data. The park's alert misses a dog in the sheep paddock because three training photos
of dogs are labelled "sheep": fix the labels, remove the copies, make the boxes and masks agree.</p>
</div>""")

<callout: theory>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Why data-centric</div>
<p style="margin:6px 0 0 0;">The data is where the cheap gains are. Andrew Ng gave the second approach its name in 2021. Northcutt,
Athalye and Mueller (2021) found label errors in the <em>test sets</em> of ten widely used benchmarks: on
average at least 3.3 % of the labels, and at least 6 % of ImageNet's validation set. That is enough
to change which model looks best. A model-centric fix would be a bigger network, another loss or
more epochs.</p>
</div>""")

<callout: go-deeper>

### A new batch from the park

About 150 photos arrive from the park's cameras, to be checked before next week's training. Five steps made
the batch, and each can go wrong: ten kinds of problem are planted, **at runtime from `SEED`**:

1. **The camera operator typed one label per photo.**
   - **wrong label**: a typo, the wrong class.
2. **The intern split the photos into train and test.**
   - **train/test overlap**: a copy whose two files sit in *different* splits (whatever kind of copy it is),
     so the test score is too good.
3. **The keeper counted the animals per photo.**
   - **missing box**: an animal the keeper counted, but the annotation team did not annotate.
4. **The annotation team drew a box and a mask per animal.**
   - **ugly mask**: a mask that does not match its animal (section 8).
5. **Cameras and uploads saved the files.**
   - **exact duplicate**: the same file twice.
   - **flipped copy**: a photo mirrored and saved again.
   - **rotated copy**: a photo turned by 90° and saved again.
   - **recompressed copy**: a photo resized and saved at a lower JPEG quality.
   - **unreadable**: a file that does not open.
   - **grayscale**: a photo saved without colour.

In [ ]:
# @title Plant the new batch (seeded, at runtime; nothing is redistributed) { display-mode: "form" }
CHALLENGE_DIR = DATA_DIR / "cache" / "challenge"
N_BATCH_ORIGINALS = 24 if SMOKE else 140  # distinct photos in the batch (N-014: fewer in smoke mode)
TEST_SHARE = 0.2
RECOMPRESS_SCALE = 0.7
CHALLENGE_TYPES = ["exact duplicate", "flipped copy", "rotated copy", "recompressed copy",
                   "train/test overlap", "unreadable", "grayscale", "wrong label", "missing box", "ugly mask"]
COPY_KIND = {"exact duplicate": "exact", "flipped copy": "flipped", "rotated copy": "rotated",
             "recompressed copy": "recompressed", "train/test overlap": "recompressed"}
N_COPIES = {"exact duplicate": 1 if SMOKE else 2, "flipped copy": 1, "rotated copy": 1,
            "recompressed copy": 1, "train/test overlap": 1}
N_WRONG_LABELS = 2 if SMOKE else 3
UGLY_MASK_KINDS = ["shifted", "swollen"]  # two of section 8's five kinds


def challenge_pool() -> pd.DataFrame:
    """Rows of clean_images that pass every check of sections 3–8 at the default settings: no flag, colourful,
    no crowd region, headcount = boxes, and every judged mask agrees with its box and has a plain shape."""
    judged = animals[animals["mask_area"] >= MIN_JUDGED_AREA]
    bad_mask = judged[(judged["box_iou"] < 0.93) | (judged["fill"] < 0.15) | (judged["fill"] > 0.95)
                      | (judged["pieces"] > 3) | (judged["holes"] > 0) | (judged["solidity"] > 0.97)]
    crowd = masks.loc[masks["iscrowd"] == 1, "image_id"]
    gray = image_stats.loc[image_stats["looks_gray"], "id"]
    n_boxes = boxes.groupby("image_id").size()
    excluded = set(bad_mask["image_id"]) | set(crowd) | set(gray)
    ok = (clean_images["flags"] == "") & ~clean_images["id"].isin(excluded)
    ok &= clean_images["n_animals"] == clean_images["id"].map(n_boxes)
    return clean_images[ok]


def _jpeg(image: Image.Image, quality: int) -> bytes:
    buf = BytesIO()
    image.save(buf, "JPEG", quality=quality)
    return buf.getvalue()


def _orientation_gap(path, transpose) -> int:
    """Bits between the pHash of a photo and of its flipped or rotated version (small if symmetric)."""
    thumb = load_thumb(path, THUMB_FOR_HASH)
    return int(phash(thumb) - phash(thumb.transpose(transpose)))


def _move_annotation(rle: dict, bbox, kind: str, width: int, height: int,
                     new_size: tuple) -> tuple[dict, list]:
    """An annotation (RLE mask + box) carried over to an exact, flipped, rotated or resized copy."""
    x, y, w, h = bbox
    if kind == "exact":
        return rle, [x, y, w, h]
    mask = decode(rle)
    if kind == "flipped":
        return encode(mask[:, ::-1]), [width - x - w, y, w, h]
    if kind == "rotated":  # 90° counter-clockwise, like PIL's ROTATE_90
        return encode(np.rot90(mask)), [y, width - x - w, h, w]
    sx, sy = new_size[0] / width, new_size[1] / height  # resized: x and y scale separately (section 3)
    resized = cv2.resize(mask, new_size, interpolation=cv2.INTER_NEAREST)
    return encode(resized), [x * sx, y * sy, w * sx, h * sy]


def plant_challenge(dest: Path, n_originals: int, seed: int):
    """Write the batch: n_originals photos of challenge_pool() plus planted copies and problems.

    Returns (files, batch, batch_animals, key). files: {name: path}. batch: one row per file (file, class,
    split, n_animals, width, height, path). batch_animals: one row per annotated animal (id, file, class,
    bbox, rle, mask_area). key: one dict per planted problem (type, files, class, detail); for a copy,
    `files` holds the copy and its source. Every problem gets its own photo. Deterministic for a given seed.
    """
    rng = random.Random(seed)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    pool = challenge_pool()
    chosen = pool.sample(n=min(n_originals, len(pool)), random_state=seed).set_index("id").sort_index()
    ids = list(chosen.index)
    shuffled = ids[:]
    rng.shuffle(shuffled)
    n_test = max(2, round(TEST_SHARE * len(ids)))
    split = {i: "test" if k < n_test else "train" for k, i in enumerate(shuffled)}
    label = dict(chosen["class"])
    anns = animals[animals["image_id"].isin(ids)].set_index("id")
    order = ids[:]
    rng.shuffle(order)
    used: set = set()

    def take(ok=lambda i: True) -> int:
        """The next unused photo in the seeded order that satisfies ok (the next unused one if none does)."""
        free = [i for i in order if i not in used]
        i = next((i for i in free if ok(i)), free[0])
        used.add(i)
        return i

    def resize_safe(i) -> bool:  # masks that stay judged after resizing keep agreeing with their box
        a = anns[anns["image_id"] == i]
        return bool((a.loc[a["mask_area"] * RECOMPRESS_SCALE**2 >= MIN_JUDGED_AREA, "box_iou"] >= 0.95).all())

    in_place = []  # (type, photo id, detail)
    # A missing box: one animal of a photo with several loses its annotation.
    i = take(lambda i: chosen.at[i, "n_animals"] >= 2)
    dropped = rng.choice(sorted(anns.index[anns["image_id"] == i]))
    in_place.append(("missing box", i, f"the keeper counted {chosen.at[i, 'n_animals']}, "
                                       f"the team annotated {chosen.at[i, 'n_animals'] - 1}"))
    # Ugly masks: section 8's planting on the photos still free, two of its five kinds kept.
    free_anns = anns[~anns["image_id"].isin(used)].reset_index()
    spoiled, spoil_key = plant_hunt9(free_anns, chosen.reset_index(), seed)
    spoiled_rle = spoiled.set_index("id")["rle"]
    new_rle = {}
    for _, k in spoil_key[spoil_key["kind"].isin(UGLY_MASK_KINDS)].iterrows():
        new_rle[k["ann_id"]] = spoiled_rle[k["ann_id"]]
        used.add(k["image_id"])
        in_place.append(("ugly mask", k["image_id"], f"{k['kind']} mask of a {k['class']} ({k['how']})"))
    for _ in range(N_WRONG_LABELS):
        i = take()
        label[i] = rng.choice([c for c in CLASSES if c != chosen.at[i, "class"]])
        in_place.append(("wrong label", i, f"labelled {label[i]}, the photo shows a {chosen.at[i, 'class']}"))
    gray_id, unreadable_id = take(), take()
    in_place += [("grayscale", gray_id, "saved without colour (mode L)"),
                 ("unreadable", unreadable_id, "the second half of the file is missing")]
    # Flipped and rotated copies come from photos that do not look the same after the turn (no symmetry).
    flip, turn = Image.Transpose.FLIP_LEFT_RIGHT, Image.Transpose.ROTATE_90
    condition = {"flipped copy": lambda i: _orientation_gap(chosen.at[i, "path"], flip) >= 20,
                 "rotated copy": lambda i: _orientation_gap(chosen.at[i, "path"], turn) >= 20,
                 "recompressed copy": resize_safe,
                 "train/test overlap": lambda i: split[i] == "train" and resize_safe(i)}
    copies = [(t, take(condition.get(t, lambda i: True))) for t, n in N_COPIES.items() for _ in range(n)]

    # Every file of the batch: the originals (two of them spoiled in place), then the copies.
    entries = []
    for i in ids:
        path, size = Path(chosen.at[i, "path"]), (int(chosen.at[i, "width"]), int(chosen.at[i, "height"]))
        data = path.read_bytes()
        if i == gray_id:
            with Image.open(path) as im:
                data = _jpeg(im.convert("L"), 92)
        elif i == unreadable_id:
            data = data[: len(data) // 2]
        entries.append({"data": data, "source": i, "kind": None, "split": split[i], "size": size})
    for t, i in copies:
        path, (w, h) = Path(chosen.at[i, "path"]), (int(chosen.at[i, "width"]), int(chosen.at[i, "height"]))
        kind = COPY_KIND[t]
        with Image.open(path) as im:
            rgb = im.convert("RGB")
        if kind == "exact":
            data, size = path.read_bytes(), (w, h)
        elif kind == "flipped":
            data, size = _jpeg(ImageOps.mirror(rgb), 92), (w, h)
        elif kind == "rotated":
            data, size = _jpeg(rgb.transpose(Image.Transpose.ROTATE_90), 92), (h, w)
        else:
            size = (max(8, round(w * RECOMPRESS_SCALE)), max(8, round(h * RECOMPRESS_SCALE)))
            data = _jpeg(rgb.resize(size), 55)
        other = "test" if split[i] == "train" else "train"
        entries.append({"data": data, "source": i, "kind": kind, "type": t, "size": size,
                        "split": other if t == "train/test overlap" else split[i]})
    slots = list(range(len(entries)))
    rng.shuffle(slots)
    rows, ann_rows = [], []
    for slot, e in zip(slots, entries, strict=True):
        e["name"] = f"park_{slot:03d}.jpg"
        (dest / e["name"]).write_bytes(e["data"])
        i = e["source"]
        rows.append({"file": e["name"], "class": label[i], "split": e["split"],
                     "n_animals": int(chosen.at[i, "n_animals"]), "width": e["size"][0],
                     "height": e["size"][1], "path": str(dest / e["name"])})
        for ann_id, a in anns[anns["image_id"] == i].iterrows():
            if ann_id == dropped:
                continue
            rle, bbox = _move_annotation(new_rle.get(ann_id, a["rle"]), a["bbox"], e["kind"] or "exact",
                                         int(chosen.at[i, "width"]), int(chosen.at[i, "height"]), e["size"])
            ann_rows.append({"file": e["name"], "class": a["class"],
                             "bbox": [round(float(v), 2) for v in bbox],
                             "rle": rle, "mask_area": int(mask_utils.area(rle))})

    name_of = {(e["source"], e["kind"]): e["name"] for e in entries}
    key = [{"type": t, "files": [name_of[(i, None)]], "class": chosen.at[i, "class"], "detail": d}
           for t, i, d in in_place]
    for t, i in copies:
        copy, source = name_of[(i, COPY_KIND[t])], name_of[(i, None)]
        where = next(e["split"] for e in entries if e["name"] == copy)
        key.append({"type": t, "files": [copy, source], "class": chosen.at[i, "class"],
                    "detail": f"{COPY_KIND[t]} copy of {source} ({split[i]}), the copy is in {where}"})
    key.sort(key=lambda k: (CHALLENGE_TYPES.index(k["type"]), k["files"][0]))
    table = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
    annotated = pd.DataFrame(ann_rows).sort_values("file").reset_index(drop=True)
    annotated.insert(0, "id", range(1, len(annotated) + 1))
    files = {r["file"]: Path(r["path"]) for _, r in table.iterrows()}
    return files, table, annotated, key


def show_batch(file_names, title: str, caption=None, cols: int = 5, annotations: bool = False,
               show: bool = True):
    """Gallery of batch files, each framed in the colour of its label (the `class` column of `batch`).

    annotations=True paints the annotation team's masks in their class colour and draws their boxes dashed.
    A file that does not open shows as an empty frame. `caption(name) -> str` writes the title of each photo.
    show=False returns the figure instead of drawing it (for answer_box).
    """
    names = list(file_names)
    if not names:
        print(f"{title}: no files to show.")
        return
    by_file = batch.set_index("file")
    cols = min(cols, len(names))
    n_rows = -(-len(names) // cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(2.9 * cols, 3.0 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, name in zip(axes.ravel(), names, strict=False):
        row = by_file.loc[name]
        ax.axis("on")
        if is_readable(row["path"]):
            thumb = load_thumb(row["path"], 256)
            ax.imshow(thumb)
            if annotations:
                scale = thumb.width / row["width"]
                overlay = np.zeros((thumb.height, thumb.width, 4))
                own = batch_animals[batch_animals["file"] == name]
                for _, a in own.iterrows():
                    small = cv2.resize(decode(a["rle"]), thumb.size, interpolation=cv2.INTER_NEAREST) > 0
                    overlay[small] = (*mpl.colors.to_rgb(CLASS_COLORS[a["class"]]), 0.55)
                ax.imshow(overlay)
                for _, a in own.iterrows():
                    x, y, w, h = a["bbox"]
                    ax.add_patch(mpl.patches.Rectangle((x * scale, y * scale), w * scale, h * scale,
                                                       fill=False, lw=1.5, ls="--",
                                                       edgecolor=CLASS_COLORS[a["class"]]))
        else:
            ax.text(0.5, 0.5, "cannot be opened", ha="center", va="center", transform=ax.transAxes, color=INK)
        frame(ax, CLASS_COLORS[row["class"]])
        ax.set_title(caption(name) if caption else f"{name}\n{row['class']} · {row['split']}", fontsize=8)
    fig.suptitle(title, color=INK)
    fig.tight_layout()
    if not show:
        return fig
    display(fig)
    plt.close(fig)
    return None

The next cell writes the batch (a few seconds) and loads three objects: `batch_files` (name → path), `batch`
(one row per file: label as typed, split, the keeper's headcount `n_animals`, size) and `batch_animals`
(one row per annotated animal: class, box, mask as RLE).

In [ ]:
# @title Write the batch, then show its tables and a first look at a dozen photos { display-mode: "form" }
batch_files, batch, batch_animals, _CHALLENGE_KEY = plant_challenge(CHALLENGE_DIR, N_BATCH_ORIGINALS, SEED)
RESULTS["challenge_types"] = len({k["type"] for k in _CHALLENGE_KEY})
print(f"{len(batch_files)} files in {CHALLENGE_DIR}/ · {(batch['split'] == 'test').sum()} in test · "
      f"{len(batch_animals)} annotated animals")
print(f"planted: {len(_CHALLENGE_KEY)} problems of {RESULTS['challenge_types']} kinds")
display(batch.drop(columns="path").head())
show_batch(batch["file"].head(12), "the first dozen files of the batch (frame = the label as typed)", cols=6)

Nothing in the tables says "problem": that is what a real delivery looks like.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Your tools, and where they came from</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">exact_duplicate_groups(paths)</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">find_copies(files, threshold, use_variants=True)</code> (section 5): pairs
  of files that are copies, and what kind of copy;</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">is_readable(path)</code> (helper defined in section 5's toolbox): does the file open at all? (<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">False</code> for a
  file PIL cannot decode);</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">measure_image(path, thumb)[0]</code> (section 3): <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">mode</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">colourfulness</code> of a photo (0 for grayscale);</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">batch["n_animals"]</code> against the number of rows per file in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">batch_animals</code> (section 7);</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">mask_shape(decode(rle), bbox)</code> (section 8): <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">box_iou</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">fill</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">pieces</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">holes</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">solidity</code>;</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">show_batch(names, title, annotations=True)</code>: look at any files, with their boxes and masks.</li>
</ul>
<p style="margin:6px 0 0 0;">Every problem was planted in a photo that was clean before, so each one the checks find is a planted one,
not a real COCO error.</p>
</div>""")

<callout: go-deeper>

Two collapsed cells follow: the scoring helpers, and **the demo**: the ten checks run on the batch, one line
per kind with the files each found, in a closed toggle the presenter opens.

In [ ]:
# @title Score helpers: check_findings, reveal_answers { display-mode: "form" }
def check_findings(findings, verbose: bool = True) -> dict:
    """Score a list of (file, kind) pairs against the planted problems: found k of n per kind.

    A planted problem is found when one pair names its kind and one of its files (for a copy, the copy or its
    source). Returns {"found", "planted", "per_type": {kind: [found, planted]}, "unmatched"}.
    """
    if not isinstance(findings, (list, tuple)):
        raise SystemExit("findings must be a list of (file, kind) pairs, "
                         "e.g. [('park_999.jpg', 'grayscale')]")
    pairs, skipped = set(), []
    for item in findings:
        if (isinstance(item, (list, tuple)) and len(item) == 2 and all(isinstance(v, str) for v in item)
                and item[1].strip().lower() in CHALLENGE_TYPES):
            pairs.add((Path(item[0].strip()).name, item[1].strip().lower()))
        else:
            skipped.append(item)
    hit = [any(kind == k["type"] and name in k["files"] for name, kind in pairs) for k in _CHALLENGE_KEY]
    matched = {(name, kind) for name, kind in pairs
               if any(kind == k["type"] and name in k["files"] for k in _CHALLENGE_KEY)}
    per_type = {t: [sum(h for h, k in zip(hit, _CHALLENGE_KEY, strict=True) if k["type"] == t),
                    sum(k["type"] == t for k in _CHALLENGE_KEY)] for t in CHALLENGE_TYPES}
    score = {"found": int(sum(hit)), "planted": len(_CHALLENGE_KEY), "per_type": per_type,
             "unmatched": len(pairs) - len(matched)}
    if verbose:
        for t, (k, n) in per_type.items():
            print(f"  {'✅' if k == n else '⬜'} {t:19s} found {k} of {n}")
        print(f"found {score['found']} of {score['planted']} planted problems · {score['unmatched']} of your "
              "findings match no planted problem")
        if skipped:
            print(f"⚠️ skipped {len(skipped)} entries that are not (file, kind) pairs with a kind from "
                  f"CHALLENGE_TYPES, e.g. {skipped[0]!r}")
    return score


def reveal_answers() -> None:
    """The answer key in a closed toggle: one line per planted problem, then the files with each problem."""
    lines = [f"{k['type']:19s} {' & '.join(k['files'])}: {k['detail']}" for k in _CHALLENGE_KEY]
    type_of = {k["files"][0]: k["type"] for k in _CHALLENGE_KEY}
    in_annotations = {"wrong label", "missing box", "ugly mask"}
    file_problems = [k["files"][0] for k in _CHALLENGE_KEY if k["type"] not in in_annotations]
    fig_files = show_batch(file_problems, "problems in the files (copies, unreadable, grayscale)",
                           caption=lambda n: f"{type_of[n]}\n{n}", cols=6, show=False)
    annotation_problems = [k["files"][0] for k in _CHALLENGE_KEY if k["type"] in in_annotations]
    label_of = batch.set_index("file")["class"]
    fig_labels = show_batch(annotation_problems,
                            "problems in the labels and annotations (frame = the label as typed)",
                            caption=lambda n: f"{type_of[n]}\n{n} · labelled {label_of[n]}",
                            cols=6, annotations=True, show=False)
    answer_box("The Challenge's answer key: every planted problem and its files", lines,
               [fig_files, fig_labels])

In [ ]:
# @title Demo: the ten checks run on the batch (open the code only if you are stuck) { display-mode: "form" }
_REF_HASH_THRESHOLD = 8  # section 5's default
_REF_IOU_MIN = 0.9  # section 8's default
_TYPE_OF_COPY = {"exact": "exact duplicate", "flipped": "flipped copy", "rotated": "rotated copy",
                 "recompressed": "recompressed copy"}


def _reference_findings() -> list[tuple[str, str]]:
    """One check per kind, with the tools of sections 3–8; returns (file, kind) pairs."""
    found = []
    readable = {n: p for n, p in batch_files.items() if is_readable(p)}
    found += [(n, "unreadable") for n in batch_files if n not in readable]
    split_of = batch.set_index("file")["split"]
    for _, r in find_copies(readable, _REF_HASH_THRESHOLD, use_variants=True).iterrows():
        across = split_of[r["a"]] != split_of[r["b"]]
        found.append((r["b"], "train/test overlap" if across else _TYPE_OF_COPY[r["kind"]]))
    for n, p in readable.items():
        stats = measure_image(p, 64)[0]
        if stats["mode"] != "RGB" or stats["colourfulness"] < 2.0:
            found.append((n, "grayscale"))
    per_file = batch_animals.groupby("file")
    annotated_class = per_file["class"].agg(lambda s: s.mode().iloc[0])
    n_annotated = per_file.size()
    for _, r in batch.iterrows():
        if r["file"] in annotated_class.index and annotated_class[r["file"]] != r["class"]:
            found.append((r["file"], "wrong label"))
        if int(n_annotated.get(r["file"], 0)) != r["n_animals"]:
            found.append((r["file"], "missing box"))
    for _, a in batch_animals[batch_animals["mask_area"] >= MIN_JUDGED_AREA].iterrows():
        if mask_shape(decode(a["rle"]), a["bbox"])["box_iou"] < _REF_IOU_MIN:
            found.append((a["file"], "ugly mask"))
    return sorted(set(found))


_reference = _reference_findings()
_reference_score = check_findings(_reference, verbose=False)
RESULTS["challenge_reference"] = {"found": _reference_score["found"], "planted": _reference_score["planted"],
                                  "unmatched": _reference_score["unmatched"]}
_TOOL_OF = {"unreadable": "is_readable", "exact duplicate": "find_copies (MD5)",
            "flipped copy": "find_copies (variant hashes)", "rotated copy": "find_copies (variant hashes)",
            "recompressed copy": "find_copies (pHash)", "train/test overlap": "a copy pair + batch['split']",
            "grayscale": "measure_image: mode / colourfulness", "wrong label": "label vs the animals' class",
            "missing box": "headcount vs boxes", "ugly mask": "mask_shape: box IoU"}
_demo = [f"{kind:19s} ← {_TOOL_OF[kind]:36s} {', '.join(f for f, k in _reference if k == kind) or '—'}"
         for kind in CHALLENGE_TYPES]
answer_box(f"Demo: what the ten checks found ({_reference_score['found']} of {_reference_score['planted']} "
           f"planted, {_reference_score['unmatched']} false alarms)",
           ["kind                ← check                                files"] + _demo)

### 🔴 Optional · Run the hunt yourself

In class, the reference solution above is the demo: it found every planted problem, and the answer key
below shows which check found which. On your own (or in a fast group), do the hunt yourself.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-questions" style="background:#E0F7F4;color:#00564D;border-left:6px solid #00564D;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">❓ Questions</div>
<p style="margin:6px 0 0 0;">The ten headers in the next cell are your plan: one per kind, with its tool and what it reads.</p>
</div>""")

<callout: questions>

**Q24.** 🔴 The hunt. Six checks are written for you: kinds 1 to 5 and 7. Write the other four under their
headers: the kinds where two sources disagree. Before you do, predict: which kinds could you never spot in
the photo itself, however long you looked? Then collect what you find in `findings` and run the score.
Rerun both as often as you like.

In [ ]:
# Q24  Your hunt: one header per kind = the plan. Kinds 1-5 and 7 are written; write 6, 8, 9 and 10.

# 1. unreadable — tool: is_readable — reads: the files (written for you)
cant_open = sorted(name for name, path in batch_files.items() if not is_readable(path))
print("unreadable:", cant_open)

# 2. exact duplicate — tool: find_copies, kind "exact" — reads: the files (written for you)
opens = {n: p for n, p in batch_files.items() if n not in cant_open}
pairs = find_copies(opens, HASH_THRESHOLD, use_variants=True)  # its "kind" column names the sort of copy


def of_kind(kind):
    return pairs.loc[pairs["kind"] == kind, ["a", "b"]].values.tolist()


print("exact duplicate:", of_kind("exact"))

# 3. flipped copy — tool: find_copies, kind "flipped" — reads: the files (written for you)
print("flipped copy:", of_kind("flipped"))

# 4. rotated copy — tool: find_copies, kind "rotated" — reads: the files (written for you)
print("rotated copy:", of_kind("rotated"))

# 5. recompressed copy — tool: find_copies, kind "recompressed" — reads: the files (written for you)
print("recompressed copy:", of_kind("recompressed"))

# 6. train/test overlap — tool: the same pairs, then batch["split"] of both files — reads: files + batch


# 7. grayscale — tool: measure_image(path, 64)[0]["mode"] and ["colourfulness"] — reads: the files (written)
looks_gray = [n for n, p in opens.items()
              if (s := measure_image(p, 64)[0])["mode"] != "RGB" or s["colourfulness"] < 2.0]
print("grayscale:", looks_gray)

# 8. wrong label — tool: batch["class"] vs the class of the file's animals — reads: batch + batch_animals


# 9. missing box — tool: batch["n_animals"] vs batch_animals.groupby("file").size() — reads: both tables


# 10. ugly mask — tool: mask_shape(decode(a["rle"]), a["bbox"])["box_iou"] per row a — reads: batch_animals

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · Your findings</div>
<p style="margin:6px 0 0 0;">Write one <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">(file, kind)</code> pair per problem into <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">findings</code>, with the kind spelled as in <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">CHALLENGE_TYPES</code>.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">For a copy, name either of its two files.</li>
<li style="margin:2px 0;">Build the list from your variables, e.g. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">[(f, "unreadable") for f in cant_open] + [...]</code>.</li>
<li style="margin:2px 0;"><code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">check_findings</code> prints "found k of n" per kind: n problems of that kind were planted, and k of them are
  named by one of your pairs. It also counts the pairs that match nothing; it never says which ones.</li>
</ul>
</div>""")

<callout: your-turn>

In [ ]:
# ✏️ YOUR TURN
# One (file, kind) pair per problem; kind is one of CHALLENGE_TYPES. For a copy, either of its files counts.
findings = [
    # ("park_999.jpg", "grayscale"),  (the format; there is no park_999.jpg)
]

challenge_score = check_findings(findings)

💡 Read the ⬜ lines as a to-do list: each is a kind you have not caught yet, and its header in the Q24
cell names a tool. The reference solution above found every problem with this notebook's tools alone.

### The answer key

Open the worked answer, then run the next cell to see the files of your batch.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q24): which check finds which kind</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Unreadable</strong>: <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">is_readable(path)</code> on every file. Run it first, because every other check that opens the
  photo stumbles on this file.</li>
<li style="margin:2px 0;"><strong>Exact, flipped, rotated and recompressed copies</strong>: <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">find_copies(..., use_variants=True)</code> on the files
  that open. MD5 finds the exact ones; the perceptual hash of each photo's six variants finds the
  recompressed, flipped and rotated ones. Plain hashing would miss two of the four.</li>
<li style="margin:2px 0;"><strong>Train/test overlap</strong>: the same pairs, then <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">batch["split"]</code> of both files. A pair across the splits is
  leakage whatever kind of copy it is (section 6). In the planted batch it is a recompressed copy, so it
  hides among the near-duplicates.</li>
<li style="margin:2px 0;"><strong>Grayscale</strong>: <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">measure_image</code> gives <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">mode</code> "L" and a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">colourfulness</code> of 0 (section 3).</li>
<li style="margin:2px 0;"><strong>Wrong label</strong>: no tool looks at the pixels and says "this is a dog". But the annotation team's animals
  carry their own class, so a file whose label disagrees with the class of its animals is a wrong label
  (or a wrong annotation; a human decides).</li>
<li style="margin:2px 0;"><strong>Missing box</strong>: the keeper's headcount <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">n_animals</code> vs the number of animals annotated per file
  (section 7).</li>
<li style="margin:2px 0;"><strong>Ugly mask</strong>: <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">mask_shape(...)["box_iou"]</code> below 0.9 (section 8). The two planted kinds, a shifted and a
  swollen mask, both stop matching their box.</li>
</ul>
<p style="margin:6px 0 0 0;">The four kinds that cannot be seen in the pixels are train/test overlap, wrong label, missing box and ugly
mask. Each is a disagreement between two sources: two splits, a label and its annotation, a headcount and
the boxes, a mask and its box. That is the data-centric habit: get a second source and compare.</p>
<p style="margin:6px 0 0 0;"><strong>A finding that matches nothing</strong> is the wrong kind for the right file (a copy across the splits is a
<em>train/test overlap</em>, not a recompressed copy), or a false alarm.</p>
<p style="margin:6px 0 0 0;"><strong>Looking or comparing?</strong> Looking works for the copies and the grayscale photo, if you happen to see both
files. The wrong labels, the missing box and the ugly masks need two sources side by side: nobody spots
them by scrolling through 150 photos.</p>
</details>
</div>""")

<callout: answer>

In [ ]:
# @title Show the answer key: every planted problem, then its files { display-mode: "form" }
reveal_answers()

💡 For the room: which of these could you have seen by scrolling through the photos, and which only by
comparing two sources?

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you see</strong> a new batch → <strong>do</strong> run the same short list every time: does it open, is it a copy
  (also across the splits), does its mode say colour, does the label agree with the annotation, does the
  headcount agree with the boxes, does the mask agree with its box.</li>
<li style="margin:2px 0;"><strong>When you see</strong> a problem that no pixel shows (wrong label, missing box, ugly mask, overlap) → <strong>do</strong> get
  a second source and compare the two.</li>
<li style="margin:2px 0;"><strong>When you remove files</strong> → <strong>do</strong> write down "removed N, kept M" and why.</li>
<li style="margin:2px 0;"><strong>When you finish here</strong> → <strong>do</strong> run the same list on your own dataset in section 10.</li>
</ul>
</div>""")

<callout: takeaway>

## 10. Your project dataset

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Run the checklist of sections 3 to 6 on your own folder-per-class dataset, and answer the guiding questions
for it the way we answered them for the park.</p>
</div>""")

<callout: objective>

Your project has its own dataset, and it deserves the same twenty minutes of looking before anybody trains
on it. This section runs the checklist on **any folder-per-class dataset**: one sub-folder per class, the
images inside. The sub-folder names become the class names.

```
my_dataset/                ← MY_DATASET_DIR points here (or at my_dataset.zip)
├── daisy/      img_001.jpg, img_002.png, …
├── rose/       …
└── tulip/      …
```

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-your-turn" style="background:#FFF4D6;color:#7A4A00;border-left:6px solid #7A4A00;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✏️ Your turn · Point the notebook at your dataset (you change one line)</div>
<p style="margin:6px 0 0 0;">Colab cannot see your laptop's folders, so the dataset first goes where Colab can read it. Pick <strong>one</strong> of
the two options, then do the last step.</p>
<p style="margin:6px 0 0 0;"><strong>Option A · Upload it to this Colab session</strong> (fastest; gone when the session ends)</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Zip the dataset folder</strong> on your laptop: Mac, right-click the folder → <em>Compress</em>; Windows,
   right-click → <em>Send to</em> → <em>Compressed (zipped) folder</em>. Colab uploads files, not folders.</li>
<li style="margin:2px 0;">Click the 📁 <strong>Files</strong> icon in Colab's left bar and <strong>drag the zip into it</strong> (or use the upload icon at
   the top of the pane). Wait until the progress circle at the bottom left disappears.</li>
<li style="margin:2px 0;"><strong>Right-click the zip → Copy path.</strong> It looks like <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/my_dataset.zip</code>; the notebook unzips it.</li>
</ol>
<p style="margin:6px 0 0 0;">⚠️ Colab deletes uploaded files when the session ends (closing the tab for a while, or <em>Disconnect and
delete runtime</em>): next time, upload the zip again.</p>
<p style="margin:6px 0 0 0;"><strong>Option B · Use Google Drive</strong> (stays between sessions; slower the first time)</p>
<ol style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Put the dataset on your Drive</strong> at <a href="https://drive.google.com" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">drive.google.com</a>: <em>New → Folder upload</em>
   for the folder, or <em>File upload</em> for a zip.</li>
<li style="margin:2px 0;"><strong>Connect Drive to Colab:</strong> in the 📁 Files pane, click the <strong>Mount Drive</strong> icon (a folder with the
   Drive logo) and allow access in the window that opens. A <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">drive/</code> folder appears in the pane.</li>
<li style="margin:2px 0;">Open <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">drive/MyDrive/</code>, find the folder whose sub-folders are your classes (or the zip), <strong>right-click →
   Copy path.</strong> It looks like <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/drive/MyDrive/project/my_dataset</code>.</li>
</ol>
<p style="margin:6px 0 0 0;"><strong>Last step, both options:</strong> paste the path between the quotes in the next cell, click into that cell, and
run everything from there: <strong>Runtime → Run cell and below</strong>. If the path is not right, the cell after it
stops with a plain message that says what to change.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;">✅ <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/my_dataset.zip</code> (option A) or <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">/content/drive/MyDrive/project/my_dataset</code> (option B): the
  folder that holds <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">daisy/</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">rose/</code>, <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">tulip/</code>, or a zip of it.</li>
<li style="margin:2px 0;">❌ <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/my_dataset/daisy</code>: one class folder, one level too deep.</li>
<li style="margin:2px 0;">❌ a folder that holds <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">train/</code> and <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">val/</code>: point at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/train</code> instead (then at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/val</code>, if you like).</li>
</ul>
<p style="margin:6px 0 0 0;">Leave it at <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">None</code> to see the whole section on a small park sample first.</p>
</div>""")

<callout: your-turn>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Detection or segmentation project, and big datasets</div>
<p style="margin:6px 0 0 0;">Put all images in one folder inside <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">MY_DATASET_DIR</code> (e.g. <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">…/project/all/</code>). The class counts then say
nothing, but sizes, modes, copies and the split still do (the box and mask checks are sections 7 and 8).
Images deeper inside a class folder count, other files are listed and skipped. Datasets above
<code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">MY_MAX_IMAGES</code> photos are sampled down, keeping the class shares, so the checks stay at a minute or two.</p>
</div>""")

<callout: go-deeper>

In [ ]:
# ✏️ YOUR TURN
MY_DATASET_DIR = None  # paste your path between quotes, e.g. "/content/my_dataset.zip"

In [ ]:
# @title Dataset reader: finds your class folders, or says exactly what to change { display-mode: "form" }
import zipfile

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}
SPLIT_NAMES = {"train", "training", "val", "valid", "validation", "test", "testing", "dev"}
MY_MAX_IMAGES = 2000  # larger datasets are sampled down to this many photos, stratified by class


def _visible(folder: Path) -> list[Path]:
    """The entries of `folder` that are not hidden or system files (.DS_Store, __MACOSX, …)."""
    return sorted(f for f in folder.iterdir() if not f.name.startswith((".", "_")))


def _is_image(f: Path) -> bool:
    return f.is_file() and f.suffix.lower() in IMAGE_SUFFIXES


def _contents(folder: Path, limit: int = 8) -> str:
    """One line per entry: 'name/ (12 images)' for folders, the name for files."""
    lines = []
    for f in _visible(folder)[:limit]:
        n = sum(_is_image(g) for g in f.rglob("*")) if f.is_dir() else None
        lines.append(f"    {f.name}/  ({n} images)" if f.is_dir() else f"    {f.name}")
    more = len(_visible(folder)) - limit
    return "\n".join(lines + ([f"    … and {more} more"] if more > 0 else [])) or "    (empty)"


def find_dataset(path) -> Path:
    """The folder that holds the class folders. Stops with a plain message saying what to change if the
    path cannot be used; unzips a .zip and steps into a lone wrapper folder on its own.

    e.g. find_dataset("/content/flowers.zip") → Path("data/cache/my_upload/flowers/flowers")
    """
    p = Path(str(path).strip().strip("'\"")).expanduser()
    if str(p).startswith("/content/drive") and IN_COLAB and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive  # Step 1, Drive route: connect it on first use

        drive.mount("/content/drive")
    if p.is_file() and p.suffix.lower() == ".zip":
        dest = (DATA_DIR / "cache" / "my_upload" / p.stem).resolve()  # absolute, so messages are pastable
        if not dest.exists():
            with zipfile.ZipFile(p) as z:
                z.extractall(dest)
        print(f"ℹ️ unzipped {p.name} into {dest}")
        p = dest
    if not p.exists():
        parent = next(q for q in p.parents if q.exists())
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} does not exist (a typo, or the upload is not there).\n"
                         f"   The nearest folder that exists is {parent}, which holds:\n{_contents(parent)}\n"
                         "   Copy the path from the Files pane: right-click the folder → Copy path.")
    if p.is_file():
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} is a file. Point at the folder that holds your class "
                         f"folders, or at a .zip. It sits in {p.parent}, which holds:\n{_contents(p.parent)}")
    while True:  # a zip often wraps everything in one more folder: step into it
        subdirs = [d for d in _visible(p) if d.is_dir()]
        loose = [f for f in _visible(p) if _is_image(f)]
        if len(subdirs) == 1 and not loose and any(d.is_dir() for d in _visible(subdirs[0])):
            print(f"ℹ️ {p.name}/ holds only the folder {subdirs[0].name}/: using that.")
            p = subdirs[0]
            continue
        break
    names = {d.name.lower() for d in subdirs}
    if subdirs and names <= SPLIT_NAMES:
        first = p / sorted(d.name for d in subdirs)[0]
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds split folders ({', '.join(sorted(names))}), "
                         f"not class folders. Point at one of them, e.g. {first}, and run the section "
                         "again for the other.")
    if not subdirs and loose:
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds {len(loose)} images but no class folders. If this "
                         f"is one class folder, point one level up, at {p.parent}. If your project has no "
                         "classes (detection, segmentation), move the images into one folder inside, e.g. "
                         f"{p / 'all'}.")
    if not any(_is_image(f) for f in p.rglob("*")):
        raise SystemExit(f"❌ MY_DATASET_DIR: {p} holds no image files "
                         f"({', '.join(sorted(IMAGE_SUFFIXES))}). "
                         f"It holds:\n{_contents(p)}")
    return p


def list_folder_per_class(root) -> pd.DataFrame:
    """One row per image file under `root`: columns path, file, class (the top-level folder it sits in).

    e.g. a row: {"path": "/content/flowers/rose/img_7.jpg", "file": "rose/img_7.jpg", "class": "rose"}
    """
    rows = []
    for class_dir in (d for d in _visible(Path(root)) if d.is_dir()):
        for f in sorted(class_dir.rglob("*")):
            if _is_image(f) and not f.name.startswith("."):
                rows.append({"path": str(f), "file": str(f.relative_to(root)), "class": class_dir.name})
    return pd.DataFrame(rows, columns=["path", "file", "class"])


def folder_notes(root: Path, files: pd.DataFrame) -> list[str]:
    """Warnings about what the reader skipped or what makes the checks weak (one line each)."""
    root = Path(root)
    notes = []
    loose = [f.name for f in _visible(root) if _is_image(f)]
    if loose:
        notes.append(f"⚠️ {len(loose)} images sit directly in {root.name}/, outside any class folder: "
                     "skipped.")
    others = [f for d in _visible(root) if d.is_dir() for f in d.rglob("*")
              if f.is_file() and not _is_image(f) and not f.name.startswith(".")]
    skipped = Counter(f.suffix.lower() or "(no extension)" for f in others)
    if skipped:
        notes.append("⚠️ skipped files that are not images: "
                     + ", ".join(f"{n} × {ext}" for ext, n in skipped.most_common()))
    empty = [d.name for d in _visible(root) if d.is_dir() and d.name not in set(files["class"])]
    if empty:
        notes.append(f"⚠️ class folders with no images: {', '.join(empty)}.")
    if files["class"].nunique() == 1:
        notes.append("⚠️ only one class folder: check 1 has nothing to compare (fine for a detection "
                     "project).")
    return notes

The next cell reads your folder (or, with `None`, builds the park sample: 20 cleaned photos per class plus
**three planted problems**, an exact copy of a cat, a mirrored dog and a grayscale bird). It prints what it
found and shows three images per class: **check that these are your images and your class names** before
reading anything else.

In [ ]:
# @title Read your folder (or build the sample), and show what was found { display-mode: "form" }
MY_SAMPLE_DIR = DATA_DIR / "cache" / "my_dataset"
MY_SAMPLE_PER_CLASS = 4 if SMOKE else 20  # photos per class in the sample (N-014: fewer in smoke mode)


def build_my_sample(dest: Path, per_class: int, seed: int) -> Path:
    """Folder-per-class copy of a few clean Play Dataset photos, plus an exact copy, a mirror, a grayscale."""
    shutil.rmtree(dest, ignore_errors=True)  # rebuilt from scratch: re-running gives the same sample (N-013)
    rng = random.Random(seed)
    pool = clean_images[clean_images["flags"] == ""]
    picked = {}
    for c in CLASSES:
        of_class = sorted(pool.loc[pool["class"] == c, "path"])
        picked[c] = rng.sample(of_class, min(per_class, len(of_class)))
        (dest / c).mkdir(parents=True, exist_ok=True)
        for p in picked[c]:
            shutil.copyfile(p, dest / c / Path(p).name)
    first = {c: Path(picked[c][0]) for c in CLASSES if picked[c]}
    shutil.copyfile(first["cat"], dest / "cat" / f"{first['cat'].stem}_copy.jpg")
    with Image.open(first["dog"]) as im:
        ImageOps.mirror(im.convert("RGB")).save(dest / "dog" / f"{first['dog'].stem}_b.jpg", quality=92)
    with Image.open(first["bird"]) as im:
        im.convert("L").save(dest / "bird" / first["bird"].name, quality=92)
    return dest


if MY_DATASET_DIR is not None and not SMOKE:
    my_source, my_root = "yours", find_dataset(MY_DATASET_DIR)
else:
    my_source, my_root = "sample", build_my_sample(MY_SAMPLE_DIR, MY_SAMPLE_PER_CLASS, SEED)
    print("MY_DATASET_DIR is None: running on the park sample. Paste your path above to run on yours.")
my_files = list_folder_per_class(my_root)
for _note in folder_notes(my_root, my_files):
    print(_note)

if len(my_files) > MY_MAX_IMAGES:
    _n_all = len(my_files)
    my_files = pd.concat([g.sample(max(1, round(len(g) * MY_MAX_IMAGES / _n_all)), random_state=SEED)
                          for _, g in my_files.groupby("class")])
    print(f"⚠️ {_n_all} images found: the checks run on a stratified sample of {len(my_files)}.")
my_files = my_files.sort_values(["class", "file"]).reset_index(drop=True)
MY_CLASSES = sorted(my_files["class"].unique())
# Park classes keep their colour; any other class name borrows one from the same palette (L-003).
_palette = list(CLASS_COLORS.values())
MY_COLORS = {c: CLASS_COLORS.get(c, _palette[i % len(_palette)]) for i, c in enumerate(MY_CLASSES)}
print(f"✅ source: {my_source} · {my_root} · {len(my_files)} images in {len(MY_CLASSES)} class folders: "
      + ", ".join(f"{c} ({n})" for c, n in my_files["class"].value_counts().reindex(MY_CLASSES).items()))

_shown = MY_CLASSES[:8]  # a first look: three images per class (at most eight classes)
fig, axes = plt.subplots(len(_shown), 3, figsize=(7, 2.2 * len(_shown)), squeeze=False)
for r, c in enumerate(_shown):
    _of_class = my_files[my_files["class"] == c].head(3)
    for k in range(3):
        ax = axes[r, k]
        ax.axis("off")
        if k < len(_of_class):
            try:
                ax.imshow(load_thumb(_of_class.iloc[k]["path"], 192))
            except Exception:  # an unreadable file: check 2 lists it
                ax.text(0.5, 0.5, "cannot be opened", ha="center", va="center", transform=ax.transAxes)
            _name = Path(_of_class.iloc[k]["file"]).name
            ax.set_title(f"{c} · {_name if len(_name) <= 24 else _name[:20] + '…'}", fontsize=8)
fig.suptitle("A first look: are these your images and your classes?", color=INK)
fig.tight_layout()
plt.show()

Four checks run next and end in one report card. Check 1 counts the images per folder, as section 4 did for
the park: the **ratio** is the
largest class divided by the smallest (in the park about 130 cats against 50 sheep, 2 : 1 to 3 : 1).

In [ ]:
# @title Check 1: images per class, and the largest : smallest ratio { display-mode: "form" }
my_counts = my_files["class"].value_counts().reindex(MY_CLASSES)
fig, ax = plt.subplots(figsize=(max(5, 0.8 * len(MY_CLASSES) + 2), 3.2))
ax.bar(MY_CLASSES, my_counts.to_numpy(), color=[MY_COLORS[c] for c in MY_CLASSES])
ax.set_ylabel("images")
ax.set_title(f"Images per class folder ({len(my_files)} in total)")
ax.tick_params(axis="x", rotation=30 if len(MY_CLASSES) > 6 else 0)
plt.tight_layout()
plt.show()
my_imbalance = float(my_counts.max() / my_counts.min())
print(f"largest {my_counts.idxmax()} ({my_counts.max()}) · smallest {my_counts.idxmin()} ({my_counts.min()}) "
      f"· ratio {my_imbalance:.1f} : 1")

💡 Look at the ratio: up to 3 : 1 is common; from 10 : 1 accuracy hides the small class (section 4).

Check 2 opens every file with `is_readable`, then reads its size (width × height) and its colour mode (`RGB`
stores three colour channels, `L` one gray channel).

In [ ]:
# @title Check 2: readable? size and colour mode of every file { display-mode: "form" }
# (section 3's measure_image on a small thumbnail)
_my_rows = []
for _, r in my_files.iterrows():
    ok = is_readable(r["path"])
    stats = measure_image(r["path"], thumb=64)[0] if ok else {}
    _my_rows.append({"file": r["file"], "class": r["class"], "readable": ok, "width": stats.get("width"),
                  "height": stats.get("height"), "mode": stats.get("mode", "unreadable")})
my_stats = pd.DataFrame(_my_rows)
_ok = my_stats[my_stats["readable"]]

fig, ax = plt.subplots(figsize=(5.5, 4))
for c in MY_CLASSES:
    sub = _ok[_ok["class"] == c]
    ax.scatter(sub["width"], sub["height"], s=18, alpha=0.7, color=MY_COLORS[c], label=c)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Image sizes per class")
ax.legend(frameon=False, fontsize=8, ncol=2 if len(MY_CLASSES) > 6 else 1)
plt.tight_layout()
plt.show()
print(f"unreadable files: {int((~my_stats['readable']).sum())}")
print("colour modes:", " · ".join(f"{m} {n}" for m, n in my_stats["mode"].value_counts().items()))
print(f"distinct sizes: {_ok.groupby(['width', 'height']).ngroups} · "
      f"long side from {int(_ok[['width', 'height']].max(axis=1).min())} "
      f"to {int(_ok[['width', 'height']].max(axis=1).max())} px")
display(my_stats[my_stats["mode"] != "RGB"].head(10))

💡 Look for a second cluster of sizes or any mode that is not `RGB`: each is a second source to decide on.

Check 3 is section 5's copy hunt: exact copies by file hash, and near, flipped and rotated copies by the
**distance** between perceptual hashes (differing bits out of 64). Pairs at or below your `HASH_THRESHOLD`
from section 5 count as copies.

In [ ]:
# @title Check 3: exact, recompressed, flipped and rotated copies { display-mode: "form" }
# (all pairs, in blocks to keep memory small)
_readable = my_stats.index[my_stats["readable"]].to_numpy()
_my_paths = my_files.loc[_readable, "path"].tolist()
_md5 = [file_md5(p) for p in _my_paths]
_variant_bits = {v: [] for v in VARIANT_NAMES}
for p in _my_paths:
    for v, im in variants(load_thumb(p, THUMB_FOR_HASH)).items():
        _variant_bits[v].append(phash(im))
_variant_bits = {v: hash_bits(h) for v, h in _variant_bits.items()}
_plain = _variant_bits["original"]
_n = len(_my_paths)
_best = np.full((_n, _n), 65, dtype=np.int16)  # smallest distance over the variants, and which variant
_best_v = np.zeros((_n, _n), dtype=np.int8)
for k, v in enumerate(VARIANT_NAMES):
    for start in range(0, _n, 256):
        d = hash_distances(_variant_bits[v][start:start + 256], _plain)
        better = d < _best[start:start + 256]
        _best[start:start + 256][better] = d[better]
        _best_v[start:start + 256][better] = k

_pairs = []
for i, j in zip(*np.nonzero(np.triu(_best <= HASH_THRESHOLD, k=1)), strict=True):
    a, b = _readable[i], _readable[j]
    kind = "exact" if _md5[i] == _md5[j] else KIND_OF_VARIANT[VARIANT_NAMES[_best_v[i, j]]]
    _pairs.append({"a": my_files.at[a, "file"], "b": my_files.at[b, "file"], "kind": kind,
                   "distance": int(_best[i, j]),
                   "same class": my_files.at[a, "class"] == my_files.at[b, "class"],
                   "ia": a, "ib": b})
my_copies = pd.DataFrame(_pairs, columns=["a", "b", "kind", "distance", "same class", "ia", "ib"])
print(f"{len(my_copies)} copy pairs at HASH_THRESHOLD = {HASH_THRESHOLD}:",
      " · ".join(f"{k} {n}" for k, n in my_copies["kind"].value_counts().items()) or "none")
print(f"pairs whose two files sit in different class folders (one label is wrong): "
      f"{int((~my_copies['same class']).sum())}")
display(my_copies.drop(columns=["ia", "ib"]).head(10))

_show = my_copies.head(4)
if len(_show):
    fig, axes = plt.subplots(len(_show), 2, figsize=(5, 2.4 * len(_show)), squeeze=False)
    for row, (_, pr) in zip(axes, _show.iterrows(), strict=True):
        for ax, idx in zip(row, [pr["ia"], pr["ib"]], strict=True):
            ax.imshow(load_thumb(my_files.at[idx, "path"], 256))
            ax.set_title(f"{my_files.at[idx, 'file']}\n{pr['kind']}", fontsize=8)
            frame(ax, MY_COLORS[my_files.at[idx, "class"]])
    plt.tight_layout()
    plt.show()

💡 Look at the pairs across class folders first: one of the two labels is wrong.

Check 4 suggests a split: it groups the copies, then makes a stratified (every class in both halves, in the
same shares), group-aware split.

In [ ]:
# @title Check 4: a split suggestion, copies grouped, stratified by class { display-mode: "form" }
# Copies form groups that stay on one side; the split is stratified by class.
import warnings

from sklearn.model_selection import StratifiedGroupKFold

_parent = list(range(len(my_files)))  # union-find: every copy pair joins its two files' groups


def _find(i: int) -> int:
    while _parent[i] != i:
        _parent[i] = _parent[_parent[i]]
        i = _parent[i]
    return i


for _, pr in my_copies.iterrows():
    _parent[_find(pr["ia"])] = _find(pr["ib"])
my_groups = np.array([_find(i) for i in range(len(my_files))])

_smallest = int(my_counts.min())
_n_splits = min(5, max(2, _smallest))
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # a class with fewer photos than folds only triggers a warning
    _splitter = StratifiedGroupKFold(n_splits=_n_splits, shuffle=True, random_state=SEED)
    _train_idx, _test_idx = next(_splitter.split(my_files, my_files["class"], groups=my_groups))
my_split = my_files[["file", "class"]].copy()
my_split["split"] = "train"
my_split.loc[_test_idx, "split"] = "test"
_split_copies = sum(my_split.at[pr["ia"], "split"] != my_split.at[pr["ib"], "split"]
                    for _, pr in my_copies.iterrows())
display(pd.crosstab(my_split["class"], my_split["split"]).reindex(MY_CLASSES))
_test_small = int((my_split["split"] == "test").groupby(my_split["class"]).sum().min())
print(f"test share {len(_test_idx) / len(my_files):.0%} · copy pairs across train/test: {_split_copies}")
if _test_small < 10:
    print(f"💡 the smallest class has only {_test_small} test images: prefer stratified {_n_splits}-fold "
          "cross-validation with the same groups, and report the mean and the spread over the folds.")
else:
    print("💡 a stratified 80/20 hold-out with these groups is enough; keep the test half for the end.")
print("If photos share a camera, a session, a burst or a patient, add that to the groups (section 6).")
_split_csv = DATA_DIR / "cache" / "my_split.csv"
my_split.to_csv(_split_csv, index=False)
print(f"split saved to {_split_csv}")

💡 Look at "copy pairs across train/test": it must be 0. The split is written to `data/cache/my_split.csv`,
and **Colab deletes that folder when the session ends**: download it now (Files pane → right-click the
file → Download) or copy it to your Drive, or it is gone next week.

The checklist in one place, to paste into your project notes:

In [ ]:
# @title The report card for this dataset { display-mode: "form" }
_n_unreadable = int((~my_stats["readable"]).sum())
_n_not_rgb = int(((my_stats["mode"] != "RGB") & my_stats["readable"]).sum())
my_report = [
    ("✅" if my_imbalance < 3 else "⚠️", f"classes: {len(MY_CLASSES)}, imbalance {my_imbalance:.1f} : 1"),
    ("✅" if _n_unreadable == 0 else "⚠️", f"unreadable files: {_n_unreadable}"),
    ("✅" if _n_not_rgb == 0 else "⚠️", f"files not stored as RGB: {_n_not_rgb}"),
    ("✅" if len(my_copies) == 0 else "⚠️", f"copy pairs: {len(my_copies)} "
     f"({int((~my_copies['same class']).sum())} across class folders)"),
    ("✅" if _split_copies == 0 else "⚠️", f"split: {len(_test_idx)} test / {len(_train_idx)} train, "
     f"copy pairs across the split: {_split_copies}"),
]
for mark, line in my_report:
    print(mark, line)
RESULTS["swap_in_ok"] = bool(len(my_files) > 0 and len(my_stats) == len(my_split) == len(my_files))
RESULTS["swap_in"] = {"source": my_source, "n_images": int(len(my_files)), "n_classes": len(MY_CLASSES),
                      "copies": int(len(my_copies)), "not_rgb": _n_not_rgb}

### After the lab · Read your results and answer the guiding questions

Everything above runs in class. This block is for you to do afterwards, on your own dataset: it explains how
to read each check and ends with the seven guiding questions.

**Q25.** 🟢 On the sample the folders are cat, dog, bird, horse, sheep and zebra with 20 photos each, plus
three planted problems (an exact copy of a cat, a mirrored dog, a grayscale bird). Which check caught each
one? Then run the checks on your own dataset and compare.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-answer" style="background:#F6EFE4;color:#5C4027;border-left:6px solid #5C4027;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">

<details style="margin:0;"><summary style="cursor:pointer;font-weight:bold;">💡 Worked answer (Q25, on the sample)</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>Check 1:</strong> every folder holds the same number of photos, plus the planted copies in cat and dog, so
  the ratio is close to 1 : 1. That is a property of the sample, not of the park.</li>
<li style="margin:2px 0;"><strong>Check 2:</strong> all sizes fall in the Play Dataset's few clusters (long side 640 or 500 px). The grayscale
  bird shows up as one <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">L</code> file.</li>
<li style="margin:2px 0;"><strong>Check 3:</strong> the exact copy of the cat and the mirrored dog show up as an <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">exact</code> and a <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">flipped</code> pair,
  both inside one folder. The split keeps each pair on one side.</li>
<li style="margin:2px 0;">If a check missed one of them, look at the <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">HASH_THRESHOLD</code> you set in section 5.</li>
</ul>
</details>
</div>""")

<callout: answer>

💡 How to read the ratio:

- **2 or 3 to 1:** common and harmless if you watch the small class's recall.
- **10 to 1 and beyond:** accuracy hides the small class completely (section 4's ⚠️ Pitfall).
- **Always ask:** is the *expensive* class the small one?

💡 How to read the sizes and modes:

- **One tight cluster of sizes:** one source.
- **A second cluster, or one class sitting apart:** a second source, and the model may learn the source
  instead of the animal.
- **Every file that is not RGB** is listed in check 2: decide whether to convert it or drop it.

💡 Every copy pair is a decision for a human:

- **Keep one file of each pair.**
- **Look twice at pairs from different class folders:** one of the two labels is wrong.
- **Many pairs on near-identical shots** (a fixed camera, a microscope) are genuinely different photos:
  lower `HASH_THRESHOLD` in section 5 and rerun.

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · The split is only a suggestion</div>
<p style="margin:6px 0 0 0;">The split comes from what the folders show. Only you know what else ties photos together, such as one
camera, one day, one patient; section 6 showed that a split which ignores such groups reports a better score
than the model will earn.</p>
</div>""")

<callout: go-deeper>

#### The guiding questions

These questions are the notebook in short. Each one has the park's answer, and an empty line for yours.

**Q26.** 🟢 Double-click the next cell and answer each guiding question for your project dataset, using
the output of the checklist above and what you know about where the data comes from.

1. **What is the task, and which mistake is the expensive one?**
   - *The park:* classification of who is in front of the camera; the expensive mistake is a missed dog
     in the sheep paddock, so the metric is recall on dog (section 4).
   - *Your dataset:*
2. **Do the images look like what the model will see in use?**
   - *The park:* only partly. COCO has close-ups, sofas and people; a fixed paddock camera sees small
     animals on grass (section 2).
   - *Your dataset:*
3. **How balanced are the classes, and is the expensive class a small one?**
   - *The park:* cat and dog about 130 photos each, sheep about 50, roughly 2 : 1 to 3 : 1. Dog is not
     rare, but accuracy would still hide a weak recall on dog (section 4).
   - *Your dataset:*
4. **Which sizes, aspect ratios and colour modes are there, and what must be resized or converted?**
   - *The park:* nearly every photo has a long side of 640 or 500 px, in landscape and portrait; files
     that are not RGB get converted (section 3).
   - *Your dataset:*
5. **Are there copies, flipped or rotated copies, unreadable or odd files, or doubtful labels?**
   - *The park:* a few copies, removed before splitting (section 5's "removed N, kept M"); photos with
     several animals make the label ambiguous (section 4).
   - *Your dataset:*
6. **What ties photos together, and which split keeps those groups apart?**
   - *The park:* COCO has no groups, so a stratified split after deduplication; the trail cameras have
     bursts and locations, so they are split by camera (section 6).
   - *Your dataset:*
7. **What must be fixed before training?**
   - *The park:* drop the copies, convert the files that are not RGB, decide the label of multi-animal
     photos, and resize without distorting the aspect ratio (section 3's 🔀 Callout).
   - *Your dataset:*

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaway</div>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>When you get a new folder-per-class dataset</strong> → <strong>do</strong> run the four checks (classes, sizes and modes,
  copies, split) before anything else; they take a few minutes.</li>
<li style="margin:2px 0;"><strong>When a check lists files</strong> (not RGB, copy pairs, pairs across class folders) → <strong>do</strong> decide per file:
  convert, keep one, or fix the label.</li>
<li style="margin:2px 0;"><strong>When you leave</strong> → <strong>do</strong> download <code style="font-family:'Roboto Mono',SFMono-Regular,Menlo,Consolas,monospace;font-size:0.92em;background:rgba(0,0,0,0.07);padding:1px 4px;border-radius:3px;">data/cache/my_split.csv</code> (Colab deletes it) and bring it, with your
  answers to the guiding questions,
  to next week's training.</li>
</ul>
</div>""")

<callout: takeaway>

## 11. Wrap-up & next week

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-objective" style="background:#B9E3EA;color:#08505C;border-left:6px solid #08505C;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🎯 Objective</div>
<p style="margin:6px 0 0 0;">Look back at the three things you set out to do, see the park's numbers for each, and know where next
week's training starts.</p>
</div>""")

<callout: objective>

Nothing was trained today, and yet we now know a lot about whether the park's data is fit for its three
jobs. The next cell collects the numbers the earlier sections measured, grouped by the three outcomes from
the title banner.

In [ ]:
# @title The park's numbers from today, one block per learning outcome { display-mode: "form" }
_photos = RESULTS["images_per_class"]
_boxes = RESULTS["boxes_per_class"]
_pixels = RESULTS["pixels_per_class"]
_big, _small = max(_photos, key=_photos.get), min(_photos, key=_photos.get)
recap = {
    "1 · Inspect and judge the fit": [
        f"{RESULTS['n_images']} photos of {len(CLASSES)} classes, every one measured "
        f"({RESULTS['image_stats_n']} rows of image stats)",
        f"largest class {_big} ({_photos[_big]}), smallest {_small} ({_photos[_small]}): "
        f"{_photos[_big] / _photos[_small]:.1f} : 1",
    ],
    "2 · Find and fix errors": [
        f"removed {RESULTS['n_removed']}, kept {RESULTS['n_clean']} "
        "(copies, before any split)",
    ],
    "3 · Split, metric, and the three tasks": [
        f"test photos with a burst sibling in train: random split {RESULTS['sibling_rate_random']:.0%}, "
        f"split by camera {RESULTS['sibling_rate_by_camera']:.0%}",
        f"'always sheep' on a paddock week: accuracy {RESULTS['accuracy']:.0%}, "
        f"recall on dog {RESULTS['recall_dog']:.0%} → the alert is judged on {RESULTS['alert_metric']}",
        f"bird: {_photos['bird'] / sum(_photos.values()):.0%} of photos, "
        f"{_boxes['bird'] / sum(_boxes.values()):.0%} of boxes, {_pixels['bird']:.0%} of pixels",
    ],
}
for outcome, lines in recap.items():
    print(outcome)
    for line in lines:
        print("   ", line)

💡 Every line is a decision you can now defend: what to fix, how to split, what to measure, and in which
unit (photo, box or pixel) to count.

<details><summary>About the numbers</summary>

In smoke mode the numbers are tiny; the full run gives the park's real ones.

</details>

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-takeaway" style="background:#E2F0C4;color:#3A5A0B;border-left:6px solid #3A5A0B;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">✅ Takeaways</div>
<p style="margin:6px 0 0 0;">You can now do the three things from the banner:</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>1 · Inspect.</strong> <strong>When you see</strong> a new image dataset → <strong>do</strong> look at many photos before computing
  anything, then measure sizes, modes, colours and class counts.</li>
<li style="margin:2px 0;"><strong>2 · Fix errors.</strong> <strong>When you see</strong> files that may be the same photo → <strong>do</strong> hash the files for exact
  copies and the pixels for near copies, and write down "removed N, kept M".</li>
<li style="margin:2px 0;"><strong>3 · Split and measure.</strong> <strong>When you see</strong> photos tied together by a camera or a burst → <strong>do</strong>
  deduplicate, then split by that group. Always say what you count: photos, boxes or pixels.</li>
</ul>
<details style="margin:6px 0 0 0;">
<summary style="cursor:pointer;font-weight:bold;">How to do each step, with examples</summary>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><strong>1 · Inspect:</strong> an outlier points at a bug; a photo the park's camera would never take points
  at a poor fit.</li>
<li style="margin:2px 0;"><strong>2 · Fix errors:</strong> flipped and rotated variants are near copies too, so hash the pixels as well
  as the bytes;
  open every file once, and let a human judge the least typical labels and the ugliest masks.</li>
<li style="margin:2px 0;"><strong>3 · Split and measure:</strong> pick the metric from the expensive mistake: recall on dog for the
  paddock alert, counting error for the headcount, IoU per class for the outlines.</li>
</ul>
</details>
</div>""")

<callout: takeaway>

### Next week: training

The cleaned table and the split go into a model: first a dumb end-to-end baseline, then a model that
overfits, then regularisation (Karpathy's recipe, continued). **Bring along** your answers to the guiding
questions of section 10 and your split (`data/cache/my_split.csv`, or the one you choose by hand).

In [ ]:
# @title
from IPython.display import HTML; HTML(r"""<div class="callout callout-go-deeper" style="background:#F3F4F6;color:#4B5563;border-left:6px solid #4B5563;border-radius:6px;padding:12px 14px;line-height:1.5;font-family:'Google Sans',Roboto,'Helvetica Neue',Arial,sans-serif;font-size:14px;width:100%;box-sizing:border-box;white-space:normal;overflow-wrap:break-word;margin:2px 0;">
<div style="font-weight:bold;">🔍 Go deeper · Augmentation and experiment tracking</div>
<p style="margin:6px 0 0 0;">The flipped and rotated copies you hunted are what augmentation makes on purpose, on the training set
only, after the split. Training also means many runs, say five learning rates for the dog-in-the-paddock
model, and a tracking tool records each run's settings, curves and scores.</p>
<ul style="margin:6px 0 0 0;padding-left:22px;">
<li style="margin:2px 0;"><a href="https://neptune.ai/blog/data-augmentation-in-python" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Data augmentation in Python (Neptune.ai)</a>:
  techniques and libraries.</li>
<li style="margin:2px 0;"><a href="https://docs.ultralytics.com/guides/yolo-data-augmentation/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">YOLO data augmentation (Ultralytics)</a>:
  for detection, where boxes must move with the pixels.</li>
<li style="margin:2px 0;"><a href="https://wandb.ai/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Weights &amp; Biases</a> and <a href="https://neptune.ai/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">Neptune.ai</a>: hosted trackers for runs.</li>
<li style="margin:2px 0;"><a href="https://www.tensorflow.org/tensorboard" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">TensorBoard</a> and <a href="https://mlflow.org/" target="_blank" rel="noopener" style="color:inherit;text-decoration:underline;">MLflow</a>: trackers on
  your own machine, open source.</li>
</ul>
</div>""")

<callout: go-deeper>